In [1]:
%reload_ext autoreload
%autoreload 2
import sys, os
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

# Add project root to path
project_root = '/sdf/group/neutrino/pgranger/larnd-sim-jax/src'
if project_root not in sys.path:
    sys.path.insert(0, project_root)

%cd /sdf/group/neutrino/pgranger/larnd-sim-jax

from larndsim.consts_jax import build_params_class, load_detector_properties, load_lut
from larndsim.losses_jax import llhd_loss, adc2charge, mse_adc
from optimize.strategies import LUTSimulation, LUTProbabilisticSimulation, ProbabilisticLossStrategy, CollapsedProbabilisticLossStrategy
from optimize.dataio import TracksDataset, DataLoader
from optimize.ranges import ranges

import optimize.strategies
import logging
logging.getLogger(optimize.strategies.__name__).setLevel(logging.WARNING)

In [2]:
# --- Configuration ---

from larndsim.consts_jax import RecombinationMode

# INPUT_FILE = 'prepared_data/input_1.h5'
# INPUT_FILE = '/sdf/data/neutrino/cyifan/diffsim_input/true_ending_muon_edep_5cm_vol2cm_range_0.2-cm_CSDA.h5'
INPUT_FILE = '/sdf/data/neutrino/cyifan/diffsim_input/true_through_muon_edep_10cm_vol1cm.h5'
# INPUT_FILE = '/sdf/data/neutrino/cyifan/dunend_train_prod/prod_mod0_mpvmpr/production_2043327/job_25174367_0000/output_25174367_0000-edepsim_lbl_trklen2cm_containment2cm_costheta0.966_range_0.05cm_ntrack_1_seg_0.h5'
# INPUT_FILE = '/sdf/data/neutrino/cyifan/dunend_train_prod/prod_mod0_mpvmpr/production_2043327/job_25174367_0000/output_25174367_0000-edepsim_lbl_trklen2cm_containment2cm_costheta0.966_range_0.05cm_ntrack_1_seg_0.h5'
LUT_FILE = 'src/larndsim/detector_properties/response_44_v2a_full_tick.npz'
DET_PROPS = 'src/larndsim/detector_properties/module0.yaml'
PIXEL_LAYOUTS = 'src/larndsim/pixel_layouts/multi_tile_layout-2.4.16_v4.yaml'

# Parameters to optimize (and their ranges)

# RELEVANT_PARAMS = ['alpha', 'beta', 'R_param', 'lifetime', 'tran_diff', 'long_diff', 'eField', 'shift_x', 'shift_y', 'shift_z']
RELEVANT_PARAMS = ['Ab', 'kb', 'lifetime', 'tran_diff', 'long_diff', 'eField', 'shift_x', 'shift_y', 'shift_z']

# Simulation Settings
# ELECTRON_SAMPLING_RESOLUTION = 0.01
ELECTRON_SAMPLING_RESOLUTION = 0.1
NUMBER_PIX_NEIGHBORS = 4
SIGNAL_LENGTH = 150

# --- Setup Parameters ---

ParamsClass = build_params_class(RELEVANT_PARAMS)
ref_params = load_detector_properties(ParamsClass, DET_PROPS, PIXEL_LAYOUTS)
ref_params = ref_params.replace(
    electron_sampling_resolution=ELECTRON_SAMPLING_RESOLUTION,
    number_pix_neighbors=NUMBER_PIX_NEIGHBORS,
    signal_length=SIGNAL_LENGTH,
    time_window=SIGNAL_LENGTH,
    # recombination_mode=RecombinationMode.ELLIPSOID,
    # RESET_NOISE_CHARGE = 0.1,
    #DISCRIMINATION_THRESHOLD = 5000,
    # UNCORRELATED_NOISE_CHARGE = 0.1,
    # Ensure noise params are zero for deterministic checks if needed, but llhd handles noise in prob
)

response, ref_params = load_lut(LUT_FILE, ref_params)

# Set nominal values
initial_values = {p: ranges[p]['nom'] for p in RELEVANT_PARAMS}
current_params = ref_params.replace(**initial_values)

print("Parameters initialized.")

In [ ]:
# --- Initialize Strategies ---

# 1. Stochastic Strategy for Generating Target (Ground Truth)
target_strategy = LUTSimulation(response)

# 2. Probabilistic Strategy for Prediction (Differentiable)
pred_strategy = LUTProbabilisticSimulation(response)

# 3. Loss Strategy - now computes proper negative log-likelihood
# No need to pass llhd_loss, the strategy implements it internally
# loss_strategy = ProbabilisticLossStrategy(sigma_charge=500.0)
loss_strategy = ProbabilisticLossStrategy(
    sigma_charge=ref_params.UNCORRELATED_NOISE_CHARGE, , apply_deadtime=True)

print("Strategies initialized.")
print("Loss: Negative log-likelihood with Gaussian charge prior (sigma=500 electrons)")

In [4]:

# --- Helper: normalise target output key names ---
# target_strategy.predict() returns 'hit_pixels' while ProbabilisticLossStrategy
# expects 'pixel_id'.  Centralise the renaming so every call site is consistent.

def format_target_output(raw: dict) -> dict:
    """Ensure target dict has 'pixel_id' key (alias for 'hit_pixels')."""
    out = dict(raw)
    if 'hit_pixels' in out and 'pixel_id' not in out:
        out['pixel_id'] = out['hit_pixels']
    return out


In [5]:
# --- Load Data ---

dataset = TracksDataset(filename=INPUT_FILE, nevents=-1, max_nbatch=20, max_batch_len=300,
                        electron_sampling_resolution=ELECTRON_SAMPLING_RESOLUTION)
dataloader = DataLoader(dataset, batch_size=1)

# Get one batch
tracks_batch = dataloader[0]
track_fields = dataset.get_track_fields()


tracks = jax.device_put(tracks_batch.reshape(-1, len(track_fields)))

In [6]:
import h5py
def get_full_dedx_distrib(ifile):
    with h5py.File(ifile, 'r') as f:
        tracks = f['segments'][:]
    return tracks['dEdx']

distrib = get_full_dedx_distrib(INPUT_FILE)
q = jnp.linspace(0.0, 1.0, 100)
selected = distrib[distrib < 5]
ref_dedx_distrib = selected
ref_quantiles = jnp.quantile(selected, q)

plt.figure(figsize=(8,6))
plt.hist(selected, bins=100, density=True, alpha=0.6, color='g')
plt.plot(ref_quantiles, q, marker='o', linestyle='-', color='r')
plt.xlabel('dE/dx (MeV/cm)')
plt.ylabel('Cumulative Probability')
plt.title('dE/dx Distribution with Quantiles')
plt.grid()

In [7]:

# ═══════════════════════════════════════════════════════════
# PHASE 1.1 — Forward model smoke test
# Runs both strategies on one batch at nominal params.
# Checks shapes, value ranges, NaN/Inf, and pixel overlap.
# ═══════════════════════════════════════════════════════════
import numpy as np

current_params = ref_params.replace(signal_length=400, tran_diff=1e-20)

_smk_tracks = jax.device_put(dataloader[0].reshape(-1, len(track_fields)))
event_idx   = track_fields.index('eventID')
valid_seg   = _smk_tracks[:, event_idx] >= 0
print(f"Batch shape: {_smk_tracks.shape}  |  valid segments: {int(valid_seg.sum())}")

# -- Stochastic target --
_smk_target = format_target_output(
    target_strategy.predict(current_params, _smk_tracks, track_fields, 42)
)
print(f"\n[target] hit_pixels shape : {_smk_target['hit_pixels'].shape}")
print(f"[target] adcs  range      : [{float(_smk_target['adcs'].min()):.1f}, {float(_smk_target['adcs'].max()):.1f}]")
print(f"[target] ticks range      : [{int(_smk_target['ticks'].min())}, {int(_smk_target['ticks'].max())}]")

# -- Probabilistic prediction --
_smk_pred = pred_strategy.predict(current_params, _smk_tracks, track_fields, 42)
print(f"\n[pred]   unique_pixels shape  : {_smk_pred['unique_pixels'].shape}")
print(f"[pred]   adcs_distrib  shape  : {_smk_pred['adcs_distrib'].shape}")
print(f"[pred]   hit_prob    shape  : {_smk_pred['hit_prob'].shape}")
print(f"[pred]   adcs_distrib  range  : [{float(_smk_pred['adcs_distrib'].min()):.3f}, {float(_smk_pred['adcs_distrib'].max()):.3f}]")

has_nan_pred = bool(jnp.any(jnp.isnan(_smk_pred['adcs_distrib'])) |
                    jnp.any(jnp.isnan(_smk_pred['hit_prob'])))
print(f"\nNaN in pred output : {has_nan_pred}  (should be False)")

# -- Pixel overlap target ↔ pred --
tgt_set  = set(_smk_target['hit_pixels'].tolist())
pred_set = set(_smk_pred['unique_pixels'].tolist())
n_match  = len(tgt_set & pred_set)
print(f"Pixel overlap      : {n_match}/{len(tgt_set)} target pixels matched in pred "
      f"({100*n_match/max(1,len(tgt_set)):.1f} %)")
print(f"Pixels only in target: {len(tgt_set - pred_set)}  "
      f"only in pred: {len(pred_set - tgt_set)}")

# -- Quick loss value --
_smk_loss, _smk_aux = loss_strategy.compute(current_params, _smk_pred, _smk_target)
print(f"\nLoss at nominal params : {float(_smk_loss):.4f}")


In [9]:

# ═══════════════════════════════════════════════════════════
# PHASE 1.2 — Full PPP-NLL component decomposition
#
# The ProbabilisticLossStrategy computes a Poisson Point
# Process negative log-likelihood with four terms:
#
#   NLL = -Σ log P(tick|pix)          [tick LL, matched hits]
#         -Σ log P(charge|tick,pix)   [charge LL, matched hits]
#         -log(ε) × N_unmatched        [false-negative penalty]
#         +Σ exp(ticks_prob)            [false-positive penalty]
#
# This cell visualises each component and their contributions
# to the total NLL budget.
# ═══════════════════════════════════════════════════════════
from optimize.strategies import compute_occurrence_indices

_lc_target  = _smk_target
_lc_pred    = _smk_pred
_params     = current_params

t_pixels    = _lc_target['hit_pixels']
t_ticks     = _lc_target['ticks'].astype(int)
t_adcs      = _lc_target['adcs']
p_pixels    = _lc_pred['unique_pixels']
p_ticks_prob= _lc_pred['hit_prob']       # (Npix, Ntrig, Ntick) — log-probs
p_adcs_dist = _lc_pred['adcs_distrib']   # (Npix, Ntrig, Ntick)

# ── Pixel matching ─────────────────────────────────────────
pidx  = jnp.searchsorted(p_pixels, t_pixels)
pidx  = jnp.clip(pidx, 0, p_pixels.shape[0] - 1)
match = (p_pixels[pidx] == t_pixels) & (t_pixels >= 0)
trig  = compute_occurrence_indices(t_pixels)

eps        = float(loss_strategy.eps)
log_eps    = float(jnp.log(eps))        # ~-18.4 for eps=1e-8
sig        = float(loss_strategy.sigma_charge)
sig_ke     = sig / 1000.0

# ── 1. Tick log-intensity ──────────────────────────────────
# Apply the same floor as compute() — jnp.maximum(raw_logp, log(eps))
# Without this, ticks with near-zero probability give log-probs
# far below -18 (e.g. -1000 from log-softmax over many ticks).
tick_logp_all = jnp.maximum(p_ticks_prob[pidx, trig, t_ticks], log_eps)
tick_logp     = tick_logp_all[match]
tick_nll_sum  = float(-jnp.sum(tick_logp))

# ── 2. Charge log-intensity ────────────────────────────────
t_chg  = adc2charge(t_adcs, _params)
p_chg  = adc2charge(p_adcs_dist[pidx, trig, t_ticks], _params)
chg_ll     = -0.5 * ((t_chg - p_chg) / sig_ke) ** 2 \
              - 0.5 * float(jnp.log(2 * jnp.pi * sig_ke**2))
chg_ll_matched = chg_ll[match]
chg_nll_sum    = float(-jnp.sum(chg_ll_matched))

# ── 3. False-negative penalty (unmatched target pixels) ───
n_unmatched   = int(jnp.sum(~match))
fn_penalty    = -log_eps * n_unmatched    # positive contribution to NLL

# ── 4. False-positive penalty (expected total hits) ───────
fp_penalty = float(jnp.sum(jnp.exp(p_ticks_prob)))

# ── Summary ────────────────────────────────────────────────
total_nll = tick_nll_sum + chg_nll_sum + fn_penalty + fp_penalty
n_matched = int(jnp.sum(match))
n_target  = len(t_pixels)
n_pred_pix = p_pixels.shape[0]

print("══════════════════════════════════════════")
print("  PPP-NLL component breakdown")
print("══════════════════════════════════════════")
print(f"  Target hits          : {n_target}")
print(f"  Matched hits         : {n_matched}  ({100*n_matched/max(n_target,1):.1f}%)")
print(f"  Unmatched hits       : {n_unmatched}  ({100*n_unmatched/max(n_target,1):.1f}%)")
print(f"  Predicted pixels     : {n_pred_pix}")
print()
print(f"  [1] Tick  -LL sum    : {tick_nll_sum:>12.2f}")
print(f"  [2] Charge -LL sum   : {chg_nll_sum:>12.2f}")
print(f"  [3] FN penalty       : {fn_penalty:>12.2f}  (log_eps={log_eps:.2f}  × {n_unmatched})")
print(f"  [4] FP penalty       : {fp_penalty:>12.2f}  (Σ exp(tick_logp) over all pix/ticks)")
print(f"  ─────────────────────────────────────")
print(f"  Total NLL            : {total_nll:>12.2f}")
print()
print(f"  Per-hit averages (matched hits only):")
print(f"    tick  logp : {float(jnp.mean(tick_logp)):.4f}  ± {float(jnp.std(tick_logp)):.4f}")
print(f"    charge  ll : {float(jnp.mean(chg_ll_matched)):.4f}  ± {float(jnp.std(chg_ll_matched)):.4f}")
print()
# How many hits were actually floored?
n_floored = int(jnp.sum(p_ticks_prob[pidx, trig, t_ticks][match] < log_eps))
print(f"  Hits whose tick logp was below floor (< {log_eps:.1f}): {n_floored} / {n_matched}")


# ══════════════════════════════════════════════════════════
# Figure layout:
#   Row 1 — per-hit distributions (matched) + FP heatmap
#   Row 2 — NLL budget + scatter + match rate
# ══════════════════════════════════════════════════════════
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle("PPP-NLL Full Component Decomposition", fontsize=13, fontweight='bold')

# ── [0,0] Tick log-prob distribution ──────────────────────
ax = axes[0, 0]
tick_logp_raw = p_ticks_prob[pidx, trig, t_ticks][match]   # before floor, for comparison
ax.hist(np.array(tick_logp_raw), bins=60, histtype='stepfilled', alpha=0.4,
        color='gray', label='raw (no floor)')
ax.hist(np.array(tick_logp), bins=60, histtype='step', lw=1.5,
        color='steelblue', label=f'floored at {log_eps:.1f}')
ax.axvline(float(jnp.mean(tick_logp)), color='navy', ls='--', lw=1.5,
           label=f'mean={float(jnp.mean(tick_logp)):.3f}')
ax.set_xlabel('log P(tick | pixel)  [nats]')
ax.set_ylabel('counts')
ax.set_title('[1] Tick log-intensity')
ax.legend(fontsize=8)
ax.set_yscale('log')

# ── [0,1] Charge log-likelihood distribution ──────────────
ax = axes[0, 1]
ax.hist(np.array(chg_ll_matched), bins=60, histtype='stepfilled', alpha=0.6,
        color='darkorange', label=f'n={n_matched}')
ax.axvline(float(jnp.mean(chg_ll_matched)), color='saddlebrown', ls='--', lw=1.5,
           label=f'mean={float(jnp.mean(chg_ll_matched)):.3f}')
ax.set_xlabel(f'log P(charge | tick,pixel)  [σ={sig:.0f} e⁻]')
ax.set_ylabel('counts')
ax.set_title('[2] Charge log-intensity')
ax.legend(fontsize=8)

# ── [0,2] Scatter: tick logp vs charge ll ─────────────────
ax = axes[0, 2]
ax.scatter(np.array(tick_logp), np.array(chg_ll_matched),
           s=4, alpha=0.3, color='purple')
ax.set_xlabel('tick log P  [nats]')
ax.set_ylabel('charge log P  [nats]')
ax.set_title('Tick vs Charge LL (per matched hit)')

# ── [1,0] NLL budget bar chart ────────────────────────────
ax = axes[1, 0]
components = ['Tick -LL\n(matched)', 'Charge -LL\n(matched)',
              'FN penalty\n(unmatched)', 'FP penalty\n(expected hits)']
values     = [tick_nll_sum, chg_nll_sum, fn_penalty, fp_penalty]
colors_bar = ['steelblue', 'darkorange', 'crimson', 'seagreen']
bars = ax.bar(components, values, color=colors_bar, alpha=0.8, edgecolor='k', linewidth=0.5)
for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + total_nll*0.005,
            f'{val:.1f}', ha='center', va='bottom', fontsize=8)
ax.set_ylabel('NLL contribution')
ax.set_title('NLL Budget')
ax.axhline(0, color='k', lw=0.7)

# ── [1,1] FP penalty: expected hits per pixel histogram ───
ax = axes[1, 1]
exp_hits_per_pix = np.array(jnp.sum(jnp.exp(p_ticks_prob), axis=(1, 2)))
ax.hist(exp_hits_per_pix, bins=60, histtype='stepfilled', alpha=0.6, color='seagreen')
ax.axvline(float(jnp.mean(jnp.array(exp_hits_per_pix))), color='darkgreen', ls='--', lw=1.5,
           label=f'mean={float(jnp.mean(jnp.array(exp_hits_per_pix))):.3f}')
ax.set_xlabel('Expected hits per predicted pixel')
ax.set_ylabel('pixel count')
ax.set_title('[4] FP penalty: exp hits per pixel')
ax.legend(fontsize=8)

# ── [1,2] Pixel match statistics ──────────────────────────
ax = axes[1, 2]
categories = ['Matched\ntarget hits', 'Unmatched\n(FN)', 'Predicted\npixels (FP src)']
vals_match = [n_matched, n_unmatched, n_pred_pix]
clrs_match = ['steelblue', 'crimson', 'seagreen']
ax.bar(categories, vals_match, color=clrs_match, alpha=0.8, edgecolor='k', linewidth=0.5)
for i, v in enumerate(vals_match):
    ax.text(i, v + max(vals_match)*0.01, str(v), ha='center', va='bottom', fontsize=9)
ax.set_ylabel('count')
match_pct = 100 * n_matched / max(n_target, 1)
ax.set_title(f'Match rate: {match_pct:.1f}%  ({n_matched}/{n_target} target hits)')

plt.tight_layout()
plt.show()


In [9]:

# ═══════════════════════════════════════════════════════════
# PHASE 1.3 — dEdx-only fit (no physics parameter update, no prior)
#
# Physics parameters are frozen at their nominal values.
# All regularisation / prior weights are set to 0.
# One log_dedx_mean + log_dedx_sigma is fitted per ORIGINAL
# segment (before chopping). jnp.take broadcasts each scalar
# to all chopped sub-rows of the same parent — the gradient
# is accumulated back to the parent automatically.
# ═══════════════════════════════════════════════════════════
import jax
import jax.numpy as jnp
from jax import value_and_grad
import optax
import numpy as np

# ── Hyper-parameters ──────────────────────────────────────────────────────────
_dedxonly_num_batches  = 1
_dedxonly_local_lr     = 5e-2
_dedxonly_num_epochs   = 200
_dedxonly_inner_steps  = 1

# ── Loss function: pure hit NLL, no priors, fixed physics params ──────────────
def compute_loss_dedx_only(local_params, tracks_batch, parent_seg_ids_jax, target_out, key):
    """
    Loss = hit NLL only.  Physics params stay at current_params (nominal).

    local_params:
        log_dedx_mean  : shape [N_orig_segs]  — one value per original segment
        log_dedx_sigma : shape [N_orig_segs]  — one value per original segment

    parent_seg_ids_jax : int32 array [N_chopped_rows], values in [0, N_orig_segs).
        Padded rows carry -1 and are handled by clipping + the eventID mask.
    """
    # Broadcast one value per original segment to every chopped row
    safe_ids = jnp.clip(parent_seg_ids_jax, 0, local_params['log_dedx_mean'].shape[0] - 1)
    log_dedx_per_row   = jnp.take(local_params['log_dedx_mean'],  safe_ids)  # [N_rows]
    log_sigma_per_row  = jnp.take(local_params['log_dedx_sigma'], safe_ids)  # [N_rows]

    dedx_mean  = jnp.exp(log_dedx_per_row)
    dedx_sigma = jnp.exp(log_sigma_per_row)

    # Reparameterisation trick
    eps         = jax.random.normal(key, shape=dedx_mean.shape)
    eps = 0
    dedx_sample = jnp.maximum(dedx_mean + dedx_sigma * eps, 1e-3)

    dedx_idx       = track_fields.index('dEdx')
    dE_idx         = track_fields.index('dE')
    dx_idx         = track_fields.index('dx')
    updated_tracks = tracks_batch.at[:, dedx_idx].set(dedx_sample)
    updated_tracks = updated_tracks.at[:, dE_idx].set(
        dedx_sample * updated_tracks[:, dx_idx]
    )

    pred_dist     = pred_strategy.predict(current_params, updated_tracks, track_fields, 42)
    hit_loss, aux = loss_strategy.compute(current_params, pred_dist, target_out)
    return hit_loss, aux

vg_dedxonly = value_and_grad(compute_loss_dedx_only, argnums=0, has_aux=True)

# ── Local optimizer + cache ───────────────────────────────────────────────────
_dedxonly_optimizer = optax.adam(_dedxonly_local_lr)
_dedxonly_cache     = {}   # batch_idx → {'log_dedx_mean', 'log_dedx_sigma', 'opt_state'}

# ── RNG ───────────────────────────────────────────────────────────────────────
_dedxonly_rng = jax.random.PRNGKey(99)

# ── History ───────────────────────────────────────────────────────────────────
_dedxonly_losses     = []
_dedxonly_dedx_dist  = []
_dedxonly_dedx_scale = []

current_params = ref_params.replace()

# ── Training loop ─────────────────────────────────────────────────────────────
for _epoch in range(_dedxonly_num_epochs):
    for _bidx in range(min(_dedxonly_num_batches, len(dataloader))):

        # 1. Prepare batch
        _tracks_b = jax.device_put(
            dataloader[_bidx].reshape(-1, len(track_fields))
        )
        _target_b = format_target_output(
            target_strategy.predict(current_params, _tracks_b, track_fields, _bidx)
        )

        # 2. Retrieve parent segment ids (populated when dataloader[_bidx] was called above)
        #    parent_seg_ids[j] = index of the original pre-chop segment for chopped row j
        #    Padded rows have -1; clipped to 0 inside the loss (they are masked by eventID anyway)
        _parent_ids_np  = dataset.get_parent_segment_ids(_bidx)
        _parent_ids_jax = jax.device_put(jnp.array(_parent_ids_np, dtype=jnp.int32))
        _N_orig         = int(_parent_ids_np[_parent_ids_np >= 0].max()) + 1  # # original segs

        # 3. Initialise local state on first visit
        if _bidx not in _dedxonly_cache:
            _init_log_mean  = jnp.full((_N_orig,), 0, dtype=jnp.float32)
            _init_log_sigma = jnp.full((_N_orig,), 0,  dtype=jnp.float32)
            _init_lp = {'log_dedx_mean': _init_log_mean,
                        'log_dedx_sigma': _init_log_sigma}
            _dedxonly_cache[_bidx] = {
                'log_dedx_mean':  _init_log_mean,
                'log_dedx_sigma': _init_log_sigma,
                'opt_state':      _dedxonly_optimizer.init(_init_lp),
            }

        _local_params    = {
            'log_dedx_mean':  _dedxonly_cache[_bidx]['log_dedx_mean'],
            'log_dedx_sigma': _dedxonly_cache[_bidx]['log_dedx_sigma'],
        }
        _local_opt_state = _dedxonly_cache[_bidx]['opt_state']

        # 4. Optimise local dEdx (no global physics step)
        for _ in range(_dedxonly_inner_steps):
            _dedxonly_rng, _step_key = jax.random.split(_dedxonly_rng)
            (_loss_val, _aux), _grad_local = vg_dedxonly(
                _local_params, _tracks_b, _parent_ids_jax, _target_b, _step_key
            )
            _local_updates, _local_opt_state = _dedxonly_optimizer.update(
                _grad_local, _local_opt_state, _local_params
            )
            _local_params = optax.apply_updates(_local_params, _local_updates)

        # 5. Persist local state
        _dedxonly_cache[_bidx]['log_dedx_mean']  = jnp.minimum(_local_params['log_dedx_mean'], jnp.log(10))
        _dedxonly_cache[_bidx]['log_dedx_sigma'] = _local_params['log_dedx_sigma']
        _dedxonly_cache[_bidx]['opt_state']      = _local_opt_state

        # 6. Log metrics — compare per-original-segment fitted vs true dEdx
        _ev_idx      = track_fields.index('eventID')
        _dedx_idx    = track_fields.index('dEdx')

        # Gather true dEdx for each original segment (first chopped row of each parent)
        _parent_ids_np_valid = _parent_ids_np[_parent_ids_np >= 0]
        _tracks_b_np         = np.array(_tracks_b)
        _valid_rows          = np.where(_parent_ids_np >= 0)[0]

        # For each original segment take the true dEdx from its first chopped row
        _, _first_occ = np.unique(_parent_ids_np_valid, return_index=True)
        _first_rows   = _valid_rows[_first_occ]
        _true_dedx    = _tracks_b_np[_first_rows, _dedx_idx]      # [N_orig]
        _fitted_dedx  = np.exp(np.array(_local_params['log_dedx_mean']))  # [N_orig]

        _scale = float(np.mean(_fitted_dedx/ np.clip(_true_dedx, 1e-6, None), where=_fitted_dedx<8))
        _dist  = float(np.mean(np.abs(_fitted_dedx - _true_dedx), where=_fitted_dedx<8))

        _dedxonly_losses.append(float(_loss_val))
        _dedxonly_dedx_scale.append(_scale)
        _dedxonly_dedx_dist.append(_dist)

        if _bidx % 5 == 0 or _bidx == _dedxonly_num_batches - 1:
            print(
                f"Ep {_epoch:02d} | B {_bidx:02d} | "
                f"N_orig_segs={_N_orig} | "
                f"HitNLL={float(_loss_val):.4f} | "
                f"dEdx_scale={_scale:.4f} | "
                f"dEdx_dist={_dist:.4f}"
            )

# ── Summary plot ──────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt

_steps = np.arange(len(_dedxonly_losses))
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
axes[0].semilogy(_steps, _dedxonly_losses)
axes[0].set_title('Hit NLL (no priors)'); axes[0].set_xlabel('Step')
axes[1].plot(_steps, _dedxonly_dedx_scale)
axes[1].axhline(1.0, color='grey', linestyle=':', lw=0.8)
axes[1].set_title('dEdx scale  (fitted / true)\nper original segment'); axes[1].set_xlabel('Step')
axes[2].plot(_steps, _dedxonly_dedx_dist, color='tab:orange')
axes[2].set_title('Mean |fitted dEdx − true dEdx|  (MeV/cm)\nper original segment'); axes[2].set_xlabel('Step')
fig.suptitle('dEdx-only fit  —  physics fixed at nominal, no priors, 1 param per original segment')
fig.tight_layout()
plt.show()


In [10]:

# ═══════════════════════════════════════════════════════════
# eField-only fit with true dEdx
# dEdx is taken directly from the input tracks (not optimised).
# Only log(eField) is updated.
# ═══════════════════════════════════════════════════════════
import jax
import jax.numpy as jnp
from jax import value_and_grad
import optax
import numpy as np
import matplotlib.pyplot as plt

# ── Config ────────────────────────────────────────────────────────────────────
_ef_param_name  = 'Ab'
_ef_param_nom   = float(getattr(current_params, _ef_param_name))
_ef_param_init  = _ef_param_nom * 0.9    # start 10 % below nominal
_ef_lr          = 2e-2
_ef_num_epochs  = 300
_ef_num_batches = 1

# ── Fixed target (computed once) ──────────────────────────────────────────────
_ef_targets = {}
for _bidx in range(min(_ef_num_batches, len(dataloader))):
    _tb = jax.device_put(dataloader[_bidx].reshape(-1, len(track_fields)))
    _ef_targets[_bidx] = format_target_output(
        target_strategy.predict(current_params, _tb, track_fields, _bidx)
    )

# ── Loss: use true dEdx from tracks, only vary eField ────────────────────────
def loss_efield_only(log_efield, tracks_batch, target_out):
    param_value    = jnp.exp(log_efield)
    updated_params = current_params.replace(**{_ef_param_name: param_value})
    # tracks_batch already carries true dEdx — no modification needed
    pred_dist      = pred_strategy.predict(updated_params, tracks_batch, track_fields, 42)
    hit_loss, _    = loss_strategy.compute(updated_params, pred_dist, target_out)
    return hit_loss

vg_ef = value_and_grad(loss_efield_only)

# ── Optimizer ─────────────────────────────────────────────────────────────────
_ef_theta     = jnp.log(jnp.array(_ef_param_init, dtype=jnp.float32))
_ef_optimizer = optax.adam(_ef_lr)
_ef_opt_state = _ef_optimizer.init(_ef_theta)

# ── History ───────────────────────────────────────────────────────────────────
_ef_losses    = []
_ef_param_hist = []

# ── Training loop ─────────────────────────────────────────────────────────────
for _epoch in range(_ef_num_epochs):
    for _bidx in range(min(_ef_num_batches, len(dataloader))):
        _tracks_b = jax.device_put(dataloader[_bidx].reshape(-1, len(track_fields)))
        _target_b = _ef_targets[_bidx]

        _loss_val, _grad = vg_ef(_ef_theta, _tracks_b, _target_b)
        _upd, _ef_opt_state = _ef_optimizer.update(_grad, _ef_opt_state, _ef_theta)
        _ef_theta = optax.apply_updates(_ef_theta, _upd)

        _p_val = float(jnp.exp(_ef_theta))
        _ef_losses.append(float(_loss_val))
        _ef_param_hist.append(_p_val)

    if _epoch % 30 == 0 or _epoch == _ef_num_epochs - 1:
        print(f"Ep {_epoch:03d} | {_ef_param_name}={_p_val:.5f} "
              f"({_p_val/_ef_param_nom:.4f}×nom) | NLL={float(_loss_val):.4f}")

# ── Plot ──────────────────────────────────────────────────────────────────────
_steps = np.arange(len(_ef_losses))
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

ax1.semilogy(_steps, _ef_losses)
ax1.set_xlabel('Epoch'); ax1.set_title('Hit NLL')

ax2.plot(_steps, np.array(_ef_param_hist) / _ef_param_nom)
ax2.axhline(1.0, color='grey', linestyle=':', lw=0.8)
ax2.set_xlabel('Epoch'); ax2.set_title(f'{_ef_param_name} / nominal')

fig.suptitle(f'eField-only fit with true dEdx  (init = {_ef_param_init:.4f})')
fig.tight_layout()
plt.show()

print(f"\nFinal {_ef_param_name} = {float(jnp.exp(_ef_theta)):.6f}  "
      f"(nominal = {_ef_param_nom:.6f}, ratio = {float(jnp.exp(_ef_theta))/_ef_param_nom:.4f})")


In [8]:

# ═══════════════════════════════════════════════════════════
# PHASE 2b — Split (bifurcated) Student-t prior on per-segment dEdx
#
# Assigns independent degrees of freedom (nu_L, nu_R) and scales
# (scale_L, scale_R) to the left and right of the mode, joined
# continuously at loc with a common peak height.
#
# This is more flexible than the single-gamma skewed-t above:
#   - Left side: steeper / higher nu → sharp MIP cutoff
#   - Right side: heavier / lower nu → Landau-like delta-ray tail
#
# Exports: STUDENT_NU_L, STUDENT_NU_R, STUDENT_SCALE_L, STUDENT_SCALE_R,
#          split_t_logpdf_jax, student_t_nll (split version)
#          STUDENT_LOC is updated to the new fitted mode.
# ═══════════════════════════════════════════════════════════
import jax
import jax.numpy as jnp
import jax.scipy.stats as jstats
import optax
import numpy as np
import matplotlib.pyplot as plt

# ── JAX-native Split Student-t log-pdf ────────────────────────────────────────
def split_t_logpdf_jax(x, nu_L, nu_R, loc, scale_L, scale_R):
    """
    Split (bifurcated) Student-t log-pdf.

    The two halves share the same peak height at `loc` and are
    normalised so the total integral is 1.

    Parameters
    ----------
    x       : array — observed values
    nu_L    : scalar — degrees of freedom, left  half  (> 1)
    nu_R    : scalar — degrees of freedom, right half  (> 1)
    loc     : scalar — location (mode / join point)
    scale_L : scalar — scale for the left  half  (> 0)
    scale_R : scalar — scale for the right half  (> 0)
    """
    # log-normalisation constant of the standard Student-t kernel
    def log_c(nu):
        return (jax.scipy.special.gammaln((nu + 1.0) / 2.0)
                - jax.scipy.special.gammaln(nu / 2.0)
                - 0.5 * jnp.log(nu * jnp.pi))

    log_c_L = log_c(nu_L)
    log_c_R = log_c(nu_R)

    # Standardised residuals
    z_L = (x - loc) / scale_L
    z_R = (x - loc) / scale_R

    # Unnormalised kernel: subtract log_c so both sides have peak = 0
    log_K_L = jstats.t.logpdf(z_L, df=nu_L, loc=0.0, scale=1.0) - log_c_L
    log_K_R = jstats.t.logpdf(z_R, df=nu_R, loc=0.0, scale=1.0) - log_c_R

    # Half-integrals for total normalisation: Z = Z_L + Z_R
    Z_L = 0.5 * scale_L * jnp.exp(-log_c_L)
    Z_R = 0.5 * scale_R * jnp.exp(-log_c_R)
    log_Z = jnp.log(Z_L + Z_R)

    # Piecewise assembly — differentiable via jnp.where
    log_K = jnp.where(x < loc, log_K_L, log_K_R)
    return log_K - log_Z

# ── 1.  Load reference dEdx data (MIP region) ─────────────────────────────────
_split_data     = get_full_dedx_distrib(INPUT_FILE)
_split_data     = _split_data[(_split_data > 1.0) & (_split_data < 5.0)]
_split_data_jnp = jnp.array(_split_data)

# ── 2.  NLL loss ───────────────────────────────────────────────────────────────
def _split_loss_fn(p, x):
    nu_L    = jax.nn.softplus(p['nu_L_raw'])    + 1.0
    nu_R    = jax.nn.softplus(p['nu_R_raw'])    + 1.0
    scale_L = jax.nn.softplus(p['scale_L_raw'])
    scale_R = jax.nn.softplus(p['scale_R_raw'])
    loc     = p['loc']
    return -jnp.mean(split_t_logpdf_jax(x, nu_L=nu_L, nu_R=nu_R, loc=loc,
                                         scale_L=scale_L, scale_R=scale_R))

# ── 3.  Initialise — left steeper, right heavier ──────────────────────────────
_sp_std = float(jnp.std(_split_data_jnp))
_split_p0 = {
    'nu_L_raw':    jnp.array(5.0),
    'nu_R_raw':    jnp.array(1.0),
    'loc':         jnp.median(_split_data_jnp),
    'scale_L_raw': jnp.array(_sp_std * 0.5),
    'scale_R_raw': jnp.array(_sp_std * 1.5),
}

# ── 4.  Optimiser ─────────────────────────────────────────────────────────────
_split_opt   = optax.adam(0.05)
_split_state = _split_opt.init(_split_p0)

@jax.jit
def _split_step(p, s, x):
    loss, g = jax.value_and_grad(_split_loss_fn)(p, x)
    upd, ns = _split_opt.update(g, s, p)
    return optax.apply_updates(p, upd), ns, loss

# ── 5.  Training loop ──────────────────────────────────────────────────────────
_sp = _split_p0
for _ep in range(1000):
    _sp, _split_state, _sp_loss = _split_step(_sp, _split_state, _split_data_jnp)
    if _ep % 200 == 0:
        _nL = float(jax.nn.softplus(_sp['nu_L_raw']) + 1.0)
        _nR = float(jax.nn.softplus(_sp['nu_R_raw']) + 1.0)
        _sL = float(jax.nn.softplus(_sp['scale_L_raw']))
        _sR = float(jax.nn.softplus(_sp['scale_R_raw']))
        print(f"Ep {_ep:04d} | loss={float(_sp_loss):.4f} | "
              f"loc={float(_sp['loc']):.3f} | "
              f"nu_L={_nL:.2f} nu_R={_nR:.2f} | "
              f"sc_L={_sL:.4f} sc_R={_sR:.4f}")

# ── 6.  Extract final parameters ──────────────────────────────────────────────
final_nu_L    = jax.nn.softplus(_sp['nu_L_raw'])    + 1.0
final_nu_R    = jax.nn.softplus(_sp['nu_R_raw'])    + 1.0
final_scale_L = jax.nn.softplus(_sp['scale_L_raw'])
final_scale_R = jax.nn.softplus(_sp['scale_R_raw'])
final_loc     = _sp['loc']

print(f"\nFinal Split-t Fit:")
print(f"  Location (loc)  : {float(final_loc):.4f}")
print(f"  Left  — nu={float(final_nu_L):.3f}  scale={float(final_scale_L):.4f}")
print(f"  Right — nu={float(final_nu_R):.3f}  scale={float(final_scale_R):.4f}")

# ── 7.  Visual validation ─────────────────────────────────────────────────────
_x_sp  = np.linspace(1.0, 5.0, 500)
_x_spj = jnp.array(_x_sp)
_pdf_split = np.exp(np.array(split_t_logpdf_jax(
    _x_spj, nu_L=final_nu_L, nu_R=final_nu_R,
    loc=final_loc, scale_L=final_scale_L, scale_R=final_scale_R)))
plt.figure(figsize=(9, 4))
plt.hist(_split_data, bins=100, density=True, alpha=0.40,
         color='steelblue', label='Data (MIP region)')
plt.plot(_x_sp, _pdf_split,  'r-',  lw=2,
         label=f'Split-t  (ν_L={float(final_nu_L):.1f}, ν_R={float(final_nu_R):.1f})')
plt.xlabel('dEdx (MeV/cm)'); plt.ylabel('Density')
plt.title('Split Student-t vs Skewed Student-t fit to reference dEdx')
plt.legend(fontsize=9); plt.grid(True, alpha=0.3); plt.yscale('log')
plt.tight_layout(); plt.show()

# ── 8.  Export constants ──────────────────────────────────────────────────────
STUDENT_NU_L    = final_nu_L
STUDENT_NU_R    = final_nu_R
STUDENT_SCALE_L = final_scale_L
STUDENT_SCALE_R = final_scale_R
STUDENT_LOC     = final_loc   # update to split-t fitted mode

def student_t_nll(x, nu_L, nu_R, loc, scale_L, scale_R):
    """NLL penalty under the Split Student-t prior."""
    return -jnp.sum(split_t_logpdf_jax(
        x, nu_L=nu_L, nu_R=nu_R, loc=loc,
        scale_L=scale_L, scale_R=scale_R))


In [ ]:

# ═══════════════════════════════════════════════════════════
# Multi-batch joint fit: shared calibration param + per-batch per-segment dEdx
#
# Design:
#   - norm_param  : single scalar, shared across ALL batches (sigmoid-bounded)
#   - log_dedx_mean[b]: per-batch array, stored in _sj_dedx_cache[b]
#   - Separate Adam optimizers so dEdx arrays of different sizes are independent
#   - Each epoch: loop over every batch, update norm_param + that batch's dEdx
# ═══════════════════════════════════════════════════════════
import jax, jax.numpy as jnp, optax, numpy as np
import matplotlib.pyplot as plt
from jax import value_and_grad
from optimize.fit_params import map_norm_to_phys, map_phys_to_norm

# ── Config ────────────────────────────────────────────────────────────────────
_sj_param_name  = 'Ab'
_sj_param_nom   = float(getattr(current_params, _sj_param_name))
_sj_param_init  = _sj_param_nom * 1.1     # start 10 % above nominal
_sj_lr          = 1e-1
_sj_num_epochs  = 100
_sj_num_batches = 5
print(f"Multi-batch joint fit — {_sj_num_batches} batch(es), {_sj_num_epochs} epochs")

current_params = ref_params.replace(RESET_NOISE_CHARGE=0.2*ref_params.RESET_NOISE_CHARGE, UNCORRELATED_NOISE_CHARGE=0.2*ref_params.UNCORRELATED_NOISE_CHARGE)

# ── Pre-compute per-batch data (targets fixed at nominal params + true dEdx) ──
_sj_batches = []
for _bidx in range(_sj_num_batches):
    _tracks_b       = jax.device_put(dataloader[_bidx].reshape(-1, len(track_fields)))
    _target_b       = format_target_output(
        target_strategy.predict(current_params, _tracks_b, track_fields, _bidx)
    )
    _pids_np        = dataset.get_parent_segment_ids(_bidx)
    _pids_jax       = jax.device_put(jnp.array(_pids_np, dtype=jnp.int32))

    # Warm-start: init log_dedx from STUDENT_LOC
    _pv             = _pids_np[_pids_np >= 0]
    _vrows          = np.where(_pids_np >= 0)[0]
    _, _focc        = np.unique(_pv, return_index=True)
    _true_dedx      = np.array(_tracks_b)[_vrows[_focc], track_fields.index('dEdx')]
    _N_orig         = len(_true_dedx)
    _log_dedx0      = jnp.array(
        np.log(np.full(_N_orig, float(STUDENT_LOC))), dtype=jnp.float32
    )

    _sj_batches.append({
        'tracks':    _tracks_b,
        'target':    _target_b,
        'pids':      _pids_jax,
        'true_dedx': _true_dedx,
        'log_dedx':  _log_dedx0,
    })
    print(f"  Batch {_bidx}: {_N_orig} original segments, {len(_pids_np)} sub-steps")

# ── Shared calibration parameter (sigmoid-bounded) ────────────────────────────
_sj_norm_param  = map_phys_to_norm(
    jnp.array(_sj_param_init, dtype=jnp.float32), _sj_param_name
)
print(f"\nSigmoid init: {_sj_param_name} = {_sj_param_init:.6f}  "
      f"→  norm_param = {float(_sj_norm_param):.4f}")
print(f"  bounds: [{ranges[_sj_param_name]['min']:.4f}, {ranges[_sj_param_name]['max']:.4f}]")

# ── Optimizers ────────────────────────────────────────────────────────────────
_sj_opt_p   = optax.adam(_sj_lr)
_sj_state_p = _sj_opt_p.init(_sj_norm_param)

_sj_opt_d   = [optax.adam(_sj_lr) for _ in range(_sj_num_batches)]
_sj_state_d = [
    _sj_opt_d[b].init(_sj_batches[b]['log_dedx'])
    for b in range(_sj_num_batches)
]

# ── Loss function ─────────────────────────────────────────────────────────────
def _sj_loss(norm_param, log_dedx_mean, tracks_batch, parent_ids, target_out):
    param_value    = map_norm_to_phys(norm_param, _sj_param_name)
    safe_ids       = jnp.clip(parent_ids, 0, log_dedx_mean.shape[0] - 1)
    dedx_sample    = jnp.maximum(jnp.exp(jnp.take(log_dedx_mean, safe_ids)), 1e-3)

    dedx_idx       = track_fields.index('dEdx')
    dE_idx         = track_fields.index('dE')
    dx_idx         = track_fields.index('dx')
    updated_tracks = tracks_batch.at[:, dedx_idx].set(dedx_sample)
    updated_tracks = updated_tracks.at[:, dE_idx].set(dedx_sample * updated_tracks[:, dx_idx])

    updated_params = current_params.replace(**{_sj_param_name: param_value})
    pred_dist      = pred_strategy.predict(updated_params, updated_tracks, track_fields, 42)
    hit_loss, _    = loss_strategy.compute(updated_params, pred_dist, target_out)

    dedx_loss  = student_t_nll(dedx_sample, STUDENT_NU_L, STUDENT_NU_R, STUDENT_LOC, STUDENT_SCALE_L, STUDENT_SCALE_R)* 0.01
    total_loss = hit_loss + dedx_loss
    return total_loss, {'hit_loss': hit_loss, 'dedx_loss': dedx_loss}

_sj_vg = value_and_grad(_sj_loss, argnums=(0, 1), has_aux=True)

# ── History ───────────────────────────────────────────────────────────────────
_sj_losses          = []
_sj_hit_losses      = []
_sj_dedx_losses     = []
_sj_param_hist      = []
_sj_dedx_dist       = []
_sj_dedx_snapshots  = []
_sj_snapshot_every  = max(1, _sj_num_epochs // 10)

# ── Training loop ─────────────────────────────────────────────────────────────
for _epoch in range(_sj_num_epochs):
    _epoch_loss      = 0.0
    _epoch_hit_loss  = 0.0
    _epoch_dedx_loss = 0.0
    _epoch_dedxd     = 0.0

    for _b, _bd in enumerate(_sj_batches):
        ((_loss_val, _aux), (_g_param, _g_dedx)) = _sj_vg(
            _sj_norm_param, _bd['log_dedx'],
            _bd['tracks'], _bd['pids'], _bd['target']
        )

        _upd_p, _sj_state_p = _sj_opt_p.update(_g_param, _sj_state_p)
        _sj_norm_param = optax.apply_updates(_sj_norm_param, _upd_p)

        _upd_d, _sj_state_d[_b] = _sj_opt_d[_b].update(_g_dedx, _sj_state_d[_b])
        _bd['log_dedx'] = optax.apply_updates(_bd['log_dedx'], _upd_d)

        _fitted_dedx      = np.exp(np.array(_bd['log_dedx']))
        _epoch_loss      += float(_loss_val)
        _epoch_hit_loss  += float(_aux['hit_loss'])
        _epoch_dedx_loss += float(_aux['dedx_loss'])
        _epoch_dedxd     += float(np.mean(np.abs(_fitted_dedx - _bd['true_dedx'])))

    _mean_loss      = _epoch_loss      / _sj_num_batches
    _mean_hit_loss  = _epoch_hit_loss  / _sj_num_batches
    _mean_dedx_loss = _epoch_dedx_loss / _sj_num_batches
    _mean_ddist     = _epoch_dedxd     / _sj_num_batches
    _p_val          = float(map_norm_to_phys(_sj_norm_param, _sj_param_name))

    _sj_losses.append(_mean_loss)
    _sj_hit_losses.append(_mean_hit_loss)
    _sj_dedx_losses.append(_mean_dedx_loss)
    _sj_param_hist.append(_p_val)
    _sj_dedx_dist.append(_mean_ddist)

    if _epoch % _sj_snapshot_every == 0 or _epoch == _sj_num_epochs - 1:
        _snap_fitted = np.concatenate([np.exp(np.array(_bd['log_dedx'])) for _bd in _sj_batches])
        _snap_true   = np.concatenate([_bd['true_dedx']                   for _bd in _sj_batches])
        _sj_dedx_snapshots.append((_epoch, _snap_fitted.copy(), _snap_true.copy()))

    if _epoch % 30 == 0 or _epoch == _sj_num_epochs - 1:
        print(f"Ep {_epoch:03d} | {_sj_param_name}={_p_val:.5f} "
              f"({_p_val/_sj_param_nom:.4f}×nom) | "
              f"Total={_mean_loss:.4f} | HitNLL={_mean_hit_loss:.4f} | "
              f"dEdxNLL={_mean_dedx_loss:.4f} | dEdx_dist={_mean_ddist:.4f}")

# ── Summary plots ─────────────────────────────────────────────────────────────
_steps = np.arange(len(_sj_losses))
fig, axes = plt.subplots(1, 4, figsize=(17, 4))

# Combined loss breakdown
axes[0].plot(_steps, _sj_losses,      label='Total',    color='black',     lw=1.8)
axes[0].plot(_steps, _sj_hit_losses,  label='Hit NLL',  color='steelblue', lw=1.2, ls='--')
axes[0].plot(_steps, _sj_dedx_losses, label='dEdx NLL', color='tomato',    lw=1.2, ls=':')
axes[0].set_title('Loss components (mean over batches)')
axes[0].set_xlabel('Epoch')
axes[0].legend(fontsize=8)

# Hit NLL alone (dominant term — finer scale)
axes[1].plot(_steps, _sj_hit_losses, color='steelblue')
axes[1].set_title('Hit NLL (mean over batches)')
axes[1].set_xlabel('Epoch')

# dEdx regularisation NLL
axes[2].plot(_steps, _sj_dedx_losses, color='tomato')
axes[2].set_title('dEdx prior NLL × weight (mean over batches)')
axes[2].set_xlabel('Epoch')

# Calibration parameter
axes[3].plot(_steps, np.array(_sj_param_hist) / _sj_param_nom, color='tab:green')
axes[3].axhline(1.0, color='grey', linestyle=':', lw=0.8)
axes[3].set_title(f'{_sj_param_name} / nominal')
axes[3].set_xlabel('Epoch')

fig.suptitle(f'Multi-batch joint fit: {_sj_param_name} + per-batch dEdx')
fig.tight_layout(); plt.show()

_final_p = float(map_norm_to_phys(_sj_norm_param, _sj_param_name))
print(f"\nFinal {_sj_param_name} = {_final_p:.6f}  "
      f"(nominal = {_sj_param_nom:.6f}, ratio = {_final_p/_sj_param_nom:.4f})")
print(f"Final loss breakdown (last epoch):  "
      f"Total={_sj_losses[-1]:.4f}  |  "
      f"HitNLL={_sj_hit_losses[-1]:.4f}  |  "
      f"dEdxNLL={_sj_dedx_losses[-1]:.4f}")


In [11]:

# ═══════════════════════════════════════════════════════════
# Prior-weight sweep: joint fit of lifetime + dEdx
#
# Re-runs the same multi-batch fit from scratch for each value in
# PRIOR_WEIGHT_SWEEP, then overlays:
#   - lifetime / nominal  convergence trace
#   - mean dEdx MAE (|fitted − true|) trace
#   - final summary bar chart
# ═══════════════════════════════════════════════════════════

_sj_param_name  = 'kb'
_sj_param_nom   = float(getattr(current_params, _sj_param_name))
_sj_param_init  = _sj_param_nom * 1.1     # start 10 % above nominal
_sj_lr          = 1e-1
_sj_num_epochs  = 200
_sj_num_batches = 5
print(f"Multi-batch joint fit — {_sj_num_batches} batch(es), {_sj_num_epochs} epochs")

import matplotlib.cm as cm

# ── Sweep config ──────────────────────────────────────────────────────────────
PRIOR_WEIGHT_SWEEP = [0.0, 1e-3, 0.01, 0.1, 1.0, 10.0]   # multiply student_t_nll
_sw_num_epochs     = _sj_num_epochs    # reuse same count
_sw_lr             = _sj_lr

# ── Helper: run one full joint fit and return history ─────────────────────────
def _run_joint_fit(prior_weight):
    """Return (param_hist_ratio, dedx_mae_hist) for one prior weight."""

    # Reset per-batch dEdx to warm-start value (same as the original cell)
    _batches = []
    for _bidx in range(_sj_num_batches):
        _bd = _sj_batches[_bidx]   # re-use pre-computed tracks / targets
        _N  = len(_bd['true_dedx'])
        _batches.append({
            'tracks':    _bd['tracks'],
            'target':    _bd['target'],
            'pids':      _bd['pids'],
            'true_dedx': _bd['true_dedx'],
            'log_dedx':  jnp.array(np.log(np.full(_N, float(STUDENT_LOC))),
                                   dtype=jnp.float32),
        })

    # Reset shared param
    _norm = map_phys_to_norm(
        jnp.array(_sj_param_init, dtype=jnp.float32), _sj_param_name
    )

    # Separate optimizers per run
    _opt_p   = optax.adam(_sw_lr)
    _state_p = _opt_p.init(_norm)
    _opt_d   = [optax.adam(_sw_lr) for _ in range(_sj_num_batches)]
    _state_d = [_opt_d[b].init(_batches[b]['log_dedx']) for b in range(_sj_num_batches)]

    def _loss_fn(norm_param, log_dedx, tracks_b, pids, target):
        param_value    = map_norm_to_phys(norm_param, _sj_param_name)
        safe_ids       = jnp.clip(pids, 0, log_dedx.shape[0] - 1)
        dedx_s         = jnp.maximum(jnp.exp(jnp.take(log_dedx, safe_ids)), 1e-3)

        dedx_idx       = track_fields.index('dEdx')
        dE_idx         = track_fields.index('dE')
        dx_idx         = track_fields.index('dx')
        upd_tracks     = tracks_b.at[:, dedx_idx].set(dedx_s)
        upd_tracks     = upd_tracks.at[:, dE_idx].set(dedx_s * upd_tracks[:, dx_idx])

        upd_params     = current_params.replace(**{_sj_param_name: param_value})
        pred           = pred_strategy.predict(upd_params, upd_tracks, track_fields, 42)
        hit_loss, _    = loss_strategy.compute(upd_params, pred, target)

        dedx_nll       = student_t_nll(
            dedx_s, STUDENT_NU_L, STUDENT_NU_R, STUDENT_LOC,
            STUDENT_SCALE_L, STUDENT_SCALE_R
        )
        return hit_loss + prior_weight * dedx_nll

    _vg = jax.value_and_grad(_loss_fn, argnums=(0, 1))

    _param_hist = []
    _mae_hist   = []

    for _ep in range(_sw_num_epochs):
        _ep_param  = 0.0
        _ep_mae    = 0.0

        for _b, _bd in enumerate(_batches):
            _loss_val, (_g_p, _g_d) = _vg(
                _norm, _bd['log_dedx'], _bd['tracks'], _bd['pids'], _bd['target']
            )
            _upd_p, _state_p = _opt_p.update(_g_p, _state_p)
            _norm = optax.apply_updates(_norm, _upd_p)

            _upd_d, _state_d[_b] = _opt_d[_b].update(_g_d, _state_d[_b])
            _bd['log_dedx'] = optax.apply_updates(_bd['log_dedx'], _upd_d)

            _ep_param += float(map_norm_to_phys(_norm, _sj_param_name))
            _ep_mae   += float(np.mean(np.abs(
                np.exp(np.array(_bd['log_dedx'])) - _bd['true_dedx']
            )))

        _param_hist.append(_ep_param / _sj_num_batches / _sj_param_nom)
        _mae_hist.append(_ep_mae / _sj_num_batches)

    final_p = float(map_norm_to_phys(_norm, _sj_param_name))
    return _param_hist, _mae_hist, final_p

# ── Run sweep ─────────────────────────────────────────────────────────────────
_sw_results = {}   # prior_weight → (param_hist_ratio, mae_hist, final_p)

for _pw in PRIOR_WEIGHT_SWEEP:
    print(f"Running prior_weight={_pw:.3g} ...", end='  ', flush=True)
    _ph, _mh, _fp = _run_joint_fit(_pw)
    _sw_results[_pw] = (_ph, _mh, _fp)
    print(f"final {_sj_param_name}={_fp:.5f}  (×{_ph[-1]:.4f} nom)  "
          f"final MAE={_mh[-1]:.4f}")

# ── Comparison plots ──────────────────────────────────────────────────────────
_n_sw   = len(PRIOR_WEIGHT_SWEEP)
_colors = cm.plasma(np.linspace(0.05, 0.95, _n_sw))
_epochs = np.arange(_sw_num_epochs)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle(
    f"Prior-weight sweep — joint fit: {_sj_param_name} + per-segment dEdx\n"
    f"({_sj_num_batches} batches, {_sw_num_epochs} epochs, "
    f"lr={_sw_lr}, init={_sj_param_init:.4g}  nom={_sj_param_nom:.4g})",
    fontsize=11
)

for (_pw, (_ph, _mh, _fp)), _c in zip(_sw_results.items(), _colors):
    _lbl = f"w={_pw:.3g}"
    axes[0].plot(_epochs, _ph, color=_c, lw=1.8, label=_lbl)
    axes[1].plot(_epochs, _mh, color=_c, lw=1.8, label=_lbl)

axes[0].axhline(1.0, color='k', ls='--', lw=1.0, label='Nominal (truth)')
axes[0].set_title(f'{_sj_param_name} / nominal')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Ratio')
axes[0].legend(fontsize=8); axes[0].grid(True, alpha=0.3)

axes[1].set_title('dEdx MAE  (mean |fitted − true| MeV/cm)')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('MAE (MeV/cm)')
axes[1].legend(fontsize=8); axes[1].grid(True, alpha=0.3)

# Summary bar: final lifetime ratio & dEdx MAE per weight
_x  = np.arange(_n_sw)
_w  = 0.35
_final_ratios = [_sw_results[pw][0][-1] for pw in PRIOR_WEIGHT_SWEEP]
_final_maes   = [_sw_results[pw][1][-1] for pw in PRIOR_WEIGHT_SWEEP]
_xlbls        = [f"{pw:.3g}" for pw in PRIOR_WEIGHT_SWEEP]

ax2b = axes[2].twinx()
axes[2].bar(_x - _w/2, _final_ratios, width=_w, color='steelblue', alpha=0.75,
            label=f'{_sj_param_name} ratio')
ax2b.bar(_x + _w/2, _final_maes,   width=_w, color='tomato',    alpha=0.75,
         label='dEdx MAE')
axes[2].axhline(1.0, color='steelblue', ls='--', lw=0.9)
axes[2].set_xticks(_x); axes[2].set_xticklabels(_xlbls, rotation=30)
axes[2].set_xlabel('Prior weight'); axes[2].set_ylabel(f'{_sj_param_name} / nominal', color='steelblue')
ax2b.set_ylabel('Final dEdx MAE (MeV/cm)', color='tomato')
axes[2].set_title('Final values vs prior weight')
axes[2].legend(loc='upper left', fontsize=8)
ax2b.legend(loc='upper right', fontsize=8)
axes[2].grid(axis='y', alpha=0.3)

fig.tight_layout()
plt.show()


In [9]:
1

In [11]:

# ═══════════════════════════════════════════════════════════
# Loss scan over the calibration parameter
# with the fitted dEdx values from the joint fit frozen.
#
# For each value of _sj_param_name on a grid over [min, max],
# we evaluate: total loss, hit NLL, and dEdx prior NLL
# across all batches — using _sj_batches[b]['log_dedx'] as-is.
# ═══════════════════════════════════════════════════════════
import numpy as np
import jax.numpy as jnp
import matplotlib.pyplot as plt

# ── Scan configuration ────────────────────────────────────────────────────────
_scan_param  = _sj_param_name
_scan_n_pts  = 40
_scan_lo     = ranges[_scan_param]['min']
# _scan_lo     = 0.05
_scan_hi     = ranges[_scan_param]['max']
_scan_values = np.linspace(_scan_lo, _scan_hi, _scan_n_pts)
_scan_nom    = float(getattr(ref_params, _scan_param))

print(f"Scanning {_scan_param} over [{_scan_lo:.4f}, {_scan_hi:.4f}] "
      f"({_scan_n_pts} points), nominal = {_scan_nom:.4f}")
print(f"Using fitted log_dedx from {_sj_num_batches} batches (frozen).")

# ── No-gradient eval function ─────────────────────────────────────────────────
def _eval_scan(param_val):
    """Return (total, hit_nll, dedx_nll) mean over all batches."""
    _updated = current_params.replace(**{_scan_param: float(param_val)})
    _dedx_i  = list(track_fields).index('dEdx')
    _dE_i    = list(track_fields).index('dE')
    _dx_i    = list(track_fields).index('dx')
    _tot = 0.0; _hit = 0.0; _reg = 0.0
    for _bd in _sj_batches:
        _safe  = jnp.clip(_bd['pids'], 0, _bd['log_dedx'].shape[0] - 1)
        _dedxs = jnp.maximum(jnp.exp(jnp.take(_bd['log_dedx'], _safe)), 1e-3)
        _t     = _bd['tracks'].at[:, _dedx_i].set(_dedxs)
        _t     = _t.at[:, _dE_i].set(_dedxs * _t[:, _dx_i])
        _pred  = pred_strategy.predict(_updated, _t, track_fields, 42)
        _hl, _ = loss_strategy.compute(_updated, _pred, _bd['target'])
        _dl    = student_t_nll(_dedxs, STUDENT_NU_L, STUDENT_NU_R, STUDENT_LOC, STUDENT_SCALE_L, STUDENT_SCALE_R) * 0.05
        _tot  += float(_hl + _dl)
        _hit  += float(_hl)
        _reg  += float(_dl)
    n = len(_sj_batches)
    return _tot / n, _hit / n, _reg / n

# ── Run scan ──────────────────────────────────────────────────────────────────
_scan_total = np.zeros(_scan_n_pts)
_scan_hit   = np.zeros(_scan_n_pts)
_scan_dedx  = np.zeros(_scan_n_pts)

for _k, _v in enumerate(_scan_values):
    _scan_total[_k], _scan_hit[_k], _scan_dedx[_k] = _eval_scan(_v)
    if _k % 10 == 0 or _k == _scan_n_pts - 1:
        print(f"  [{_k+1:3d}/{_scan_n_pts}]  {_scan_param}={_v:.4f}"
              f"  total={_scan_total[_k]:.4f}"
              f"  hit={_scan_hit[_k]:.4f}"
              f"  dedx={_scan_dedx[_k]:.4f}")

# ── Retrieve fitted value from the last epoch ─────────────────────────────────
_fitted_val = _sj_param_hist[-1]

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle(f'Loss scan: {_scan_param}  (dEdx frozen at fitted values)', fontsize=12)

for _ax, _y, _lbl, _c in zip(
    axes,
    [_scan_total, _scan_hit, _scan_dedx],
    ['Total loss (mean)', 'Hit NLL (mean)', 'dEdx prior NLL × w (mean)'],
    ['black', 'steelblue', 'tomato'],
):
    _ax.plot(_scan_values, _y, color=_c, lw=1.8)
    _ax.axvline(_scan_nom,   color='grey',      lw=1.0, ls='--', label=f'Nominal ({_scan_nom:.4f})')
    _ax.axvline(_fitted_val, color='darkorange', lw=1.2, ls='-',
                label=f'Fitted ({_fitted_val:.4f})')
    _mi = int(np.argmin(_y))
    _ax.axvline(_scan_values[_mi], color=_c, lw=1.0, ls=':', alpha=0.7,
                label=f'Scan min ({_scan_values[_mi]:.4f})')
    _ax.set_xlabel(_scan_param); _ax.set_title(_lbl); _ax.legend(fontsize=7)

fig.tight_layout(); plt.show()

print(f"\nScan minimum (total)  : {_scan_param} = {_scan_values[np.argmin(_scan_total)]:.4f}"
      f"  (nominal={_scan_nom:.4f}, fitted={_fitted_val:.4f})")
print(f"Scan minimum (hit NLL): {_scan_param} = {_scan_values[np.argmin(_scan_hit)]:.4f}")


In [22]:

# ═══════════════════════════════════════════════════════════
# Fitted dEdx distribution vs truth — multi-panel view
#
# Panels:
#   Row 1: per-batch 1-D histograms (fitted vs true vs prior PDF)
#   Row 2: per-batch 2-D hexbin (fitted vs true)
#   Row 3: aggregate across all batches — 1-D, 2-D hexbin, residual
# ═══════════════════════════════════════════════════════════
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats as _scipy_stats

# ── Collect per-batch arrays ───────────────────────────────────────────────────
_fit_all   = []   # concatenated fitted dEdx over all batches
_true_all  = []   # concatenated true dEdx over all batches
_per_batch = []   # list of (fitted_np, true_np) per batch

for _bd in _sj_batches:
    _f = np.exp(np.asarray(_bd['log_dedx']))
    _t = np.asarray(_bd['true_dedx'])
    _per_batch.append((_f, _t))
    _fit_all.append(_f)
    _true_all.append(_t)

_fit_all  = np.concatenate(_fit_all)
_true_all = np.concatenate(_true_all)

# ── Prior PDF on a grid ────────────────────────────────────────────────────────
_x_pdf   = np.linspace(0.2, 5.0, 300)
_prior_pdf = np.exp(np.array(split_t_logpdf_jax(
    jnp.array(_x_pdf),
    nu_L=STUDENT_NU_L, nu_R=STUDENT_NU_R,
    loc=STUDENT_LOC,
    scale_L=STUDENT_SCALE_L, scale_R=STUDENT_SCALE_R,
)))

_BINS    = np.linspace(0.2, 5.0, 50)
_HB_LO, _HB_HI = 0.3, 4.5   # hexbin axis range

# ══════════════════════════════════════════════════════════════════════════════
# Row 1: per-batch 1-D histograms
# ══════════════════════════════════════════════════════════════════════════════
_nb = len(_sj_batches)
fig1, axes1 = plt.subplots(1, _nb, figsize=(4 * _nb, 4), sharey=False)
if _nb == 1:
    axes1 = [axes1]
fig1.suptitle('Per-batch dEdx: fitted vs true vs prior', fontsize=12)

for _b, ((_f, _t), _ax) in enumerate(zip(_per_batch, axes1)):
    _ax.hist(_t, bins=_BINS, density=True, alpha=0.5, color='steelblue', label='True')
    _ax.hist(_f, bins=_BINS, density=True, alpha=0.5, color='tomato',    label='Fitted')
    _ax.plot(_x_pdf, _prior_pdf, 'k--', lw=1.2, label='Split-t prior')
    _ax.axvline(float(STUDENT_LOC), color='k', lw=0.8, ls=':')
    _ax.set_xlabel('dEdx (MeV/cm)'); _ax.set_title(f'Batch {_b}  (n={len(_f)})')
    _ax.legend(fontsize=7)

fig1.tight_layout(); plt.show()

# ══════════════════════════════════════════════════════════════════════════════
# Row 2: per-batch 2-D hexbin (fitted vs true)
# ══════════════════════════════════════════════════════════════════════════════
fig2, axes2 = plt.subplots(1, _nb, figsize=(4 * _nb, 4))
if _nb == 1:
    axes2 = [axes2]
fig2.suptitle('Per-batch hexbin: fitted vs true dEdx', fontsize=12)

for _b, ((_f, _t), _ax) in enumerate(zip(_per_batch, axes2)):
    _hb = _ax.hexbin(_t, _f, gridsize=25, cmap='viridis',
                     extent=[_HB_LO, _HB_HI, _HB_LO, _HB_HI], mincnt=1)
    _ax.plot([_HB_LO, _HB_HI], [_HB_LO, _HB_HI], 'r--', lw=1, label='y=x')
    plt.colorbar(_hb, ax=_ax, label='count')
    _mae = float(np.mean(np.abs(_f - _t)))
    _ax.set_xlabel('True dEdx (MeV/cm)'); _ax.set_ylabel('Fitted dEdx (MeV/cm)')
    _ax.set_title(f'Batch {_b}  MAE={_mae:.3f}')
    _ax.legend(fontsize=7)

fig2.tight_layout(); plt.show()

# ══════════════════════════════════════════════════════════════════════════════
# Row 3: aggregate — 1-D overlay, hexbin, residual histogram
# ══════════════════════════════════════════════════════════════════════════════
_resid_all = _fit_all - _true_all

fig3, axes3 = plt.subplots(1, 3, figsize=(14, 4))
fig3.suptitle('All batches aggregated', fontsize=12)

# 1-D overlay
axes3[0].hist(_true_all, bins=_BINS, density=True, alpha=0.5, color='steelblue', label='True')
axes3[0].hist(_fit_all,  bins=_BINS, density=True, alpha=0.5, color='tomato',    label='Fitted')
axes3[0].plot(_x_pdf, _prior_pdf, 'k--', lw=1.2, label='Split-t prior')
axes3[0].axvline(float(STUDENT_LOC), color='k', lw=0.8, ls=':')
axes3[0].set_xlabel('dEdx (MeV/cm)'); axes3[0].set_title(f'Aggregate 1-D  (n={len(_fit_all)})')
axes3[0].legend(fontsize=8)

# Hexbin
_hb2 = axes3[1].hexbin(_true_all, _fit_all, gridsize=35, cmap='viridis',
                        extent=[_HB_LO, _HB_HI, _HB_LO, _HB_HI], mincnt=1)
axes3[1].plot([_HB_LO, _HB_HI], [_HB_LO, _HB_HI], 'r--', lw=1)
plt.colorbar(_hb2, ax=axes3[1], label='count')
_mae_all = float(np.mean(np.abs(_resid_all)))
axes3[1].set_xlabel('True dEdx (MeV/cm)'); axes3[1].set_ylabel('Fitted dEdx (MeV/cm)')
axes3[1].set_title(f'Aggregate hexbin  MAE={_mae_all:.3f}')

# Residual histogram
_r_lo, _r_hi = np.percentile(_resid_all, [1, 99])
_r_lim = max(abs(_r_lo), abs(_r_hi)) * 1.2
_rbins  = np.linspace(-_r_lim, _r_lim, 50)
axes3[2].hist(_resid_all, bins=_rbins, density=True, color='mediumpurple', alpha=0.75)
axes3[2].axvline(0, color='k', lw=0.8, ls='--')
axes3[2].axvline(float(np.mean(_resid_all)), color='tomato', lw=1.2,
                 label=f'mean={np.mean(_resid_all):.3f}')
axes3[2].set_xlabel('Fitted − True (MeV/cm)'); axes3[2].set_ylabel('Density')
axes3[2].set_title(f'Residuals  std={float(np.std(_resid_all)):.3f}  MAE={_mae_all:.3f}')
axes3[2].legend(fontsize=8)

fig3.tight_layout(); plt.show()

print(f"Aggregate stats  —  n={len(_fit_all)}")
print(f"  Fitted : mean={_fit_all.mean():.4f}  std={_fit_all.std():.4f}  median={np.median(_fit_all):.4f}")
print(f"  True   : mean={_true_all.mean():.4f}  std={_true_all.std():.4f}  median={np.median(_true_all):.4f}")
print(f"  Resid  : mean={_resid_all.mean():.4f}  std={_resid_all.std():.4f}  MAE={_mae_all:.4f}")


In [13]:

# ═══════════════════════════════════════════════════════════
# Fitted dEdx dependency on drift distance
#
# Drift distance = |x_mid| = |(x_start + x_end) / 2|
# (module0: drift along x; cathode at x=0, anodes at ±~30 cm)
#
# Panels:
#   1. Scatter: fitted dEdx vs drift distance (coloured by true dEdx)
#   2. 2-D hexbin: drift dist vs fitted dEdx
#   3. Profile (mean ± std) of fitted dEdx vs drift distance
#   4. Profile of residual (fitted − true) vs drift distance
# ═══════════════════════════════════════════════════════════
import numpy as np
import matplotlib.pyplot as plt

_tf_list = list(track_fields)   # plain list for .index()

def _get_field(tracks_np, field, default=np.nan):
    try:
        return tracks_np[:, _tf_list.index(field)]
    except ValueError:
        return np.full(tracks_np.shape[0], default)

# ── Gather drift distance, fitted dEdx and true dEdx per segment ──────────────
_drift_all  = []
_fit_all_d  = []
_true_all_d = []

for _b, _bd in enumerate(_sj_batches):
    _f = np.exp(np.asarray(_bd['log_dedx']))
    _t = np.asarray(_bd['true_dedx'])
    _tracks_np = np.asarray(_bd['tracks'])

    # Build seg_idx → tracks row mapping (same logic as the outlier cell)
    _pids_np = np.asarray(_bd['pids'])
    _valid   = _pids_np >= 0
    _pv      = _pids_np[_valid]
    _vrows   = np.where(_valid)[0]
    _, _focc = np.unique(_pv, return_index=True)
    _seg_rows = _vrows[_focc]   # (N_seg,)

    _rows = _seg_rows[:len(_f)]   # guard length

    xs  = _get_field(_tracks_np, 'z_start')[_rows]
    xe  = _get_field(_tracks_np, 'z_end')[_rows]

    # If x fields are missing, fall back to z (some geometries drift along z)
    if np.all(np.isnan(xs)):
        xs = _get_field(_tracks_np, 'z_start')[_rows]
        xe = _get_field(_tracks_np, 'z_end')[_rows]

    x_mid  = 0.5 * (xs + xe)
    drift  = np.abs(x_mid)

    _drift_all.append(drift)
    _fit_all_d.append(_f)
    _true_all_d.append(_t)

_drift_all  = np.concatenate(_drift_all)
_fit_all_d  = np.concatenate(_fit_all_d)
_true_all_d = np.concatenate(_true_all_d)
_resid_d    = _fit_all_d - _true_all_d

# Remove NaN segments (missing x fields)
_ok = np.isfinite(_drift_all) & np.isfinite(_fit_all_d)
_drift_all  = _drift_all[_ok]
_fit_all_d  = _fit_all_d[_ok]
_true_all_d = _true_all_d[_ok]
_resid_d    = _resid_d[_ok]

print(f"Segments with valid drift distance: {_ok.sum()} / {len(_ok)}")
print(f"Drift range: [{_drift_all.min():.2f}, {_drift_all.max():.2f}] cm")

# ── Profile helper ────────────────────────────────────────────────────────────
def _profile(x, y, bins):
    """Mean ± std in equal-width bins of x."""
    idx = np.digitize(x, bins)
    cx, means, stds, counts = [], [], [], []
    for i in range(1, len(bins)):
        sel = idx == i
        if sel.sum() < 2:
            continue
        cx.append(0.5 * (bins[i-1] + bins[i]))
        means.append(np.mean(y[sel]))
        stds.append(np.std(y[sel]))
        counts.append(sel.sum())
    return np.array(cx), np.array(means), np.array(stds), np.array(counts)

_DRIFT_BINS = np.linspace(0, _drift_all.max() * 1.02, 25)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Fitted dEdx vs drift distance', fontsize=13, fontweight='bold')

# ── Panel 1: scatter coloured by true dEdx ────────────────────────────────────
ax = axes[0, 0]
_sc = ax.scatter(_drift_all, _fit_all_d, c=_true_all_d,
                 cmap='plasma', s=4, alpha=0.4, vmin=0.5, vmax=4.0)
plt.colorbar(_sc, ax=ax, label='True dEdx (MeV/cm)')
ax.set_xlabel('Drift distance (cm)')
ax.set_ylabel('Fitted dEdx (MeV/cm)')
ax.set_title('Scatter: fitted dEdx (coloured by true dEdx)')
ax.set_ylim(0.2, 5.0)
ax.grid(True, alpha=0.3)

# ── Panel 2: 2-D hexbin ───────────────────────────────────────────────────────
ax = axes[0, 1]
_hb = ax.hexbin(_drift_all, _fit_all_d, gridsize=35, cmap='viridis', mincnt=1,
                extent=[0, _drift_all.max() * 1.02, 0.2, 5.0])
plt.colorbar(_hb, ax=ax, label='count')
ax.set_xlabel('Drift distance (cm)')
ax.set_ylabel('Fitted dEdx (MeV/cm)')
ax.set_title('Hexbin: fitted dEdx vs drift distance')
ax.grid(True, alpha=0.2)

# ── Panel 3: profile of fitted dEdx ──────────────────────────────────────────
ax = axes[1, 0]
_cx, _means, _stds, _cnts = _profile(_drift_all, _fit_all_d, _DRIFT_BINS)
ax.fill_between(_cx, _means - _stds, _means + _stds, alpha=0.25, color='tomato', label='±1σ')
ax.plot(_cx, _means, 'o-', color='tomato', lw=1.5, ms=4, label='Mean fitted dEdx')
# True dEdx profile for reference
_, _means_t, _stds_t, _ = _profile(_drift_all, _true_all_d, _DRIFT_BINS)
ax.fill_between(_cx, _means_t - _stds_t, _means_t + _stds_t, alpha=0.18, color='steelblue')
ax.plot(_cx, _means_t, 's--', color='steelblue', lw=1.2, ms=4, label='Mean true dEdx')
ax.set_xlabel('Drift distance (cm)')
ax.set_ylabel('dEdx (MeV/cm)')
ax.set_title('Profile: mean ± std of fitted & true dEdx')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

# ── Panel 4: residual profile ─────────────────────────────────────────────────
ax = axes[1, 1]
_cx_r, _means_r, _stds_r, _cnts_r = _profile(_drift_all, _resid_d, _DRIFT_BINS)
ax.fill_between(_cx_r, _means_r - _stds_r, _means_r + _stds_r,
                alpha=0.25, color='mediumpurple', label='±1σ')
ax.plot(_cx_r, _means_r, 'o-', color='mediumpurple', lw=1.5, ms=4, label='Mean residual')
ax.axhline(0, color='k', lw=1, ls='--')
ax.set_xlabel('Drift distance (cm)')
ax.set_ylabel('Fitted − True dEdx (MeV/cm)')
ax.set_title('Residual profile vs drift distance')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

# Annotate count per bin
for _x, _c in zip(_cx_r, _cnts_r):
    ax.text(_x, ax.get_ylim()[0], f'{_c}', fontsize=5, ha='center', va='bottom',
            color='grey', rotation=90)

fig.tight_layout()
plt.show()


In [22]:

# ═══════════════════════════════════════════════════════════════════════════════
# Outlier analysis: reco dEdx stuck near MPV while true dEdx is far away
#
# "Stuck-at-MPV" outlier definition:
#   |fitted - MPV| < RECO_BAND  (within a narrow band around the prior mode)
#   |true   - MPV| > TRUE_BAND  (true value is outside that band)
#
# Tunable thresholds — adjust to taste
RECO_BAND = 0.15   # MeV/cm: how close fitted must be to MPV
TRUE_BAND = 0.5    # MeV/cm: how far true must be from MPV

_mpv = float(STUDENT_LOC)

# ── Collect per-batch outlier indices ──────────────────────────────────────────
_outlier_reco  = []   # fitted values of outliers
_outlier_true  = []   # true values of outliers
_outlier_batch = []   # which batch
_outlier_idx   = []   # index within that batch

for _b, (_f, _t) in enumerate(_per_batch):
    _mask = (np.abs(_f - _mpv) < RECO_BAND) & (np.abs(_t - _mpv) > TRUE_BAND)
    _outlier_reco.append(_f[_mask])
    _outlier_true.append(_t[_mask])
    _outlier_batch.extend([_b] * int(_mask.sum()))
    _outlier_idx.extend(np.where(_mask)[0].tolist())

_outlier_reco  = np.concatenate(_outlier_reco)
_outlier_true  = np.concatenate(_outlier_true)
_outlier_batch = np.array(_outlier_batch)

_n_outliers = len(_outlier_reco)
_n_total    = len(_fit_all)
print(f"Stuck-at-MPV outliers: {_n_outliers} / {_n_total}  ({100*_n_outliers/_n_total:.1f}%)")
print(f"  RECO_BAND={RECO_BAND} MeV/cm around MPV={_mpv:.3f}")
print(f"  TRUE_BAND={TRUE_BAND} MeV/cm from MPV")
for _b in range(len(_per_batch)):
    _nb_out = int((_outlier_batch == _b).sum())
    _nb_tot = len(_per_batch[_b][0])
    print(f"  Batch {_b}: {_nb_out} / {_nb_tot} ({100*_nb_out/_nb_tot:.1f}%)")

# ── Figure 1: fitted vs true scatter coloured by batch ────────────────────────
_cmap_bt = plt.cm.get_cmap('tab10', len(_per_batch))
fig_out1, ax_out1 = plt.subplots(figsize=(6, 5))

# Background — all segments (grey)
ax_out1.scatter(_true_all, _fit_all, s=4, color='lightgrey', alpha=0.3, label='All segments', zorder=1)

# Outliers per batch
for _b in range(len(_per_batch)):
    _sel = _outlier_batch == _b
    if _sel.sum() == 0:
        continue
    ax_out1.scatter(_outlier_true[_sel], _outlier_reco[_sel], s=18,
                    color=_cmap_bt(_b), alpha=0.8, label=f'Batch {_b} outliers', zorder=3)

# Reference lines
_xlim = [0.3, 4.5]
ax_out1.plot(_xlim, [_mpv]*2, 'k--', lw=0.9, label=f'MPV={_mpv:.2f}')
ax_out1.axhspan(_mpv - RECO_BAND, _mpv + RECO_BAND, alpha=0.08, color='orange', label='Reco band')
ax_out1.plot(_xlim, _xlim, 'r--', lw=0.8, label='y=x')
ax_out1.set_xlim(_xlim); ax_out1.set_ylim(_xlim)
ax_out1.set_xlabel('True dEdx (MeV/cm)'); ax_out1.set_ylabel('Fitted dEdx (MeV/cm)')
ax_out1.set_title(f'Stuck-at-MPV outliers  (n={_n_outliers})')
ax_out1.legend(fontsize=7, loc='upper right')
fig_out1.tight_layout(); plt.show()

# ── Figure 2: distribution of true dEdx for outliers vs rest ──────────────────
_non_out_true = np.concatenate([
    _t[~((np.abs(_f - _mpv) < RECO_BAND) & (np.abs(_t - _mpv) > TRUE_BAND))]
    for _f, _t in _per_batch
])

fig_out2, axes_out2 = plt.subplots(1, 2, figsize=(11, 4))
fig_out2.suptitle('True dEdx distribution: outliers vs rest', fontsize=12)

_BINS2 = np.linspace(0.2, 5.0, 50)

axes_out2[0].hist(_non_out_true, bins=_BINS2, density=True, alpha=0.55,
                  color='steelblue', label='Non-outliers')
axes_out2[0].hist(_outlier_true, bins=_BINS2, density=True, alpha=0.65,
                  color='tomato',    label='Outliers (true)')
axes_out2[0].axvspan(_mpv - TRUE_BAND, _mpv + TRUE_BAND, alpha=0.1,
                     color='grey', label='True band exclusion')
axes_out2[0].axvline(_mpv, color='k', lw=0.8, ls='--', label=f'MPV={_mpv:.2f}')
axes_out2[0].set_xlabel('True dEdx (MeV/cm)'); axes_out2[0].set_ylabel('Density')
axes_out2[0].set_title('True dEdx — outliers vs rest')
axes_out2[0].legend(fontsize=7)

# Cumulative fraction of outliers vs true dEdx
_sorted_t = np.sort(_outlier_true)
axes_out2[1].plot(_sorted_t, np.arange(1, len(_sorted_t)+1) / len(_sorted_t),
                  color='tomato', lw=1.5)
axes_out2[1].axvline(_mpv - TRUE_BAND, color='grey', ls=':', lw=0.9)
axes_out2[1].axvline(_mpv + TRUE_BAND, color='grey', ls=':', lw=0.9)
axes_out2[1].set_xlabel('True dEdx (MeV/cm)'); axes_out2[1].set_ylabel('Cumulative fraction')
axes_out2[1].set_title('CDF of outlier true dEdx')
axes_out2[1].grid(True, alpha=0.3)

fig_out2.tight_layout(); plt.show()

# ── Figure 3: per-batch fraction of stuck-at-MPV outliers ─────────────────────
_bt_fracs   = []
_bt_labels  = []
_bt_true_hi = []   # fraction of outliers with true > MPV + TRUE_BAND
_bt_true_lo = []   # fraction with true < MPV - TRUE_BAND

for _b, (_f, _t) in enumerate(_per_batch):
    _mask = (np.abs(_f - _mpv) < RECO_BAND) & (np.abs(_t - _mpv) > TRUE_BAND)
    _frac = _mask.sum() / max(len(_f), 1)
    _bt_fracs.append(100 * _frac)
    _bt_labels.append(f'Batch {_b}')
    _hi = ((_t[_mask] > _mpv + TRUE_BAND).sum() / max(_mask.sum(), 1)) * 100
    _lo = ((_t[_mask] < _mpv - TRUE_BAND).sum() / max(_mask.sum(), 1)) * 100
    _bt_true_hi.append(_hi)
    _bt_true_lo.append(_lo)

_xs = np.arange(len(_per_batch))
fig_out3, axes_out3 = plt.subplots(1, 2, figsize=(10, 4))
fig_out3.suptitle('Per-batch outlier breakdown', fontsize=12)

axes_out3[0].bar(_xs, _bt_fracs, color=[_cmap_bt(_b) for _b in _xs])
axes_out3[0].set_xticks(_xs); axes_out3[0].set_xticklabels(_bt_labels)
axes_out3[0].set_ylabel('Outlier fraction (%)'); axes_out3[0].set_title('% stuck-at-MPV per batch')
axes_out3[0].grid(axis='y', alpha=0.3)

_w = 0.35
axes_out3[1].bar(_xs - _w/2, _bt_true_lo, width=_w, label=f'True < MPV-{TRUE_BAND}', color='steelblue')
axes_out3[1].bar(_xs + _w/2, _bt_true_hi, width=_w, label=f'True > MPV+{TRUE_BAND}', color='tomato')
axes_out3[1].set_xticks(_xs); axes_out3[1].set_xticklabels(_bt_labels)
axes_out3[1].set_ylabel('% of outliers'); axes_out3[1].set_title('Outlier direction (true side)')
axes_out3[1].legend(fontsize=7); axes_out3[1].grid(axis='y', alpha=0.3)

fig_out3.tight_layout(); plt.show()

# ── Segment-level info table for outliers ─────────────────────────────────────
# For each outlier:
#   - recover the first-occurrence row in the tracks array via pids
#   - print all available track fields alongside fitted/true dEdx
#
# Fields to always highlight if present (subset of track_fields)
_INTEREST_FIELDS = ['x_start', 'y_start', 'z_start', 'x_end', 'y_end', 'z_end',
                    'dx', 'dE', 'dEdx', 'n_electrons', 'tran_diff', 'long_diff']
_tf = list(track_fields)  # local copy as plain list

# Build per-batch mapping: seg_idx  -> row in tracks array
_seg_rows_per_batch = {}
for _b, _bd in enumerate(_sj_batches):
    _pids_np = np.asarray(_bd['pids'])
    _valid   = _pids_np >= 0
    _pv      = _pids_np[_valid]
    _vrows   = np.where(_valid)[0]
    _, _focc = np.unique(_pv, return_index=True)
    _seg_rows_per_batch[_b] = _vrows[_focc]   # shape (N_orig,)

# Determine which fields to print
_print_fields = [f for f in _INTEREST_FIELDS if f in _tf]
_extra_fields = [f for f in _tf if f not in _print_fields and f != 'dEdx'][:4]  # up to 4 extras
_all_print    = _print_fields + _extra_fields

# Header
_col_w = 10
_hdr_fixed = f"{'Batch':>5}  {'SegIdx':>6}  {'TruedEdx':>9}  {'FitdEdx':>9}  {'Delta':>8}  {'Direction':>10}"
_hdr_extra = ''.join(f"  {f[:_col_w]:>{_col_w}}" for f in _all_print if f != 'dEdx')
print("=" * (len(_hdr_fixed) + len(_hdr_extra)))
print("Stuck-at-MPV outlier segment details")
print("=" * (len(_hdr_fixed) + len(_hdr_extra)))
print(_hdr_fixed + _hdr_extra)
print("-" * (len(_hdr_fixed) + len(_hdr_extra)))

_MAX_PRINT = 200  # guard against huge tables
_printed   = 0

# rebuild per-batch outlier index list (parallels _outlier_batch / _outlier_idx)
_out_ptr = 0
for _b, (_f, _t) in enumerate(_per_batch):
    _mask    = (np.abs(_f - _mpv) < RECO_BAND) & (np.abs(_t - _mpv) > TRUE_BAND)
    _seg_idxs = np.where(_mask)[0]
    _seg_rows = _seg_rows_per_batch[_b]
    _tracks_np = np.asarray(_sj_batches[_b]['tracks'])

    for _si in _seg_idxs:
        if _printed >= _MAX_PRINT:
            print(f"  ... (truncated at {_MAX_PRINT} rows)")
            break
        _row  = int(_seg_rows[_si])
        _fval = float(_f[_si])
        _tval = float(_t[_si])
        _delta = _fval - _tval
        _direction = 'high-true' if _tval > _mpv else 'low-true'

        _row_fixed = (f"{_b:>5}  {_si:>6}  {_tval:>9.4f}  {_fval:>9.4f}"
                      f"  {_delta:>+8.4f}  {_direction:>10}")
        _row_extra = ''
        for _fn in _all_print:
            if _fn == 'dEdx':
                continue
            try:
                _fidx = _tf.index(_fn)
                _val  = float(_tracks_np[_row, _fidx])
                _row_extra += f"  {_val:>{_col_w}.4g}"
            except (ValueError, IndexError):
                _row_extra += f"  {'N/A':>{_col_w}}"
        print(_row_fixed + _row_extra)
        _printed += 1

print("-" * (len(_hdr_fixed) + len(_hdr_extra)))
print(f"Total outliers printed: {_printed}")


In [36]:

# ═══════════════════════════════════════════════════════════
# dEdx diagnostic plots after multi-batch joint fit
#   1. Distribution evolution: overlaid histograms coloured by epoch
#      + Student-t prior curve used during optimisation
#   2. 2-D comparison: fitted dEdx vs true dEdx (final snapshot)
# ═══════════════════════════════════════════════════════════
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np
from scipy.stats import t as scipy_t

# ── 1. dEdx distribution evolution across epochs ─────────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))

_n_snaps   = len(_sj_dedx_snapshots)
_cmap      = cm.plasma
_snap_epochs, _, _snap_true = _sj_dedx_snapshots[-1]   # true is the same for all
_dedx_max  = max(np.percentile(_snap_true, 99) * 1.5, 4)
_bins      = np.linspace(0, _dedx_max, 60)

# True dEdx reference (constant, shown once)
ax.hist(_snap_true, bins=_bins, histtype='step', color='black',
        linewidth=1.8, linestyle='--', density=True, label='True dEdx')

# Student-t prior curve
_nu    = float(STUDENT_NU)
_loc   = float(STUDENT_LOC)
_scale = float(STUDENT_SCALE)
_x_st  = np.linspace(0, _dedx_max, 400)
_y_st  = scipy_t.pdf(_x_st, df=_nu, loc=_loc, scale=_scale)
ax.plot(_x_st, _y_st, color='limegreen', linewidth=2.0, linestyle='-.',
        label=f'Student-t prior\n(ν={_nu:.1f}, loc={_loc:.2f}, σ={_scale:.2f})')

for _i, (_ep, _fitted, _true) in enumerate(_sj_dedx_snapshots[-1:]):
    _color = _cmap(_i / max(_n_snaps - 1, 1))
    ax.hist(_fitted, bins=_bins, histtype='step', color=_color,
            density=True, linewidth=1.0, label=f'Ep {_ep}')

ax.set_xlabel('dEdx  (MeV/cm)')
ax.set_ylabel('Density')
ax.set_title('Fitted dEdx distribution — evolution across epochs\n(all batches aggregated)')

# Colourbar to show epoch progression
_sm = cm.ScalarMappable(cmap=_cmap,
      norm=plt.Normalize(vmin=_sj_dedx_snapshots[0][0],
                         vmax=_sj_dedx_snapshots[-1][0]))
_sm.set_array([])
fig.colorbar(_sm, ax=ax, label='Epoch')
ax.legend(loc='upper right', fontsize=7, ncol=2)
fig.tight_layout(); plt.show()

# ── 2. 2-D: fitted vs true dEdx (final snapshot) ─────────────────────────────
_ep_final, _fitted_final, _true_final = _sj_dedx_snapshots[-1]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Left: hexbin density scatter
_lim = (0.5, 4)
hb = axes[0].hexbin(_true_final, _fitted_final,
                     gridsize=40, cmap='viridis', mincnt=1,
                     extent=[_lim[0], _lim[1], _lim[0], _lim[1]])
axes[0].plot(_lim, _lim, 'r--', lw=1.2, label='y = x')
fig.colorbar(hb, ax=axes[0], label='Count')
axes[0].set_xlabel('True dEdx  (MeV/cm)')
axes[0].set_ylabel('Fitted dEdx  (MeV/cm)')
axes[0].set_title(f'Fitted vs True dEdx  (epoch {_ep_final})')
axes[0].legend()

# Right: residual (fitted - true) histogram
_resid = _fitted_final - _true_final
axes[1].hist(_resid, bins=50, color='steelblue', edgecolor='white', linewidth=0.3)
axes[1].axvline(0,    color='red',  linestyle='--', lw=1.2, label='Zero bias')
axes[1].axvline(np.mean(_resid), color='orange', linestyle='-',  lw=1.2,
                label=f'Mean = {np.mean(_resid):.3f}')
axes[1].set_xlabel('Fitted − True  (MeV/cm)')
axes[1].set_ylabel('Count')
axes[1].set_title('dEdx residual distribution')
axes[1].legend()

fig.suptitle(f'Joint fit dEdx diagnostics — final epoch {_ep_final}  '
             f'({_sj_num_batches} batches, N={len(_true_final)} segments)')
fig.tight_layout(); plt.show()

print(f"Residual stats:  mean={np.mean(_resid):.4f}  "
      f"std={np.std(_resid):.4f}  "
      f"MAE={np.mean(np.abs(_resid)):.4f}  (MeV/cm)")


In [20]:
# ═══════════════════════════════════════════════════════════════════════════════
# Outlier analysis of the fitted dEdx distribution
#
# We define outliers as segments where |fitted − true| > OUTLIER_THRESH
# (or fitted > OUTLIER_ABS_MAX) and probe:
#   A. What are their physical properties (dx, n_substeps, position in track)?
#   B. Sub-step dx distribution: do outlier parents have anomalous discretisation?
#   C. Marginal loss contribution (prior NLL) of outliers
# ═══════════════════════════════════════════════════════════════════════════════
import numpy as np
import matplotlib.pyplot as plt

# ── Configuration ─────────────────────────────────────────────────────────────
OUTLIER_THRESH  = 0.5   # |fitted − true| threshold in MeV/cm
OUTLIER_ABS_MAX = 4.0   # also flag anything above this (runaway)
BIDX            = 1  # which batch to inspect (re-uses the kernel variable)

# ── Recover per-segment arrays from the multi-batch fit ───────────────────────
_bd         = _sj_batches[BIDX]
_fitted_np  = np.exp(np.asarray(_bd['log_dedx']))   # [N_orig]
_true_np    = np.asarray(_bd['true_dedx'])           # [N_orig]
_pids_np    = np.asarray(_bd['pids'])                # [N_chopped]  (int32, -1 for padding)
_tracks_b_np = np.asarray(_bd['tracks'])             # [N_chopped, n_fields]

_dedx_idx = list(track_fields).index('dEdx')
_dE_idx   = list(track_fields).index('dE')
_dx_idx   = list(track_fields).index('dx')
_N_orig   = _fitted_np.shape[0]

# Per-original-segment aggregates
_dx_per_seg     = np.zeros(_N_orig, dtype=np.float32)
_nsteps_per_seg = np.zeros(_N_orig, dtype=np.int32)
_first_row      = np.zeros(_N_orig, dtype=np.int32)

for _seg in range(_N_orig):
    _rows = np.where(_pids_np == _seg)[0]
    if len(_rows) == 0:
        continue
    _dx_per_seg[_seg]     = _tracks_b_np[_rows, _dx_idx].sum()
    _nsteps_per_seg[_seg] = len(_rows)
    _first_row[_seg]      = _rows[0]

_residual = _fitted_np - _true_np
_outlier  = (np.abs(_residual) > OUTLIER_THRESH) | (_fitted_np > OUTLIER_ABS_MAX)
_n_out    = int(_outlier.sum())
_n_norm   = _N_orig - _n_out
print(f"Batch {BIDX}: {_N_orig} original segments")
print(f"Outliers  : {_n_out}  ({100*_n_out/_N_orig:.1f} %)"
      f"  (|resid| > {OUTLIER_THRESH} MeV/cm  OR  fitted > {OUTLIER_ABS_MAX} MeV/cm)")

# ── A. Physical properties of outliers vs normal segments ─────────────────────
def _chist(ax, values, mask, xlabel, title, bins=30):
    ax.hist(values[~mask], bins=bins, density=True, alpha=0.6, color='steelblue',
            label=f'Normal (n={_n_norm})')
    if mask.any():
        ax.hist(values[mask], bins=bins, density=True, alpha=0.6, color='tomato',
                label=f'Outlier (n={_n_out})')
    ax.set_xlabel(xlabel); ax.set_title(title); ax.legend(fontsize=7)

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle(f'Outlier analysis — batch {BIDX},  {_n_out}/{_N_orig} outliers', fontsize=12)

_chist(axes[0,0], _true_np,          _outlier, 'True dEdx (MeV/cm)',      'True dEdx')
_chist(axes[0,1], _dx_per_seg,       _outlier, 'Total dx per seg (cm)',   'Segment dx')
_chist(axes[0,2], _nsteps_per_seg,   _outlier, 'N sub-steps',             'Sub-step count', bins=20)
_chist(axes[1,0], _fitted_np,        _outlier, 'Fitted dEdx (MeV/cm)',    'Fitted dEdx')
_chist(axes[1,1], np.abs(_residual), _outlier, '|Fitted − True| (MeV/cm)', 'Absolute residual')

axes[1,2].scatter(_first_row[~_outlier], _fitted_np[~_outlier], s=4,  alpha=0.3,
                  color='steelblue', label='Normal')
axes[1,2].scatter(_first_row[_outlier],  _fitted_np[_outlier],  s=20, alpha=0.8,
                  color='tomato', label='Outlier', zorder=5)
axes[1,2].axhline(OUTLIER_ABS_MAX, color='k', lw=0.8, ls='--')
axes[1,2].set_xlabel('First sub-step index in batch')
axes[1,2].set_ylabel('Fitted dEdx (MeV/cm)')
axes[1,2].set_title('Outliers vs batch position')
axes[1,2].legend(fontsize=7)
fig.tight_layout(); plt.show()

# ── B. Sub-step dx: do outlier parents have anomalous sub-step sizes? ──────────
_out_rows  = np.isin(_pids_np, np.where(_outlier)[0])
_norm_rows = np.isin(_pids_np, np.where(~_outlier)[0])
_dx_out    = _tracks_b_np[_out_rows,  _dx_idx]
_dx_norm   = _tracks_b_np[_norm_rows, _dx_idx]

fig, ax = plt.subplots(figsize=(7, 4))
_all_dx  = np.concatenate([_dx_out, _dx_norm]) if len(_dx_out) else _dx_norm
_dbins   = np.linspace(0, np.percentile(_all_dx, 99) * 1.5, 50)
ax.hist(_dx_norm, bins=_dbins, density=True, alpha=0.6, color='steelblue', label='Normal sub-steps')
if len(_dx_out):
    ax.hist(_dx_out, bins=_dbins, density=True, alpha=0.6, color='tomato', label='Outlier sub-steps')
ax.set_xlabel('Sub-step dx (cm)'); ax.set_ylabel('Density')
ax.set_title(f'Sub-step dx: outlier parents vs normal  (batch {BIDX})')
ax.legend(); fig.tight_layout(); plt.show()

# ── C. Prior NLL contribution of outliers ─────────────────────────────────────
print("\n── C. Prior NLL contribution of outliers ──")
from optimize.dedx_utils import student_t_nll, DEDX_STUDENT_NU, DEDX_STUDENT_LOC, DEDX_STUDENT_SCALE
import jax.numpy as jnp

_nll_all     = float(student_t_nll(jnp.array(_fitted_np),
                                   DEDX_STUDENT_NU, DEDX_STUDENT_LOC, DEDX_STUDENT_SCALE).mean())
_patched     = np.where(_outlier, _true_np, _fitted_np)
_nll_patched = float(student_t_nll(jnp.array(_patched),
                                   DEDX_STUDENT_NU, DEDX_STUDENT_LOC, DEDX_STUDENT_SCALE).mean())
print(f"Mean prior NLL — all fitted              : {_nll_all:.4f}")
print(f"Mean prior NLL — outliers patched to true: {_nll_patched:.4f}")
print(f"Δ (attributed to outliers, as fraction)  : {(_nll_all - _nll_patched)/abs(_nll_all)*100:.1f} %")

# ── Summary table ─────────────────────────────────────────────────────────────
_show = np.where(_outlier)[0]
print(f"\n── Outlier summary (showing up to 20 of {_n_out}) ──")
print(f"{'Seg':>5}  {'True':>6}  {'Fitted':>7}  {'Resid':>7}  {'dx':>7}  {'Nstep':>5}")
for _s in _show[:20]:
    print(f"{_s:5d}  {_true_np[_s]:6.3f}  {_fitted_np[_s]:7.3f}  "
          f"{_residual[_s]:+7.3f}  {_dx_per_seg[_s]:7.4f}  {_nsteps_per_seg[_s]:5d}")
if _n_out > 20:
    print(f"  ... ({_n_out - 20} more not shown)")


In [18]:
# All N_orig_segs entries are valid (no padding in the parameter arrays)
plt.plot(np.minimum(10, jnp.exp(_dedxonly_cache[_bidx]['log_dedx_mean'])))
# plt.plot(_tracks_b[_parent_ids_np >= 0, dedx_idx])  # true dEdx for valid chopped rows
# plt.plot(jnp.exp(_dedxonly_cache[_bidx]['log_dedx_sigma']))

# plt.xlim(0, 200)
# plt.ylim(0, 10)

In [24]:
np.where(_diff > 100)

In [31]:
_parent_ids_np  = dataset.get_parent_segment_ids(_bidx)
np.argwhere(_parent_ids_np ==201)

In [32]:
_tracks_b[7071, track_fields.index('dx')]

In [11]:
# Parameters are [N_orig_segs] — no mask needed
plt.hist2d(
    np.array(jnp.exp(_dedxonly_cache[_bidx]['log_dedx_mean'])),
    np.array(jnp.exp(_dedxonly_cache[_bidx]['log_dedx_sigma'])),
    bins=50, cmap='viridis', range=((0, 5), (0, 5))
)


In [ ]:
# _parent_ids_np >= 0 selects real (non-padded) chopped rows
_row_valid_mask = _parent_ids_np >= 0
plt.plot(_tracks_b[_row_valid_mask, dedx_idx])


In [34]:
# Distribution of (fitted dEdx - true dEdx) per original segment
_fitted = np.array(jnp.exp(_dedxonly_cache[_bidx]['log_dedx_mean']))  # [N_orig_segs]

# True dEdx for each original segment: take first chopped row of each parent
_parent_ids_valid = _parent_ids_np[_parent_ids_np >= 0]
_valid_rows       = np.where(_parent_ids_np >= 0)[0]
_, _first_occ     = np.unique(_parent_ids_valid, return_index=True)
_first_rows       = _valid_rows[_first_occ]
_true             = np.array(_tracks_b[_first_rows, track_fields.index('dEdx')])  # [N_orig_segs]

_diff = _fitted - _true

plt.figure(figsize=(7, 4))
plt.hist(_diff, bins=np.linspace(-3, 3, 60), histtype='step', color='steelblue', lw=1.5)
plt.axvline(0, color='grey', linestyle=':', lw=1)
plt.axvline(float(np.mean(_diff)), color='tab:red', linestyle='--', lw=1.5,
            label=f'mean = {float(np.mean(_diff)):.3f}')
plt.xlabel('Fitted dEdx − True dEdx  (MeV/cm)')
plt.ylabel('Segments')
plt.title('Residual distribution: fitted vs true dEdx (per original segment)')
plt.legend()
plt.tight_layout()
plt.show()

print(f"Mean residual : {float(np.mean(_diff)):.4f} MeV/cm")
print(f"Median residual : {float(np.median(_diff)):.4f} MeV/cm")
print(f"Std  residual : {float(np.std(_diff)):.4f} MeV/cm")


In [12]:
#Plot.a 2d hist of _dedxonly_differences with iteration steps on x-axis and difference on y-axis
plt.figure(figsize=(8,6))

iter_nb = np.concatenate([np.full_like(diff, step) for step, diff in enumerate(_dedxonly_differences)])
differences = np.concatenate(_dedxonly_differences)
plt.hist2d(iter_nb, differences, bins=[len(_dedxonly_differences), 50], cmap='jet', range=[[0, len(_dedxonly_differences)], [-2, 2]])
plt.colorbar(label='Count')
plt.xlabel('Iteration step')
plt.ylabel('Fitted dEdx - True dEdx (MeV/cm)')
plt.title('dEdx differences over iterations (dEdx-only fit)')
plt.show()

In [99]:
#Plot.a 2d hist of _dedxonly_sigmas with iteration steps on x-axis and sigma on y-axis
plt.figure(figsize=(8,6))

iter_nb = np.concatenate([np.full_like(diff, step) for step, diff in enumerate(_dedxonly_sigmas)])
differences = np.concatenate(_dedxonly_sigmas)
plt.hist2d(iter_nb, differences, bins=[len(_dedxonly_sigmas), 50], cmap='jet', range=[[0, len(_dedxonly_sigmas)], [0, 2]])
plt.colorbar(label='Count')
plt.xlabel('Iteration step')
plt.ylabel('Fitted dEdx sigma (MeV/cm)')
plt.title('dEdx sigma over iterations (dEdx-only fit)')
plt.show()



In [101]:
for i in [0, 50, 100, 299, 500, 999]:
    plt.hist(_dedxonly_differences[i], label=f'Iter {i}', histtype='step', bins=np.linspace(-2, 2, 50))
plt.xlabel('Fitted dEdx - True dEdx (MeV/cm)')
plt.ylabel('Count')
plt.title('Distribution of dEdx differences at selected iterations')
plt.legend()

In [102]:
for i in [0, 50, 100, 299]:
    plt.hist(_dedxonly_sigmas[i], label=f'Iter {i}', histtype='step', bins=np.linspace(0, 2, 50))
plt.xlabel('Fitted dEdx sigma (MeV/cm)')
plt.ylabel('Count')
plt.title('Distribution of dEdx sigma at selected iterations')
plt.legend()

In [103]:
#Plot the distribution of pulls as a function of iteration step, where pull = (fitted dEdx - true dEdx) / fitted sigma
plt.figure(figsize=(8,6))
iter_nb = np.concatenate([np.full_like(diff, step) for step, diff in enumerate(_dedxonly_differences)])
sigmas = np.concatenate(_dedxonly_sigmas)
differences = np.concatenate(_dedxonly_differences)
pulls = differences / sigmas
plt.hist2d(iter_nb, pulls, bins=[len(_dedxonly_differences), 50], cmap='jet', range=[[0, len(_dedxonly_differences)], [-5, 5]])
plt.colorbar(label='Count')
plt.xlabel('Iteration step')
plt.ylabel('Pull = (fitted dEdx - true dEdx) / fitted sigma')
plt.title('Pull distribution over iterations (dEdx-only fit)')

In [96]:
#Plot slices of the pull distribution at iterations 0, 50, 100, 299
plt.figure(figsize=(8,6))
for i in [0, 50, 100, 299]:
    iter_mask = np.full_like(differences, False, dtype=bool)
    iter_mask[iter_nb == i] = True
    plt.hist(pulls[iter_mask], label=f'Iter {i}', histtype='step', bins=np.linspace(-5, 5, 50))
plt.xlabel('Pull = (fitted dEdx - true dEdx) / fitted sigma')
plt.ylabel('Count')
plt.title('Pull distribution at selected iterations (dEdx-only fit)')

In [17]:
import jax.numpy as jnp
from optimize.strategies import GenericLossStrategy

learnable_dedx = tracks[:, track_fields.index('dEdx')]

# Target at nominal params (used for standalone cells; training loop creates its own)
target_output = format_target_output(
    target_strategy.predict(current_params, tracks, track_fields, 42)
)

loss_strategy_stochastic = GenericLossStrategy(mse_adc)

ref_mean  = ref_quantiles[50]
ref_sigma = jnp.std(selected)

# ── Regularisation weights ─────────────────────────────────────────────────────
LAMBDA_MOYAL        = 1.0    # per-segment Moyal log-prior
LAMBDA_WASSERSTEIN  = 10.0   # global log-space Wasserstein
LAMBDA_TV           = 5.0    # total variation on log(dEdx)
LAMBDA_ENTROPY      = 1.0    # entropy penalty on sigma

# ── Total-variation regulariser ───────────────────────────────────────────────
def compute_tv_penalty(log_dedx, track_ids, event_ids):
    """Penalises large adjacent log-dEdx jumps within the same track."""
    diffs           = jnp.diff(log_dedx)
    same_track_mask = ((track_ids[1:] == track_ids[:-1]) &
                       (event_ids[1:] == event_ids[:-1]) &
                       (event_ids[1:] >= 0))
    valid_diffs     = jnp.abs(diffs) * same_track_mask
    num_valid_pairs = jnp.maximum(1.0, jnp.sum(same_track_mask))
    return jnp.sum(valid_diffs) / num_valid_pairs

# ── Entropy regulariser (prevents sigma collapse) ─────────────────────────────
def compute_entropy_penalty(log_dedx_sigma, event_ids):
    """Minimise -mean[log(sigma)] so sigma stays non-negligible."""
    valid_log_sigmas = jnp.where(event_ids >= 0, log_dedx_sigma, 0.0)
    num_valid        = jnp.maximum(1.0, jnp.sum(event_ids >= 0))
    return -jnp.sum(valid_log_sigmas) / num_valid

# ── Per-segment Gaussian prior on log_dedx (DEPRECATED — biases Landau tail) ─
def compute_dedx_gaussian_prior(log_dedx_mean, event_ids, log_dedx0, sigma_log=0.3):
    """Anchors EVERY segment toward log_dedx0.
    DEPRECATED: biases high-dEdx Landau-tail segments back toward MPV,
    causing beta to compensate by moving in the wrong direction.
    """
    valid     = event_ids >= 0
    residuals = (log_dedx_mean - log_dedx0) / sigma_log
    penalty   = 0.5 * jnp.where(valid, residuals ** 2, 0.0)
    num_valid = jnp.maximum(1.0, jnp.sum(valid))
    return jnp.sum(penalty) / num_valid

# ── Batch-mean constraint prior on log_dedx (CORRECT) ─────────────────────────
def compute_dedx_mean_prior(log_dedx_mean, event_ids, log_dedx_ref_mean, sigma_mean=0.01):
    """Penalise the BATCH MEAN of log_dedx for drifting from the reference mean.

    Breaks the beta-dEdx degeneracy: a uniform shift of all segments costs
        0.5 * ((shift / sigma_mean)^2)
    while per-segment Landau fluctuations are free.

    Args:
        log_dedx_mean     : per-segment log(dEdx) values, shape [N]
        event_ids         : event ID per segment  (padding = -1)
        log_dedx_ref_mean : reference value (= mean[log(dEdx_MC)])
        sigma_mean        : allowed batch-mean drift in log-space (default 0.01 = 1%)
    """
    valid          = event_ids >= 0
    log_dedx_valid = jnp.where(valid, log_dedx_mean, 0.0)
    num_valid      = jnp.maximum(1.0, jnp.sum(valid))
    batch_mean     = jnp.sum(log_dedx_valid) / num_valid
    return 0.5 * ((batch_mean - log_dedx_ref_mean) / sigma_mean) ** 2

# ── Reference log-mean from the fitted MC distribution ────────────────────────
_LOG_DEDX_REF_MEAN = float(jnp.mean(jnp.log(jnp.clip(jnp.array(selected), 1e-3, None))))
print(f'_LOG_DEDX_REF_MEAN = {_LOG_DEDX_REF_MEAN:.4f}  '
      f'(dEdx ref mean = {float(jnp.exp(jnp.array(_LOG_DEDX_REF_MEAN))):.4f} MeV/cm)')

# ── Master loss function ───────────────────────────────────────────────────────
def compute_loss(theta, target_out, tracks, params, fields, param_name, rng_key,
                 lambda_moyal=LAMBDA_MOYAL,
                 lambda_wass=LAMBDA_WASSERSTEIN,
                 lambda_tv=LAMBDA_TV,
                 lambda_ent=LAMBDA_ENTROPY,
                 lambda_dedx_prior=0.0,          # DEPRECATED per-segment prior
                 dedx_prior_sigma=0.3,
                 lambda_dedx_mean_prior=0.0,      # NEW batch-mean constraint
                 dedx_mean_prior_sigma=0.01):
    """
    Total loss = NLL(hits)
               + lambda_moyal      * Moyal log-prior
               + lambda_wass       * Wasserstein prior
               + lambda_tv         * TV regulariser
               + lambda_ent        * entropy penalty
               + lambda_dedx_prior * per-segment Gaussian prior  (DEPRECATED)
               + lambda_dedx_mean_prior * batch-mean constraint  (USE THIS)

    theta keys:
        log_param        : log of the single physics parameter being fitted
        log_dedx_mean    : log of per-segment dEdx mean  (shape: [N_segs])
        log_dedx_sigma   : log of per-segment dEdx sigma (shape: [N_segs])
    """
    param_value = jnp.exp(theta['log_param'])
    dedx_mean   = jnp.exp(theta['log_dedx_mean'])
    dedx_sigma  = jnp.exp(theta['log_dedx_sigma'])
    event_ids   = tracks[:, fields.index('eventID')]

    # Reparameterisation trick: sample dEdx ~ N(mean, sigma)
    eps         = jax.random.normal(rng_key, shape=dedx_mean.shape)
    dedx_sample = jnp.maximum(dedx_mean + dedx_sigma * eps, 1e-3)

    # Build updated tracks with sampled dEdx values
    dedx_idx       = fields.index('dEdx')
    dE_idx         = fields.index('dE')
    dx_idx         = fields.index('dx')
    updated_tracks = tracks.at[:, dedx_idx].set(dedx_sample)
    updated_tracks = updated_tracks.at[:, dE_idx].set(
        dedx_sample * updated_tracks[:, dx_idx]
    )

    # Forward model
    updated_params = params.replace(**{param_name: param_value})
    pred_dist      = pred_strategy.predict(updated_params, updated_tracks, fields, 42)

    # Hit-level NLL
    hit_loss, aux = loss_strategy.compute(updated_params, pred_dist, target_out)

    # Physics priors on dEdx
    track_ids    = tracks[:, fields.index('trackID')]
    moyal_pen    = moyal_log_prior(dedx_mean, MOYAL_MPV, MOYAL_XI, event_ids)
    wass_pen     = wasserstein_log_prior(dedx_mean, ref_quantiles, event_ids)
    tv_pen       = compute_tv_penalty(theta['log_dedx_mean'], track_ids, event_ids)
    ent_pen      = compute_entropy_penalty(theta['log_dedx_sigma'], event_ids)
    dedx_gau_pen = compute_dedx_gaussian_prior(
        theta['log_dedx_mean'], event_ids,
        log_dedx0=jnp.log(MOYAL_MPV), sigma_log=dedx_prior_sigma
    )
    dedx_mean_pen = compute_dedx_mean_prior(
        theta['log_dedx_mean'], event_ids,
        log_dedx_ref_mean=_LOG_DEDX_REF_MEAN,
        sigma_mean=dedx_mean_prior_sigma
    )

    loss_tot = (hit_loss
                + lambda_moyal           * moyal_pen
                + lambda_wass            * wass_pen
                + lambda_tv              * tv_pen
                + lambda_ent             * ent_pen
                + lambda_dedx_prior      * dedx_gau_pen
                + lambda_dedx_mean_prior * dedx_mean_pen)

    aux[param_name]                    = param_value
    aux['loss']                        = hit_loss
    aux['moyal_penalty']               = moyal_pen
    aux['wass_penalty']                = wass_pen
    aux['tv_penalty']                  = tv_pen
    aux['entropy_penalty']             = ent_pen
    aux['dedx_prior_penalty']          = dedx_gau_pen
    aux['dedx_mean_prior_penalty']     = dedx_mean_pen

    return loss_tot, aux


In [18]:

import jax
import jax.numpy as jnp
from jax import value_and_grad
import optax

# ── Hyper-parameters ──────────────────────────────────────────────────────────
param_name   = 'beta'
num_batches  = 5
local_lr     = 5e-3
global_lr    = 5e-3
num_epochs   = 100
inner_steps  = 3   # inner dEdx steps per outer physics-param step

# ── Global physics parameter (start 20 % above nominal for closure test) ──────
initial_guess_param = getattr(current_params, param_name) * 1.2
global_theta        = jnp.log(initial_guess_param)

global_optimizer = optax.adam(global_lr)
global_opt_state = global_optimizer.init(global_theta)

# ── Local optimizer + cache ───────────────────────────────────────────────────
local_optimizer = optax.adam(local_lr)
local_cache     = {}   # keys: 'dedx_mean', 'dedx_sigma', 'opt_state'

# ── RNG ───────────────────────────────────────────────────────────────────────
rng_key = jax.random.PRNGKey(0)

# ── Metric history ────────────────────────────────────────────────────────────
losses                   = []
param_history            = []
dedx_scale_history       = []
loss_history             = []
moyal_penalty_history    = []   # per-segment Moyal NLL
wass_penalty_history     = []   # global Wasserstein prior
tv_penalty_history       = []
dedx_dist_to_truth_history = []
entropy_penalty_history    = []

# ── Loss wrapper (argnums=(0,1) to differentiate global and local) ───────────
def batch_loss(global_log_param, local_params, tracks_batch, target_output_batch, key):
    theta = {
        'log_param':      global_log_param,
        'log_dedx_mean':  local_params['log_dedx_mean'],
        'log_dedx_sigma': local_params['log_dedx_sigma'],
    }
    return compute_loss(theta, target_output_batch, tracks_batch,
                        current_params, track_fields, param_name, key)

vg_loss = value_and_grad(batch_loss, argnums=(0, 1), has_aux=True)

# ── Training loop ─────────────────────────────────────────────────────────────
for epoch in range(num_epochs):
    for batch_idx in range(min(num_batches, len(dataloader))):

        # 1. Prepare batch
        tracks_batch = jax.device_put(
            dataloader[batch_idx].reshape(-1, len(track_fields))
        )
        target_output_batch = format_target_output(
            target_strategy.predict(current_params, tracks_batch, track_fields, batch_idx)
        )

        # 2. Initialise local state (first time this batch is seen)
        if batch_idx not in local_cache:
            # Initialise mean at Moyal MPV, sigma at Moyal xi (from prior cell)
            init_log_dedx = jnp.full(
                (tracks_batch.shape[0],),
                jnp.log(MOYAL_MPV),
                dtype=jnp.float32,
            )
            init_log_sigma = jnp.full_like(init_log_dedx, jnp.log(MOYAL_XI))
            init_local = {'log_dedx_mean': init_log_dedx,
                          'log_dedx_sigma': init_log_sigma}
            local_cache[batch_idx] = {
                'dedx_mean':  init_log_dedx,
                'dedx_sigma': init_log_sigma,
                'opt_state':  local_optimizer.init(init_local),
            }

        local_params = {
            'log_dedx_mean':  local_cache[batch_idx]['dedx_mean'],
            'log_dedx_sigma': local_cache[batch_idx]['dedx_sigma'],
        }
        local_opt_state = local_cache[batch_idx]['opt_state']

        # 3. Inner loop: optimise local dEdx only
        for _ in range(inner_steps):
            rng_key, step_key = jax.random.split(rng_key)
            (loss_val, aux), (_, grad_local) = vg_loss(
                global_theta, local_params, tracks_batch, target_output_batch, step_key
            )
            local_updates, local_opt_state = local_optimizer.update(
                grad_local, local_opt_state, local_params
            )
            local_params = optax.apply_updates(local_params, local_updates)

        # 4. Outer step: optimise global physics parameter
        rng_key, step_key = jax.random.split(rng_key)
        (loss_val, aux), (grad_global, _) = vg_loss(
            global_theta, local_params, tracks_batch, target_output_batch, step_key
        )
        global_updates, global_opt_state = global_optimizer.update(
            grad_global, global_opt_state, global_theta
        )
        global_theta = optax.apply_updates(global_theta, global_updates)

        # 5. Persist local state
        local_cache[batch_idx]['dedx_mean']  = local_params['log_dedx_mean']
        local_cache[batch_idx]['dedx_sigma'] = local_params['log_dedx_sigma']
        local_cache[batch_idx]['opt_state']  = local_opt_state

        # 6. Log metrics
        losses.append(loss_val)
        param_history.append(jnp.exp(global_theta))

        ev_idx = track_fields.index('eventID')
        valid_mask_log = tracks_batch[:, ev_idx] >= 0
        current_mean_scale = jnp.mean(
            jnp.exp(local_params['log_dedx_mean']) / tracks_batch[:, track_fields.index('dEdx')],
            where=valid_mask_log,
        )
        dedx_scale_history.append(current_mean_scale)
        dedx_dist_to_truth = jnp.mean(
            jnp.abs(jnp.exp(local_params['log_dedx_mean']) -
                    tracks_batch[:, track_fields.index('dEdx')]),
            where=valid_mask_log,
        )
        dedx_dist_to_truth_history.append(dedx_dist_to_truth)

        loss_history.append(aux['loss'])
        moyal_penalty_history.append(aux.get('moyal_penalty', 0.0))
        wass_penalty_history.append(aux.get('wass_penalty', 0.0))
        entropy_penalty_history.append(aux.get('entropy_penalty', 0.0))
        tv_penalty_history.append(aux.get('tv_penalty', 0.0))

        if batch_idx % 5 == 0 or batch_idx == num_batches - 1:
            print(
                f"Ep {epoch:02d} | B {batch_idx:02d} | "
                f"Loss={float(loss_val):.4f} | "
                f"{param_name}={float(jnp.exp(global_theta)):.5f} "
                f"({float(jnp.exp(global_theta)/getattr(current_params,param_name)):.3f}×nom) | "
                f"dEdx_dist={float(dedx_dist_to_truth):.4f} | "
                f"HitNLL={float(loss_history[-1]):.4f} | "
                f"Moyal={float(moyal_penalty_history[-1]):.4f} | "
                f"Wass={float(wass_penalty_history[-1]):.4f} | "
                f"TV={float(tv_penalty_history[-1]):.4f} | "
                f"Ent={float(entropy_penalty_history[-1]):.4f}"
            )


In [19]:

import numpy as np
import matplotlib.pyplot as plt

losses_np      = np.array([float(x) for x in losses])
param_np       = np.array([float(x) for x in param_history])
dedx_dist_np   = np.array([float(x) for x in dedx_dist_to_truth_history])
dedx_scale_np  = np.array([float(x) for x in dedx_scale_history])
hitnll_np      = np.array([float(x) for x in loss_history])
moyal_np       = np.array([float(x) for x in moyal_penalty_history])
wass_np        = np.array([float(x) for x in wass_penalty_history])
tv_np          = np.array([float(x) for x in tv_penalty_history])
ent_np         = np.array([float(x) for x in entropy_penalty_history])

steps = np.arange(len(losses_np))
nom   = float(getattr(current_params, param_name))

# ── Plot 1: total loss + parameter trace ──────────────────────────────────────
fig, ax1 = plt.subplots(figsize=(8, 4))
ax1.semilogy(steps, losses_np, color='tab:blue', label='Total loss')
ax1.semilogy(steps, hitnll_np, color='tab:cyan', linestyle='--', label='Hit NLL')
ax1.set_xlabel('Step')
ax1.set_ylabel('Loss (log scale)', color='tab:blue')
ax1.tick_params(axis='y', labelcolor='tab:blue')

ax2 = ax1.twinx()
ax2.plot(steps, param_np / nom, color='tab:green', label=f'{param_name} / nominal')
ax2.plot(steps, dedx_scale_np, color='tab:red', alpha=0.6, label='dEdx scale')
ax2.axhline(1.0, color='grey', linestyle=':', lw=0.8)
ax2.set_ylabel('Relative value', color='tab:green')
ax2.tick_params(axis='y', labelcolor='tab:green')

lines1, labs1 = ax1.get_legend_handles_labels()
lines2, labs2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labs1 + labs2, loc='upper right', fontsize=8)
fig.tight_layout()
plt.title('Optimisation trace — loss and parameter recovery')
plt.show()

# ── Plot 2: penalty breakdown ─────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(10, 6))
axes[0, 0].plot(steps, moyal_np);  axes[0, 0].set_title('Moyal NLL (per-segment prior)'); axes[0, 0].set_xlabel('Step')
axes[0, 1].plot(steps, wass_np);   axes[0, 1].set_title('Wasserstein prior (global shape)'); axes[0, 1].set_xlabel('Step')
axes[1, 0].plot(steps, tv_np);     axes[1, 0].set_title('TV penalty (log dEdx smoothness)'); axes[1, 0].set_xlabel('Step')
axes[1, 1].plot(steps, ent_np);    axes[1, 1].set_title('Entropy penalty (sigma regulariser)'); axes[1, 1].set_xlabel('Step')
fig.tight_layout()
plt.suptitle('Penalty breakdown', y=1.01)
plt.show()

# ── Plot 3: dEdx distance to truth ────────────────────────────────────────────
plt.figure(figsize=(7, 3))
plt.plot(steps, dedx_dist_np, color='tab:orange')
plt.xlabel('Step')
plt.ylabel('Mean |fitted dEdx - true dEdx|  (MeV/cm)')
plt.title('Per-segment dEdx distance to MC truth')
plt.grid(True)
plt.tight_layout()
plt.show()

print(f"\nFinal {param_name} = {float(param_np[-1]):.5f}  "
      f"(nominal = {nom:.5f},  ratio = {float(param_np[-1]/nom):.4f})")


In [ ]:

# Scan hit loss vs beta for different dEdx choices
import numpy as np

# Use batch 0 to match local_cache indexing
tracks_batch = dataloader[0]
tracks_batch = jax.device_put(tracks_batch.reshape(-1, len(track_fields)))
target_output_batch = format_target_output(
    target_strategy.predict(current_params, tracks_batch, track_fields, 0)
)

dedx_index = track_fields.index('dEdx')
dE_index = track_fields.index('dE')
dx_index = track_fields.index('dx')
event_index = track_fields.index('eventID')

true_dedx = tracks_batch[:, dedx_index]
event_ids = tracks_batch[:, event_index]
valid_mask = event_ids >= 0

if 0 not in local_cache:
    raise ValueError("local_cache is empty for batch 0. Run the optimization cell first.")

fitted_dedx = jnp.exp(local_cache[0]['dedx_mean'])
flat_dedx_value = jnp.mean(true_dedx, where=valid_mask)
flat_dedx = jnp.full_like(true_dedx, flat_dedx_value)

def build_tracks_with_dedx(dedx_values):
    updated_tracks = tracks_batch.at[:, dedx_index].set(dedx_values)
    updated_tracks = updated_tracks.at[:, dE_index].set(dedx_values * updated_tracks[:, dx_index])
    return updated_tracks

alpha_base = getattr(current_params, 'beta')
alpha_grid = jnp.linspace(0.6 * alpha_base, 1.4 * alpha_base, 25)

def scan_loss_for_dedx(dedx_values):
    scan_tracks = build_tracks_with_dedx(dedx_values)
    losses = []
    for alpha_val in alpha_grid:
        params = current_params.replace(beta=float(alpha_val))
        pred = pred_strategy.predict(params, scan_tracks, track_fields, 0)
        loss_val, _ = loss_strategy.compute(params, pred, target_output_batch)
        losses.append(float(loss_val))
    return np.array(losses)

loss_true = scan_loss_for_dedx(true_dedx)
loss_fitted = scan_loss_for_dedx(fitted_dedx)
loss_flat = scan_loss_for_dedx(flat_dedx)

plt.figure(figsize=(7, 4))
plt.plot(alpha_grid, loss_true, label='True dEdx')
plt.plot(alpha_grid, loss_fitted, label='Fitted dEdx (mean)')
plt.plot(alpha_grid, loss_flat, label='Flat dEdx (mean value)')
plt.axvline(x=alpha_base, color='gray', linestyle='--', label='Initial beta')
plt.xlabel('beta')
plt.ylabel('Hit loss')
plt.title('Hit loss vs beta for different dEdx choices')
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
with jax.debug_nans(True):
    target = target_strategy.predict(ref_params, tracks, track_fields, 42)
    pred = pred_strategy.predict(ref_params, tracks, track_fields, 42)['adcs_distrib'].sum()

In [ ]:

# --- Phase 1.3: Gradient flow validation ---
# Checks that gradients w.r.t. a single physics parameter and per-segment dEdx
# are non-zero and finite, starting from near-truth initialization.
from jax import tree_util

param_name_grad_check = 'beta'   # change to test other params
rng_key_gc = jax.random.PRNGKey(0)

tracks_batch_gc = jax.device_put(dataloader[0].reshape(-1, len(track_fields)))
target_gc = format_target_output(
    target_strategy.predict(current_params, tracks_batch_gc, track_fields, 0)
)

local_params_gc = {
    'log_dedx_mean': jnp.log(jnp.maximum(tracks_batch_gc[:, track_fields.index('dEdx')], 1e-6)),
    'log_dedx_sigma': jnp.log(jnp.full_like(tracks_batch_gc[:, track_fields.index('dEdx')], 0.1)),
}
global_log_param_gc = jnp.log(getattr(current_params, param_name_grad_check))
_, grad_key_gc = jax.random.split(rng_key_gc)

def grad_test_fn(g_log_param, l_params, key):
    return compute_loss(
        {
            'log_param': g_log_param,
            'log_dedx_mean': l_params['log_dedx_mean'],
            'log_dedx_sigma': l_params['log_dedx_sigma'],
        },
        target_gc,
        tracks_batch_gc,
        current_params,
        track_fields,
        param_name_grad_check,
        key,
    )[0]

vg_gc = jax.value_and_grad(grad_test_fn, argnums=(0, 1))

with jax.debug_nans(True):
    loss_gc, (g_param, g_local) = vg_gc(global_log_param_gc, local_params_gc, grad_key_gc)

print(f"Loss: {float(loss_gc):.5f}")
print(f"grad(log_{param_name_grad_check}) = {float(g_param):.6e}")
print(f"grad(log_dedx_mean): mean={float(jnp.mean(g_local['log_dedx_mean'])):.4e}  "
      f"max|g|={float(jnp.max(jnp.abs(g_local['log_dedx_mean']))):.4e}")
print(f"grad(log_dedx_sigma): mean={float(jnp.mean(g_local['log_dedx_sigma'])):.4e}  "
      f"max|g|={float(jnp.max(jnp.abs(g_local['log_dedx_sigma']))):.4e}")

has_nan = (jnp.any(jnp.isnan(g_local['log_dedx_mean'])) |
           jnp.any(jnp.isnan(g_local['log_dedx_sigma'])) |
           jnp.isnan(g_param))
print(f"Any NaN in gradients: {bool(has_nan)}")


In [ ]:
# (stale cell retained for backwards compat — gradient check is now in Phase 1.3 above)
# g_log_param_grad  ← this was undefined; use the Phase 1.3 cell instead
print("Gradient check results are in the Phase 1.3 cell above.")


In [ ]:
# (stale cell — grad_test_fn / global_log_param / local_params / grad_key now live
#  in the Phase 1.3 cell above with _gc-suffixed names)
print("See Phase 1.3 gradient check cell for updated diagnostics.")


In [68]:
# Compare target hit times with most probable predicted tick per hit (using retrigger index)
from optimize.strategies import compute_occurrence_indices

target_pixels = target['hit_pixels']
target_ticks = target['ticks'].astype(int)
target_adcs = target['adcs']
pred_pixels = pred['unique_pixels']
pred_ticks_prob = pred['ticks_prob']  # (Npix, Ntriggers, Nticks)
pred_adcs_distrib = pred['adcs_distrib']  # (Npix, Ntriggers, Nticks)

# Map target pixels to prediction pixel indices
pred_pixel_idx = jnp.searchsorted(pred_pixels, target_pixels)
pred_pixel_idx = jnp.clip(pred_pixel_idx, 0, pred_pixels.shape[0] - 1)
pixel_match = (pred_pixels[pred_pixel_idx] == target_pixels) & (target_pixels >= 0)

# Compute retrigger (occurrence) index per hit, consistent with ProbabilisticLossStrategy
trigger_nb = compute_occurrence_indices(target_pixels)

# Gather per-hit tick distributions for the matched pixel+trigger
pred_tick_prob_hit = pred_ticks_prob[pred_pixel_idx, trigger_nb, :]
pred_tick_argmax_hit = jnp.argmax(pred_tick_prob_hit, axis=1)

matched_target_ticks = target_ticks[pixel_match]
matched_pred_ticks = pred_tick_argmax_hit[pixel_match]

tick_diff = matched_pred_ticks - matched_target_ticks
abs_tick_diff = jnp.abs(tick_diff)

print(f"Matched hits: {matched_target_ticks.shape[0]} / {target_ticks.shape[0]}")
print(f"Mean |tick diff|: {float(jnp.mean(abs_tick_diff)):.3f}")
print(f"Median |tick diff|: {float(jnp.median(abs_tick_diff)):.3f}")

plt.hist(tick_diff, bins=50, range=(-50, 50), histtype='step', density=True)
plt.xlabel('Pred tick - target tick')
plt.ylabel('Density')
plt.title('Tick difference for matched hits (with retrigger index)')
plt.show()

# Per-hit tick term distribution (already log-prob in model output)
hit_tick_probs = pred_ticks_prob[pred_pixel_idx, trigger_nb, target_ticks]
hit_tick_probs = hit_tick_probs[pixel_match]
nll_tick = hit_tick_probs
print(f"Mean tick term per hit: {float(jnp.mean(nll_tick)):.4f}")
print(f"Median tick term per hit: {float(jnp.median(nll_tick)):.4f}")

plt.hist(nll_tick, bins=60, range=(0, 20), histtype='step', density=True)
plt.xlabel('Tick term per hit')
plt.ylabel('Density')
plt.title('Distribution of tick term per hit')
plt.show()

# Per-hit charge term distribution (use Gaussian log-likelihood; no extra log)
target_charge = adc2charge(target_adcs, ref_params)
hit_expected_charges = adc2charge(
    pred_adcs_distrib[pred_pixel_idx, trigger_nb, target_ticks], ref_params
 )
charge_diff = target_charge - hit_expected_charges
charge_diff = charge_diff[pixel_match]
sigma_charge = loss_strategy.sigma_charge
print(f"sigma_charge used: {float(sigma_charge):.6f}")
print(f"Charge diff mean: {float(jnp.mean(charge_diff)):.3f}")
print(f"Charge diff std: {float(jnp.std(charge_diff)):.3f}")

# If sigma_charge is already in electrons, do not scale by 1e-3
log_likelihood_charge = (
    -0.5 * (charge_diff / 500) ** 2
    - 0.5 * jnp.log(2 * jnp.pi * 500 ** 2)
 )
print(f"Mean charge term per hit: {float(jnp.mean(log_likelihood_charge)):.4f}")
print(f"Median charge term per hit: {float(jnp.median(log_likelihood_charge)):.4f}")

plt.hist(log_likelihood_charge, bins=60, range=(-50, 5), histtype='step', density=True)
plt.xlabel('Charge term per hit')
plt.ylabel('Density')
plt.title('Distribution of charge term per hit')
plt.show()

# Expected vs target charge (per hit)
matched_target_charge = target_charge[pixel_match]
matched_expected_charge = hit_expected_charges[pixel_match]

plt.hist2d(
    matched_target_charge,
    matched_expected_charge,
    bins=50,
    range=((4, 40), (4, 40)),
    cmap='viridis'
 )
plt.xlabel('Target charge (e-)')
plt.ylabel('Expected charge (e-)')
plt.title('Expected vs target charge (matched hits)')
plt.colorbar(label='Counts')
plt.show()

In [ ]:

# Compare target/pred tick+charge differences pre-fit vs post-fit
from optimize.strategies import compute_occurrence_indices

def compute_tick_charge_diffs(target_data, pred_data, params):
    target_pixels = target_data['hit_pixels']
    target_ticks = target_data['ticks'].astype(int)
    target_adcs = target_data['adcs']
    pred_pixels = pred_data['unique_pixels']
    pred_ticks_prob = pred_data['ticks_prob']
    pred_adcs_distrib = pred_data['adcs_distrib']

    pred_pixel_idx = jnp.searchsorted(pred_pixels, target_pixels)
    pred_pixel_idx = jnp.clip(pred_pixel_idx, 0, pred_pixels.shape[0] - 1)
    pixel_match = (pred_pixels[pred_pixel_idx] == target_pixels) & (target_pixels >= 0)

    trigger_nb = compute_occurrence_indices(target_pixels)

    pred_tick_prob_hit = pred_ticks_prob[pred_pixel_idx, trigger_nb, :]
    pred_tick_argmax_hit = jnp.argmax(pred_tick_prob_hit, axis=1)

    matched_target_ticks = target_ticks[pixel_match]
    matched_pred_ticks = pred_tick_argmax_hit[pixel_match]
    tick_diff = matched_pred_ticks - matched_target_ticks

    target_charge = adc2charge(target_adcs, params)
    hit_expected_charges = adc2charge(
        pred_adcs_distrib[pred_pixel_idx, trigger_nb, target_ticks], params
    )
    charge_diff = (hit_expected_charges - target_charge)[pixel_match]

    return tick_diff, charge_diff

# Pre-fit prediction (ref params, original tracks)
target_prefit = target_strategy.predict(ref_params, tracks, track_fields, 42)
pred_prefit = pred_strategy.predict(ref_params, tracks, track_fields, 42)
tick_diff_pre, charge_diff_pre = compute_tick_charge_diffs(target_prefit, pred_prefit, ref_params)

# Post-fit prediction (use fitted global param + fitted dEdx for batch 0)
if len(local_cache) == 0:
    raise ValueError("local_cache is empty. Run the optimization cell first.")

tracks_post = tracks
if 0 in local_cache:
    fitted_log_dedx = local_cache[0]['dedx_mean']   # key is 'dedx_mean', not 'dedx'
    dedx_index = track_fields.index('dEdx')
    dE_index = track_fields.index('dE')
    dx_index = track_fields.index('dx')
    fitted_dedx = jnp.exp(fitted_log_dedx)
    tracks_post = tracks_post.at[:, dedx_index].set(fitted_dedx)
    tracks_post = tracks_post.at[:, dE_index].set(fitted_dedx * tracks_post[:, dx_index])

post_params = ref_params.replace(**{param_name: jnp.exp(global_theta)})
target_post = target_strategy.predict(post_params, tracks_post, track_fields, 42)
pred_post = pred_strategy.predict(post_params, tracks_post, track_fields, 42)
tick_diff_post, charge_diff_post = compute_tick_charge_diffs(target_post, pred_post, post_params)

plt.hist(tick_diff_pre, bins=60, range=(-40, 40), histtype='step', density=True, label='Pre-fit')
plt.hist(tick_diff_post, bins=60, range=(-40, 40), histtype='step', density=True, label='Post-fit')
plt.xlabel('Pred tick - target tick')
plt.ylabel('Density')
plt.title('Tick difference: pre vs post')
plt.legend()
plt.show()

plt.hist(charge_diff_pre, bins=60, range=(-1, 1), histtype='step', density=True, label='Pre-fit')
plt.hist(charge_diff_post, bins=60, range=(-1, 1), histtype='step', density=True, label='Post-fit')
plt.xlabel('Expected - target charge (e-)')
plt.ylabel('Density')
plt.title('Charge difference: pre vs post')
plt.legend()
plt.show()


In [121]:
# Inspect worst tick-diff hit and its predicted tick distribution
worse_idx = jnp.argsort(jnp.abs(tick_diff_pre))[-1]
pix_id = target_prefit['hit_pixels'][worse_idx]
retrig_index = compute_occurrence_indices(target_prefit['hit_pixels'])[worse_idx]
print(f"Worst hit pixel: {pix_id}, retrigger index: {retrig_index}, tick diff: {tick_diff_pre[worse_idx]}, charge diff: {charge_diff_pre[worse_idx]}")

# Match pixel id to prediction index
pred_pixels = pred_prefit['unique_pixels']
pred_pix_idx = jnp.searchsorted(pred_pixels, pix_id)
pred_pix_idx = jnp.clip(pred_pix_idx, 0, pred_pixels.shape[0] - 1)
if pred_pixels[pred_pix_idx] != pix_id:
    raise ValueError(f"Pixel {pix_id} not found in prediction unique_pixels")

pred_tick_prob = pred_prefit['ticks_prob'][pred_pix_idx, retrig_index, :]
pred_tick_argmax = int(jnp.argmax(pred_tick_prob))
target_tick = int(target_prefit['ticks'][worse_idx])

print(f"Pred tick argmax: {pred_tick_argmax}, target tick: {target_tick}")

plt.plot(pred_tick_prob, label='Pred tick log-prob')
plt.axvline(target_tick, color='r', label='Target tick')
plt.axvline(pred_tick_argmax, color='k', linestyle='--', label='Pred argmax')
plt.xlabel('Tick')
plt.ylabel('Log-prob')
plt.title('Predicted tick distribution for worst hit')
plt.legend()
plt.xlim(740, 800)
plt.show()

In [ ]:

# Pixel overlap statistics between target (stochastic) and prediction (probabilistic)
target_pixels = target['hit_pixels']
# BUG FIX: pred_strategy.predict() returns 'unique_pixels', not 'hit_pixels'
pred_pixels = pred['unique_pixels']

joint_pixels = jnp.unique(jnp.concatenate([target_pixels, pred_pixels], axis=0))
target_set = set(target_pixels.tolist())
pred_set = set(pred_pixels.tolist())
print("Number of pixels in both distributions:", len(target_set.intersection(pred_set)))
print("Pixels only in target:", len(target_set - pred_set))
print("Pixels only in prediction:", len(pred_set - target_set))
print(f"Pixel overlap fraction (target): {len(target_set.intersection(pred_set)) / max(1, len(target_set)):.3f}")
print(f"Pixel overlap fraction (prediction): {len(target_set.intersection(pred_set)) / max(1, len(pred_set)):.3f}")


In [40]:
# Aggregate fitted dEdx across all batches (mean and sigma)
if len(local_cache) == 0:
    raise ValueError("local_cache is empty. Run the optimization cell first.")

fitted_dedx_mean_all = jnp.concatenate([jnp.exp(v['dedx_mean']) for v in local_cache.values()])
fitted_dedx_sigma_all = jnp.concatenate([jnp.exp(v['dedx_sigma']) for v in local_cache.values()])

ref_dedx = jnp.concatenate([tracks_batch[:, track_fields.index('dEdx')] for tracks_batch in dataloader[:num_batches]])

plt.hist2d(ref_dedx[ref_dedx > 0], fitted_dedx_mean_all[ref_dedx > 0], bins=40, range=((1.5, 2.5), (1.5, 2.5)), cmap='jet')
plt.title('Initial vs fitted dE/dx mean (all batches)')
plt.colorbar(label='Counts')

plt.figure()
plt.hist(fitted_dedx_mean_all[ref_dedx > 0] - ref_dedx[ref_dedx > 0], bins=50, color='purple', alpha=0.7, range=(-1, 1))
plt.title('Difference between fitted mean and reference dE/dx')
plt.xlabel('Fitted mean - Reference dE/dx')
plt.ylabel('Counts')
plt.show()

plt.figure()
plt.hist(fitted_dedx_sigma_all, bins=80, range=(0, 2), histtype='step', label='Fitted dEdx sigma', density=True)
plt.xlabel('dE/dx sigma (MeV/cm)')
plt.ylabel('Density')
plt.title('Fitted dE/dx sigma (all batches)')
plt.legend()
plt.show()

In [41]:
# Aggregate fitted dEdx across all batches
if len(local_cache) == 0:
    raise ValueError("local_cache is empty. Run the optimization cell first.")

fitted_dedx_mean_all = jnp.concatenate([jnp.exp(v['dedx_mean']) for v in local_cache.values()])
fitted_dedx_sigma_all = jnp.concatenate([jnp.exp(v['dedx_sigma']) for v in local_cache.values()])

plt.hist(fitted_dedx_mean_all, bins=100, range=(0, 5), histtype='step', label='Fitted dEdx mean', density=True)
plt.hist(distrib, bins=100, range=(0, 5), histtype='step', label='Initial dEdx', density=True)
plt.xlabel('dE/dx (MeV/cm)')
plt.legend()
plt.show()

plt.hist(fitted_dedx_sigma_all, bins=100, range=(0, 2), histtype='step', label='Fitted dEdx sigma', density=True)
plt.xlabel('dE/dx sigma (MeV/cm)')
plt.legend()
plt.show()

In [ ]:
# Aggregate fitted dEdx across all batches
if len(local_cache) == 0:
    raise ValueError("local_cache is empty. Run the optimization cell first.")

fitted_log_dedx_mean_all = jnp.concatenate([v['dedx_mean'] for v in local_cache.values()])
fitted_log_dedx_sigma_all = jnp.concatenate([v['dedx_sigma'] for v in local_cache.values()])

plt.hist(fitted_log_dedx_mean_all, bins=100, range=(-5, 7), histtype='step', label='Fitted log(dEdx mean)', density=True)
plt.hist(jnp.log(distrib), bins=100, range=(-5, 7), histtype='step', label='Initial log(dEdx)', density=True)
plt.xlabel('log(dE/dx) (log(MeV/cm))')
# plt.yscale('log')
plt.legend()
plt.show()

plt.hist(fitted_log_dedx_sigma_all, bins=100, range=(-6, 2), histtype='step', label='Fitted log(dEdx sigma)', density=True)
plt.xlabel('log(dE/dx sigma)')
plt.legend()
plt.show()

In [42]:
# Pull distribution for fitted dEdx mean/sigma vs true dEdx
if len(local_cache) == 0:
    raise ValueError("local_cache is empty. Run the optimization cell first.")

# Gather fitted mean/sigma and corresponding true dEdx across batches
fitted_dedx_mean_all = jnp.concatenate([jnp.exp(v['dedx_mean']) for v in local_cache.values()])
fitted_dedx_sigma_all = jnp.concatenate([jnp.exp(v['dedx_sigma']) for v in local_cache.values()])
true_dedx_all = jnp.concatenate([tracks_batch[:, track_fields.index('dEdx')] for tracks_batch in dataloader[:num_batches]])
event_ids_all = jnp.concatenate([tracks_batch[:, track_fields.index('eventID')] for tracks_batch in dataloader[:num_batches]])

valid_mask = (event_ids_all >= 0) & (fitted_dedx_sigma_all > 0)
pulls = (true_dedx_all - fitted_dedx_mean_all) / fitted_dedx_sigma_all
pulls = pulls[valid_mask]

plt.figure(figsize=(7, 4))
plt.hist(pulls, bins=80, range=(-5, 5), histtype='step', density=True, label='Pulls')
x = np.linspace(-5, 5, 400)
plt.plot(x, (1.0 / np.sqrt(2 * np.pi)) * np.exp(-0.5 * x**2), label='N(0,1)')
plt.xlabel('(true dEdx - fitted mean) / fitted sigma')
plt.ylabel('Density')
plt.title('dEdx pull distribution')
plt.legend()
plt.grid(True)
plt.show()

print(f"Pull mean: {float(jnp.mean(pulls)):.4f}")
print(f"Pull std: {float(jnp.std(pulls)):.4f}")

In [13]:
# 1. Compute differences for the entire array (Shift by 1)
if len(local_cache) == 0:
    raise ValueError("local_cache is empty. Run the optimization cell first.")

log_dedx_mean = local_cache[0]['dedx_mean']
diffs = jnp.diff(log_dedx_mean)
track_ids = tracks[:, track_fields.index('trackID')]
event_ids = tracks[:, track_fields.index('eventID')]
# 2. Create a mask: True if the adjacent hits belong to the same track
# Shape is also (N-1,)
same_track_mask = (track_ids[1:] == track_ids[:-1]) & (event_ids[1:] == event_ids[:-1])

# 3. Apply the mask to zero out jumps between different tracks
valid_diffs = jnp.abs(diffs) * same_track_mask

# 4. Compute the mean penalty, dividing only by the number of valid pairs
# jnp.maximum prevents division by zero if there are no valid pairs
plt.hist(diffs[same_track_mask], bins=100, range=(-1, 1), histtype='step', label='Adjacent log-dEdx mean differences (same track)', density=True);

In [20]:
plt.hist(learnable_dedx, bins=50)
plt.xlabel('dE/dx')
plt.ylabel('Frequency')
plt.title('Histogram of Learnable dE/dx')
plt.show()

In [ ]:

# Gradient diagnostic - recompute grads at the current optimizer state
# (replaces the undefined 'grads' variable that was previously here)
if len(local_cache) == 0 or 'vg_loss' not in dir():
    raise RuntimeError("Run the optimization cell (bi-level training loop) first.")

_tracks_check = jax.device_put(dataloader[0].reshape(-1, len(track_fields)))
_target_check = format_target_output(
    target_strategy.predict(current_params, _tracks_check, track_fields, 0)
)

_last_local = {
    'log_dedx_mean': local_cache[0]['dedx_mean'],
    'log_dedx_sigma': local_cache[0]['dedx_sigma'],
}

_rng_check = jax.random.PRNGKey(99)
(loss_check, aux_check), (g_global, g_local) = vg_loss(
    global_theta, _last_local, _tracks_check, _target_check, _rng_check
)

print(f"Loss at current state : {float(loss_check):.5f}")
print(f"grad(log_{param_name}) = {float(g_global):.6e}")
print(f"grad(log_dedx_mean)  : mean={float(jnp.mean(g_local['log_dedx_mean'])):.4e}  "
      f"std={float(jnp.std(g_local['log_dedx_mean'])):.4e}  "
      f"max|g|={float(jnp.max(jnp.abs(g_local['log_dedx_mean']))):.4e}")
print(f"grad(log_dedx_sigma) : mean={float(jnp.mean(g_local['log_dedx_sigma'])):.4e}  "
      f"std={float(jnp.std(g_local['log_dedx_sigma'])):.4e}")
_has_nan = (jnp.any(jnp.isnan(g_local['log_dedx_mean'])) |
            jnp.any(jnp.isnan(g_local['log_dedx_sigma'])) |
            jnp.isnan(g_global))
print(f"Any NaN in gradients : {bool(_has_nan)}")


In [15]:

# ═══════════════════════════════════════════════════════════════════════════════
# PHASE 3 — Joint optimizer (fixed: truly global physics parameter)
#
# Root cause of poor convergence in the previous version:
#   log_param was stored inside joint_cache[batch_idx], so every batch had its
#   own independent copy of beta that never shared information across batches.
#
# This version:
#   • log_param is a single shared scalar — NOT stored per-batch.
#     It receives a gradient contribution from every batch at every step.
#   • Two separate adam instances: param_opt (slow) and dedx_opt (fast).
#   • Dedx warmup: first N_WARMUP epochs only optimise dEdx; beta stays frozen.
#     This lets the nuisance parameters converge before the physics param moves,
#     so the gradient of the hit NLL w.r.t. beta is not fighting an unconverged
#     dEdx landscape.
#   • Targets are pre-generated once per epoch (deterministic seed = epoch*100+b)
#     to remove stochastic noise from re-sampling in the beta gradient signal.
#   • _LW_WASS = 1.0 locally (global LAMBDA_WASSERSTEIN = 10 untouched).
#     A large Wasserstein weight anchors the dEdx shape tightly to the reference,
#     which indirectly absorbs any charge-scale shift and prevents beta from moving.
# ═══════════════════════════════════════════════════════════════════════════════
import jax, optax, numpy as np, matplotlib.pyplot as plt
import jax.numpy as jnp
from jax import value_and_grad

# ── Hyper-parameters ──────────────────────────────────────────────────────────
JOINT_PARAM_NAME    = 'beta'
JOINT_NUM_BATCHES   = 5
JOINT_NUM_EPOCHS    = 150
N_WARMUP            = 20      # epochs to warm up dEdx before unfreezing beta
JOINT_LR_PARAMS     = 1e-3   # conservative: beta drifts slowly
JOINT_LR_DEDX       = 5e-3   # fast: nuisance dEdx adapts at every step
_LW_WASS            = 1.0    # local Wasserstein weight (global = 10.0, too high)

# ── Single global physics parameter + its own optimizer ──────────────────────
_nom             = float(getattr(current_params, JOINT_PARAM_NAME))
joint_log_param  = jnp.log(jnp.array(_nom * 1.2, dtype=jnp.float32))  # +20 % offset
param_opt        = optax.adam(JOINT_LR_PARAMS)
param_opt_state  = param_opt.init(joint_log_param)

# ── Per-batch dedx cache + its own optimizer ──────────────────────────────────
dedx_opt       = optax.adam(JOINT_LR_DEDX)
local_cache_v2 = {}   # batch_idx -> {'log_dedx_mean', 'log_dedx_sigma', 'opt_state'}

joint_rng = jax.random.PRNGKey(42)

# ── Loss: log_param and local_params are separate args so gradients are clean ─
def joint_loss_fn_v2(log_p, local_p, target_out, tracks_b, key):
    theta = {
        'log_param':      log_p,
        'log_dedx_mean':  local_p['log_dedx_mean'],
        'log_dedx_sigma': local_p['log_dedx_sigma'],
    }
    return compute_loss(theta, target_out, tracks_b,
                        current_params, track_fields, JOINT_PARAM_NAME, key,
                        lambda_wass=_LW_WASS)

vg_v2 = value_and_grad(joint_loss_fn_v2, argnums=(0, 1), has_aux=True)

# ── Metric histories ──────────────────────────────────────────────────────────
jt_losses = []; jt_param_history = []; jt_dedx_dist = []
jt_hitnll = []; jt_moyal = []; jt_wass = []
jt_grad_param_epochs = []  # per-epoch mean |∇ log β|

# ── Training loop ─────────────────────────────────────────────────────────────
print(f"{'Ep':>4}  {'beta':>9}  {'β/nom':>7}  {'|∇logβ|':>10}  "
      f"{'HitNLL':>8}  {'dEdxDist':>9}  {'phase'}")
print("-" * 68)

for epoch in range(JOINT_NUM_EPOCHS):
    grad_param_epoch = []

    # Pre-generate all targets for this epoch (fixed per-epoch seed → no noise)
    epoch_targets = {
        b: format_target_output(
            target_strategy.predict(
                current_params,
                jax.device_put(dataloader[b].reshape(-1, len(track_fields))),
                track_fields, b * 100 + epoch,
            )
        )
        for b in range(min(JOINT_NUM_BATCHES, len(dataloader)))
    }

    for batch_idx in range(min(JOINT_NUM_BATCHES, len(dataloader))):
        tracks_b = jax.device_put(dataloader[batch_idx].reshape(-1, len(track_fields)))
        target_b = epoch_targets[batch_idx]

        # Initialise per-batch dedx on first encounter
        if batch_idx not in local_cache_v2:
            _lm = jnp.full((tracks_b.shape[0],), jnp.log(MOYAL_MPV), dtype=jnp.float32)
            _ls = jnp.full_like(_lm, jnp.log(MOYAL_XI))
            _lp = {'log_dedx_mean': _lm, 'log_dedx_sigma': _ls}
            local_cache_v2[batch_idx] = {**_lp, 'opt_state': dedx_opt.init(_lp)}

        local_p = {
            'log_dedx_mean':  local_cache_v2[batch_idx]['log_dedx_mean'],
            'log_dedx_sigma': local_cache_v2[batch_idx]['log_dedx_sigma'],
        }
        dedx_state = local_cache_v2[batch_idx]['opt_state']

        joint_rng, step_key = jax.random.split(joint_rng)
        (loss_val, aux), (g_param, g_local) = vg_v2(
            joint_log_param, local_p, target_b, tracks_b, step_key
        )

        # Always update dedx nuisance
        dedx_updates, dedx_state = dedx_opt.update(g_local, dedx_state, local_p)
        local_p = optax.apply_updates(local_p, dedx_updates)
        local_cache_v2[batch_idx].update({
            'log_dedx_mean':  local_p['log_dedx_mean'],
            'log_dedx_sigma': local_p['log_dedx_sigma'],
            'opt_state':      dedx_state,
        })

        # Only update global beta after warmup
        if epoch >= N_WARMUP:
            p_upd, param_opt_state = param_opt.update(g_param, param_opt_state)
            joint_log_param = optax.apply_updates(joint_log_param, p_upd)

        grad_param_epoch.append(float(jnp.abs(g_param)))

        ev_idx = track_fields.index('eventID')
        vmask  = tracks_b[:, ev_idx] >= 0
        jt_losses.append(float(loss_val))
        jt_param_history.append(float(jnp.exp(joint_log_param)))
        jt_dedx_dist.append(float(jnp.mean(
            jnp.abs(jnp.exp(local_p['log_dedx_mean']) - tracks_b[:, track_fields.index('dEdx')]),
            where=vmask)))
        jt_hitnll.append(float(aux['loss']))
        jt_moyal.append(float(aux.get('moyal_penalty', 0.0)))
        jt_wass.append(float(aux.get('wass_penalty', 0.0)))

    jt_grad_param_epochs.append(float(np.mean(grad_param_epoch)))
    if epoch % 10 == 0 or epoch == JOINT_NUM_EPOCHS - 1:
        phase = "warmup " if epoch < N_WARMUP else "fitting"
        print(f"{epoch:4d}  {float(jnp.exp(joint_log_param)):9.5f}  "
              f"{float(jnp.exp(joint_log_param))/_nom:7.4f}  "
              f"{jt_grad_param_epochs[-1]:10.3e}  "
              f"{jt_hitnll[-1]:8.4f}  {jt_dedx_dist[-1]:9.4f}  {phase}")

# ── Trace plots ───────────────────────────────────────────────────────────────
_steps  = np.arange(len(jt_losses))
_ep     = np.arange(len(jt_grad_param_epochs))
_warmup_step = N_WARMUP * JOINT_NUM_BATCHES

fig, axes = plt.subplots(3, 1, figsize=(9, 9), sharex=False)

# — beta recovery —
axes[0].plot(_steps, np.array(jt_param_history) / _nom,
             color='tab:green', lw=1.2, label=f'{JOINT_PARAM_NAME} / nominal')
axes[0].axhline(1.0, color='grey',   linestyle=':', lw=0.9, label='Target (nominal)')
axes[0].axhline(1.2, color='red',    linestyle=':', lw=0.9, label='Init (+20 %)')
axes[0].axvline(_warmup_step, color='orange', linestyle='--', lw=1.0, label='Warmup ends')
axes[0].set_ylabel(f'{JOINT_PARAM_NAME} / nominal')
axes[0].set_title('Phase 3 — joint optimiser with shared global beta')
axes[0].legend(fontsize=8); axes[0].grid(True, alpha=0.3)

# — loss decomposition —
axes[1].semilogy(_steps, jt_losses, color='tab:blue', lw=1.0, label='Total loss')
axes[1].semilogy(_steps, jt_hitnll, color='tab:cyan', lw=1.0, ls='--', label='Hit NLL')
axes[1].semilogy(_steps, np.maximum(jt_moyal, 1e-9),
                 color='tab:orange', lw=0.8, ls='-.', label='Moyal NLL')
axes[1].semilogy(_steps, np.maximum(jt_wass, 1e-9),
                 color='tab:purple', lw=0.8, ls=':', label='Wasserstein')
axes[1].axvline(_warmup_step, color='orange', linestyle='--', lw=1.0)
axes[1].set_ylabel('Loss (log scale)'); axes[1].legend(fontsize=8, ncol=2)
axes[1].grid(True, alpha=0.3)

# — gradient magnitude per epoch —
axes[2].semilogy(_ep, jt_grad_param_epochs, color='tab:red', lw=1.2)
axes[2].axvline(N_WARMUP, color='orange', linestyle='--', lw=1.0, label='Warmup ends')
axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('Mean |∇ log β| per epoch')
axes[2].set_title('Gradient magnitude of beta — should be non-zero after warmup')
axes[2].legend(fontsize=8); axes[2].grid(True, alpha=0.3)

fig.tight_layout()
plt.show()

beta_final = float(jnp.exp(joint_log_param))
print(f"\nFinal {JOINT_PARAM_NAME} = {beta_final:.6f}  "
      f"(nominal = {_nom:.6f},  residual offset = {100*(beta_final/_nom - 1):+.2f} %)")
print(f"Mean |∇ log β| during fitting (post-warmup): "
      f"{float(np.mean(jt_grad_param_epochs[N_WARMUP:])):.3e}")


In [16]:

# ═══════════════════════════════════════════════════════════════════════════════
# PHASE 3b — Same-seed closure test
#
# Motivation: in Phase 3a the residual non-convergence could be caused by
# electronics-noise fluctuations: the target is simulated with seed A while the
# prediction is evaluated with seed B → the NLL residual has a stochastic floor
# even when beta is exactly correct.
#
# Here we pass the SAME noise seed to both the target generator and the
# probabilistic prediction:
#   noise_seed  = epoch * 1000 + batch_idx          (deterministic per step)
#   target      = target_strategy.predict(…, noise_seed)
#   pred key    = jax.random.PRNGKey(noise_seed)    (identical noise path)
#
# Consequence: when beta = beta_true AND dedx = dedx_true the loss is exactly
# the NLL of a distribution evaluated at its own sample, i.e. the theoretical
# minimum.  Any residual offset of beta from the target value is therefore
# entirely due to the optimiser / gradient quality, not noise variance.
#
# This is an upper-bound on achievable convergence.  If beta does NOT converge
# here, the issue is in the gradient signal (model mis-specification or
# regulariser dominance), not in noise.
# ═══════════════════════════════════════════════════════════════════════════════
import jax, optax, numpy as np, matplotlib.pyplot as plt
import jax.numpy as jnp
from jax import value_and_grad

# ── Hyper-parameters (same as Phase 3a; can be tuned independently) ──────────
SS_PARAM_NAME   = 'beta'
SS_NUM_BATCHES  = 5
SS_NUM_EPOCHS   = 150
SS_N_WARMUP     = 5        # short warmup — noise is gone so signal is strong
SS_LR_PARAMS    = 2e-3     # slightly more aggressive
SS_LR_DEDX      = 5e-3
SS_LW_WASS      = 1.0      # keep Wasserstein weight low

# ── Same-seed loss function ───────────────────────────────────────────────────
# We thread `noise_seed_int` into the function so it can be used to set
# step_key deterministically instead of pulling from a global PRNG chain.
def ss_loss_fn(log_p, local_p, target_out, tracks_b, noise_seed_int):
    """Same-seed variant: prediction key derived from same integer as target."""
    key = jax.random.PRNGKey(noise_seed_int)
    theta = {
        'log_param':      log_p,
        'log_dedx_mean':  local_p['log_dedx_mean'],
        'log_dedx_sigma': local_p['log_dedx_sigma'],
    }
    return compute_loss(theta, target_out, tracks_b,
                        current_params, track_fields, SS_PARAM_NAME, key,
                        lambda_wass=SS_LW_WASS)

# NOTE: noise_seed_int is static (not traced), mark it as such via closure.
# We build a fresh closure each step so JAX doesn't try to trace the int.
def make_vg(seed_int):
    def _loss(log_p, local_p, target_out, tracks_b):
        return ss_loss_fn(log_p, local_p, target_out, tracks_b, seed_int)
    return value_and_grad(_loss, argnums=(0, 1), has_aux=True)

# ── Global physics parameter ──────────────────────────────────────────────────
_ss_nom          = float(getattr(current_params, SS_PARAM_NAME))
ss_log_param     = jnp.log(jnp.array(_ss_nom * 1.2, dtype=jnp.float32))  # +20 % offset
ss_param_opt     = optax.adam(SS_LR_PARAMS)
ss_param_state   = ss_param_opt.init(ss_log_param)

# ── Per-batch dedx cache ──────────────────────────────────────────────────────
ss_dedx_opt   = optax.adam(SS_LR_DEDX)
ss_local_cache = {}

# ── Metric histories ──────────────────────────────────────────────────────────
ss_losses = []; ss_param_hist = []; ss_dedx_dist = []
ss_hitnll = []; ss_moyal = []; ss_wass = []
ss_grad_epochs = []

# ── Training loop ─────────────────────────────────────────────────────────────
print(f"{'Ep':>4}  {'beta':>9}  {'β/nom':>7}  {'|∇logβ|':>10}  "
      f"{'HitNLL':>8}  {'dEdxDist':>9}  {'phase'}")
print("-" * 68)

for epoch in range(SS_NUM_EPOCHS):
    grad_ep = []

    for batch_idx in range(min(SS_NUM_BATCHES, len(dataloader))):
        tracks_b = jax.device_put(dataloader[batch_idx].reshape(-1, len(track_fields)))

        # ── SAME seed for target and prediction ──────────────────────────────
        noise_seed = epoch * 1000 + batch_idx   # deterministic, shared

        target_b = format_target_output(
            target_strategy.predict(
                current_params, tracks_b, track_fields, noise_seed
            )
        )

        # Initialise per-batch dedx on first encounter
        if batch_idx not in ss_local_cache:
            _lm = jnp.full((tracks_b.shape[0],), jnp.log(MOYAL_MPV), dtype=jnp.float32)
            _ls = jnp.full_like(_lm, jnp.log(MOYAL_XI))
            _lp = {'log_dedx_mean': _lm, 'log_dedx_sigma': _ls}
            ss_local_cache[batch_idx] = {**_lp, 'opt_state': ss_dedx_opt.init(_lp)}

        local_p    = {k: ss_local_cache[batch_idx][k]
                      for k in ('log_dedx_mean', 'log_dedx_sigma')}
        dedx_state = ss_local_cache[batch_idx]['opt_state']

        # Build vg with the closed-over seed (avoids tracing the int)
        vg_step = make_vg(noise_seed)
        (loss_val, aux), (g_param, g_local) = vg_step(
            ss_log_param, local_p, target_b, tracks_b
        )

        # Update dedx nuisances (always)
        d_upd, dedx_state = ss_dedx_opt.update(g_local, dedx_state, local_p)
        local_p = optax.apply_updates(local_p, d_upd)
        ss_local_cache[batch_idx].update({
            'log_dedx_mean':  local_p['log_dedx_mean'],
            'log_dedx_sigma': local_p['log_dedx_sigma'],
            'opt_state':      dedx_state,
        })

        # Update global beta after warmup
        if epoch >= SS_N_WARMUP:
            p_upd, ss_param_state = ss_param_opt.update(g_param, ss_param_state)
            ss_log_param = optax.apply_updates(ss_log_param, p_upd)

        grad_ep.append(float(jnp.abs(g_param)))

        ev_idx = track_fields.index('eventID')
        vmask  = tracks_b[:, ev_idx] >= 0
        ss_losses.append(float(loss_val))
        ss_param_hist.append(float(jnp.exp(ss_log_param)))
        ss_dedx_dist.append(float(jnp.mean(
            jnp.abs(jnp.exp(local_p['log_dedx_mean']) - tracks_b[:, track_fields.index('dEdx')]),
            where=vmask)))
        ss_hitnll.append(float(aux['loss']))
        ss_moyal.append(float(aux.get('moyal_penalty', 0.0)))
        ss_wass.append(float(aux.get('wass_penalty', 0.0)))

    ss_grad_epochs.append(float(np.mean(grad_ep)))
    if epoch % 10 == 0 or epoch == SS_NUM_EPOCHS - 1:
        phase = "warmup " if epoch < SS_N_WARMUP else "fitting"
        print(f"{epoch:4d}  {float(jnp.exp(ss_log_param)):9.5f}  "
              f"{float(jnp.exp(ss_log_param))/_ss_nom:7.4f}  "
              f"{ss_grad_epochs[-1]:10.3e}  "
              f"{ss_hitnll[-1]:8.4f}  {ss_dedx_dist[-1]:9.4f}  {phase}")

# ── Trace plots ───────────────────────────────────────────────────────────────
_steps       = np.arange(len(ss_losses))
_ep          = np.arange(len(ss_grad_epochs))
_warmup_step = SS_N_WARMUP * SS_NUM_BATCHES

fig, axes = plt.subplots(3, 1, figsize=(9, 9), sharex=False)

axes[0].plot(_steps, np.array(ss_param_hist) / _ss_nom,
             color='tab:green', lw=1.5, label=f'{SS_PARAM_NAME} / nominal (same-seed)')
axes[0].axhline(1.0, color='grey',   linestyle=':', lw=1.0, label='True target')
axes[0].axhline(1.2, color='red',    linestyle=':', lw=1.0, label='Init (+20 %)')
axes[0].axvline(_warmup_step, color='orange', linestyle='--', lw=1.0, label='Warmup ends')
axes[0].set_ylabel(f'{SS_PARAM_NAME} / nominal')
axes[0].set_title('Phase 3b — same-seed closure test (electronics noise cancelled)')
axes[0].legend(fontsize=8); axes[0].grid(True, alpha=0.3)

axes[1].semilogy(_steps, ss_losses,  color='tab:blue',   lw=1.0, label='Total loss')
axes[1].semilogy(_steps, ss_hitnll,  color='tab:cyan',   lw=1.0, ls='--', label='Hit NLL')
axes[1].semilogy(_steps, np.maximum(ss_moyal, 1e-9),
                 color='tab:orange', lw=0.8, ls='-.', label='Moyal NLL')
axes[1].semilogy(_steps, np.maximum(ss_wass, 1e-9),
                 color='tab:purple', lw=0.8, ls=':', label='Wasserstein')
axes[1].axvline(_warmup_step, color='orange', linestyle='--', lw=1.0)
axes[1].set_ylabel('Loss (log scale)'); axes[1].legend(fontsize=8, ncol=2)
axes[1].grid(True, alpha=0.3)

axes[2].semilogy(_ep, ss_grad_epochs, color='tab:red', lw=1.2)
axes[2].axvline(SS_N_WARMUP, color='orange', linestyle='--', lw=1.0, label='Warmup ends')
axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('Mean |∇ log β| per epoch')
axes[2].set_title('Gradient magnitude — same-seed (should converge cleanly)')
axes[2].legend(fontsize=8); axes[2].grid(True, alpha=0.3)

fig.tight_layout()
plt.show()

ss_final = float(jnp.exp(ss_log_param))
print(f"\nSame-seed closure: final {SS_PARAM_NAME} = {ss_final:.6f}  "
      f"(nominal = {_ss_nom:.6f},  residual = {100*(ss_final/_ss_nom - 1):+.2f} %)")
print("→ If residual ≈ 0 %, the gradient signal is correct and noise was the main "
      "obstacle in Phase 3a.")
print("→ If residual ≠ 0 %, there is a model bias (regulariser dominance, gradient "
      "vanishing, or wrong dEdx initialisation).")


In [17]:

# ═══════════════════════════════════════════════════════════════════════════════
# PHASE 3c — Same-seed + three convergence fixes
#
# Root-cause analysis of the −2.1 % residual in Phase 3b:
#
#  1. Beta–dEdx degeneracy  (main):
#     Both parameters are updated simultaneously every step.  When beta is
#     slightly low, the nuisance dEdx can partially compensate by moving up,
#     settling onto a saddle instead of the true minimum.
#     FIX → Alternating freeze: odd epochs update only dEdx; even epochs
#            update only beta.  This decouples the two optimisations and
#            forces each sub-problem to converge independently.
#
#  2. LR overshoot near minimum:
#     A fixed LR of 2e-3 gives steps of ~0.4–0.8 in log-beta space —
#     too large for fine-tuning.  The gradient stays noisy (100–400) and
#     the parameter oscillates around the minimum without settling.
#     FIX → Cosine decay schedule: starts at 2e-3, decays to 5e-5 so the
#            final epochs make tiny, precise adjustments.
#
#  3. dEdx absorbs charge-scale information:
#     SS_LW_WASS = 1.0 is loose enough that nuisances drift to absorb the
#     charge-scale shift that should move beta.
#     FIX → Increase SS_LW_WASS to 3.0 to anchor the dEdx shape more
#            tightly to the reference distribution.
# ═══════════════════════════════════════════════════════════════════════════════
import jax, optax, numpy as np, matplotlib.pyplot as plt
import jax.numpy as jnp
from jax import value_and_grad

# ── Hyper-parameters ──────────────────────────────────────────────────────────
SC_PARAM_NAME   = 'beta'
SC_NUM_BATCHES  = 5
SC_NUM_EPOCHS   = 200
SC_N_WARMUP     = 5           # epochs: only dEdx updated (beta frozen)
SC_LR_DEDX      = 5e-3        # unchanged
SC_LW_WASS      = 3.0         # FIX 3: tighter anchor on dEdx shape

# FIX 2: cosine decay for beta LR  2e-3 → 5e-5 over the fitting phase
_fitting_steps  = (SC_NUM_EPOCHS - SC_N_WARMUP) * SC_NUM_BATCHES
sc_param_sched  = optax.cosine_decay_schedule(
    init_value=2e-3, decay_steps=_fitting_steps, alpha=0.025   # final = 0.025 × 2e-3 = 5e-5
)
sc_param_opt    = optax.adam(sc_param_sched)

# ── Same-seed loss closure ────────────────────────────────────────────────────
def sc_loss_fn(log_p, local_p, target_out, tracks_b, seed_int):
    key   = jax.random.PRNGKey(seed_int)
    theta = {
        'log_param':      log_p,
        'log_dedx_mean':  local_p['log_dedx_mean'],
        'log_dedx_sigma': local_p['log_dedx_sigma'],
    }
    return compute_loss(theta, target_out, tracks_b,
                        current_params, track_fields, SC_PARAM_NAME, key,
                        lambda_wass=SC_LW_WASS)

def make_sc_vg(seed_int):
    def _loss(log_p, local_p, t_out, trks):
        return sc_loss_fn(log_p, local_p, t_out, trks, seed_int)
    return value_and_grad(_loss, argnums=(0, 1), has_aux=True)

# ── Initialisation — start fresh from +20 % offset ───────────────────────────
_sc_nom         = float(getattr(current_params, SC_PARAM_NAME))
sc_log_param    = jnp.log(jnp.array(_sc_nom * 1.2, dtype=jnp.float32))
sc_param_state  = sc_param_opt.init(sc_log_param)

sc_dedx_opt     = optax.adam(SC_LR_DEDX)
sc_local_cache  = {}   # batch_idx -> {'log_dedx_mean', 'log_dedx_sigma', 'opt_state'}

# ── Metric histories ──────────────────────────────────────────────────────────
sc_losses = []; sc_param_hist = []; sc_dedx_dist = []
sc_hitnll = []; sc_moyal = []; sc_wass = []
sc_grad_epochs = []; sc_lr_hist = []

# ── Helpers ───────────────────────────────────────────────────────────────────
ev_idx     = track_fields.index('eventID')
dedx_idx   = track_fields.index('dEdx')

# ── Training loop ─────────────────────────────────────────────────────────────
print(f"{'Ep':>4}  {'beta':>9}  {'β/nom':>7}  {'|∇logβ|':>10}  "
      f"{'HitNLL':>8}  {'dEdxDist':>9}  {'LR(β)':>9}  {'mode'}")
print("-" * 78)

global_step = 0   # counts fitting steps for scheduler

for epoch in range(SC_NUM_EPOCHS):
    grad_ep = []

    # FIX 1: Alternating freeze
    #   warmup   (epoch < SC_N_WARMUP)  → only dEdx updates, beta frozen
    #   even fitting epochs             → only beta  updates, dEdx frozen
    #   odd  fitting epochs             → only dEdx  updates, beta  frozen
    fit_epoch     = epoch - SC_N_WARMUP         # negative during warmup
    update_beta   = (epoch >= SC_N_WARMUP) and (fit_epoch % 2 == 0)
    update_dedx   = (epoch < SC_N_WARMUP)  or  (fit_epoch % 2 == 1)

    for batch_idx in range(min(SC_NUM_BATCHES, len(dataloader))):
        tracks_b   = jax.device_put(dataloader[batch_idx].reshape(-1, len(track_fields)))
        noise_seed = epoch * 1000 + batch_idx

        target_b = format_target_output(
            target_strategy.predict(current_params, tracks_b, track_fields, noise_seed)
        )

        # Initialise per-batch dedx on first encounter
        if batch_idx not in sc_local_cache:
            _lm = jnp.full((tracks_b.shape[0],), jnp.log(MOYAL_MPV), dtype=jnp.float32)
            _ls = jnp.full_like(_lm, jnp.log(MOYAL_XI))
            _lp = {'log_dedx_mean': _lm, 'log_dedx_sigma': _ls}
            sc_local_cache[batch_idx] = {**_lp, 'opt_state': sc_dedx_opt.init(_lp)}

        local_p    = {k: sc_local_cache[batch_idx][k]
                      for k in ('log_dedx_mean', 'log_dedx_sigma')}
        dedx_state = sc_local_cache[batch_idx]['opt_state']

        vg_step = make_sc_vg(noise_seed)
        (loss_val, aux), (g_param, g_local) = vg_step(
            sc_log_param, local_p, target_b, tracks_b
        )

        # Conditionally apply updates (freeze the other sub-problem)
        if update_dedx:
            d_upd, dedx_state = sc_dedx_opt.update(g_local, dedx_state, local_p)
            local_p = optax.apply_updates(local_p, d_upd)
            sc_local_cache[batch_idx].update({
                'log_dedx_mean':  local_p['log_dedx_mean'],
                'log_dedx_sigma': local_p['log_dedx_sigma'],
                'opt_state':      dedx_state,
            })

        if update_beta:
            p_upd, sc_param_state = sc_param_opt.update(
                g_param, sc_param_state, sc_log_param
            )
            sc_log_param = optax.apply_updates(sc_log_param, p_upd)
            global_step += 1

        grad_ep.append(float(jnp.abs(g_param)))

        vmask = tracks_b[:, ev_idx] >= 0
        sc_losses.append(float(loss_val))
        sc_param_hist.append(float(jnp.exp(sc_log_param)))
        sc_dedx_dist.append(float(jnp.mean(
            jnp.abs(jnp.exp(local_p['log_dedx_mean']) - tracks_b[:, dedx_idx]),
            where=vmask)))
        sc_hitnll.append(float(aux['loss']))
        sc_moyal.append(float(aux.get('moyal_penalty', 0.0)))
        sc_wass.append(float(aux.get('wass_penalty', 0.0)))

    sc_grad_epochs.append(float(np.mean(grad_ep)))
    current_lr = float(sc_param_sched(max(0, global_step - 1)))
    sc_lr_hist.append(current_lr)

    if epoch % 10 == 0 or epoch == SC_NUM_EPOCHS - 1:
        if epoch < SC_N_WARMUP:
            mode = "warmup"
        elif update_beta:
            mode = "β-step"
        else:
            mode = "dEdx-step"
        print(f"{epoch:4d}  {float(jnp.exp(sc_log_param)):9.5f}  "
              f"{float(jnp.exp(sc_log_param))/_sc_nom:7.4f}  "
              f"{sc_grad_epochs[-1]:10.3e}  "
              f"{sc_hitnll[-1]:8.4f}  {sc_dedx_dist[-1]:9.4f}  "
              f"{current_lr:9.2e}  {mode}")

# ── Trace plots ───────────────────────────────────────────────────────────────
_steps       = np.arange(len(sc_losses))
_ep          = np.arange(len(sc_grad_epochs))
_warmup_step = SC_N_WARMUP * SC_NUM_BATCHES

fig, axes = plt.subplots(4, 1, figsize=(9, 12), sharex=False)

# panel 0 — beta recovery
axes[0].plot(_steps, np.array(sc_param_hist) / _sc_nom,
             color='tab:green', lw=1.5, label='β / nominal (Phase 3c)')
# compare with Phase 3b if available
if 'ss_param_hist' in dir() or 'ss_param_hist' in locals():
    _s2 = np.arange(len(ss_param_hist))
    axes[0].plot(_s2, np.array(ss_param_hist) / _sc_nom,
                 color='tab:green', lw=0.8, ls=':', alpha=0.5, label='Phase 3b (ref)')
axes[0].axhline(1.0, color='grey', linestyle=':', lw=1.0, label='True target')
axes[0].axhline(1.2, color='red',  linestyle=':', lw=1.0, label='Init (+20 %)')
axes[0].axvline(_warmup_step, color='orange', linestyle='--', lw=1.0, label='Warmup ends')
axes[0].set_ylabel('β / nominal')
axes[0].set_title('Phase 3c — alternating freeze + cosine LR + tighter Wasserstein')
axes[0].legend(fontsize=8); axes[0].grid(True, alpha=0.3)

# panel 1 — loss decomposition
axes[1].semilogy(_steps, sc_losses, color='tab:blue',   lw=1.0, label='Total loss')
axes[1].semilogy(_steps, sc_hitnll, color='tab:cyan',   lw=1.0, ls='--', label='Hit NLL')
axes[1].semilogy(_steps, np.maximum(sc_moyal, 1e-9),
                 color='tab:orange', lw=0.8, ls='-.', label='Moyal NLL')
axes[1].semilogy(_steps, np.maximum(sc_wass, 1e-9),
                 color='tab:purple', lw=0.8, ls=':', label='Wasserstein')
axes[1].axvline(_warmup_step, color='orange', linestyle='--', lw=1.0)
axes[1].set_ylabel('Loss (log scale)'); axes[1].legend(fontsize=8, ncol=2)
axes[1].grid(True, alpha=0.3)

# panel 2 — gradient magnitude
axes[2].semilogy(_ep, sc_grad_epochs, color='tab:red', lw=1.2, label='|∇ log β|')
axes[2].axvline(SC_N_WARMUP, color='orange', linestyle='--', lw=1.0, label='Warmup ends')
# shade alternating epochs
for e in range(SC_N_WARMUP, SC_NUM_EPOCHS, 2):   # beta-update epochs
    axes[2].axvspan(e, e + 1, color='tab:blue', alpha=0.07)
axes[2].set_ylabel('Mean |∇ log β| per epoch')
axes[2].set_title('Blue bands = epochs where β is updated (dEdx frozen)')
axes[2].legend(fontsize=8); axes[2].grid(True, alpha=0.3)

# panel 3 — LR schedule
axes[3].semilogy(_ep, sc_lr_hist, color='tab:purple', lw=1.2)
axes[3].set_xlabel('Epoch'); axes[3].set_ylabel('β learning rate')
axes[3].set_title('Cosine decay schedule for β')
axes[3].grid(True, alpha=0.3)

fig.tight_layout()
plt.show()

sc_final = float(jnp.exp(sc_log_param))
print(f"\nPhase 3c final {SC_PARAM_NAME} = {sc_final:.6f}  "
      f"(nominal = {_sc_nom:.6f},  residual = {100*(sc_final/_sc_nom - 1):+.2f} %)")
if 'ss_final' in dir():
    print(f"Phase 3b residual was {100*(ss_final/_sc_nom - 1):+.2f} %  "
          f"→ improvement: {abs(100*(ss_final/_sc_nom - 1)) - abs(100*(sc_final/_sc_nom - 1)):+.2f} pp")


In [19]:

# ═══════════════════════════════════════════════════════════════════════════════
# PHASE 3d — Oracle test: dEdx frozen at TRUE MC values, only beta optimised
#
# Diagnosis from Phase 3b/3c:
#   beta stalls at −2 % despite a clean gradient signal.
#   dEdxDist never reaches zero — the per-segment dEdx nuisances are drifting
#   to partially absorb the charge-scale offset that should move beta.
#   This is a DEGENERACY: a 2% lower beta + 2% higher dEdx gives the same
#   hit NLL as the true solution; the Moyal/Wasserstein priors are not tight
#   enough to break the flat valley direction:
#       charge ∝ β · (dE/dx) / [1 + β · (dE/dx) · ...]
#
# Oracle test:
#   Lock every segment's log_dedx_mean to the TRUE MC dEdx value from the
#   track record (tracks_b[:, dedx_idx]).  Only β is a free parameter.
#   With the degeneracy broken, the loss landscape collapses to a 1D problem
#   and a clear minimum must appear at β = β_true — IF the gradient signal
#   is real.
#
# This is the "best possible convergence" benchmark.  Results:
#   • If beta converges → gradient is fine; the nuisance degeneracy is the
#     only obstacle.  Fix: add a per-segment Gaussian prior on log_dedx.
#   • If beta still stalls → the hit-NLL itself has very weak sensitivity
#     to β at the segment level (e.g. Birks saturation is near-linear for
#     MIP-like dEdx).  Would need charge-sum or pulse-area observable.
# ═══════════════════════════════════════════════════════════════════════════════
import jax, optax, numpy as np, matplotlib.pyplot as plt
import jax.numpy as jnp
from jax import value_and_grad

# ── Hyper-parameters ──────────────────────────────────────────────────────────
ORC_PARAM_NAME  = 'beta'
ORC_NUM_BATCHES = 10
ORC_NUM_EPOCHS  = 100
ORC_N_WARMUP    = 0      # no dEdx warmup needed — oracle has no nuisances
ORC_LW_WASS     = 1.0   # unused (no nuisances), kept for API
_orc_sched      = optax.cosine_decay_schedule(
    init_value=5e-3, decay_steps=ORC_NUM_EPOCHS * ORC_NUM_BATCHES, alpha=0.01
)
orc_param_opt   = optax.adam(_orc_sched)

# ── Oracle loss: dEdx fixed to TRUE MC value, only log_p is differentiated ───
def orc_loss_fn(log_p, target_out, tracks_b, seed_int):
    """dEdx fixed at MC truth — tests pure beta sensitivity."""
    key       = jax.random.PRNGKey(seed_int)
    dedx_true = tracks_b[:, track_fields.index('dEdx')]
    local_p   = {
        'log_dedx_mean':  jnp.log(jnp.clip(dedx_true, 1e-3, None)),
        'log_dedx_sigma': jnp.full_like(dedx_true, jnp.log(MOYAL_XI)),
    }
    theta = {
        'log_param':      log_p,
        'log_dedx_mean':  local_p['log_dedx_mean'],
        'log_dedx_sigma': local_p['log_dedx_sigma'],
    }
    return compute_loss(theta, target_out, tracks_b,
                        current_params, track_fields, ORC_PARAM_NAME, key,
                        lambda_wass=ORC_LW_WASS)

vg_orc = value_and_grad(orc_loss_fn, argnums=0, has_aux=True)

# ── Initialisations ───────────────────────────────────────────────────────────
_orc_nom        = float(getattr(current_params, ORC_PARAM_NAME))
orc_log_param   = jnp.log(jnp.array(_orc_nom * 1.2, dtype=jnp.float32))
orc_param_state = orc_param_opt.init(orc_log_param)

# ── Histories ─────────────────────────────────────────────────────────────────
orc_losses = []; orc_param_hist = []; orc_hitnll = []
orc_moyal  = []; orc_wass = []; orc_grad_epochs = []

# ── Training loop ─────────────────────────────────────────────────────────────
print(f"{'Ep':>4}  {'beta':>9}  {'β/nom':>7}  {'|∇logβ|':>10}  {'HitNLL':>8}")
print("-" * 50)

for epoch in range(ORC_NUM_EPOCHS):
    grad_ep = []
    for batch_idx in range(min(ORC_NUM_BATCHES, len(dataloader))):
        tracks_b   = jax.device_put(dataloader[batch_idx].reshape(-1, len(track_fields)))
        noise_seed = epoch * 1000 + batch_idx

        target_b = format_target_output(
            target_strategy.predict(current_params, tracks_b, track_fields, noise_seed)
        )

        (loss_val, aux), g_param = vg_orc(
            orc_log_param, target_b, tracks_b, noise_seed
        )

        p_upd, orc_param_state = orc_param_opt.update(
            g_param, orc_param_state, orc_log_param
        )
        orc_log_param = optax.apply_updates(orc_log_param, p_upd)

        grad_ep.append(float(jnp.abs(g_param)))
        orc_losses.append(float(loss_val))
        orc_param_hist.append(float(jnp.exp(orc_log_param)))
        orc_hitnll.append(float(aux['loss']))
        orc_moyal.append(float(aux.get('moyal_penalty', 0.0)))
        orc_wass.append(float(aux.get('wass_penalty', 0.0)))

    orc_grad_epochs.append(float(np.mean(grad_ep)))
    if epoch % 10 == 0 or epoch == ORC_NUM_EPOCHS - 1:
        print(f"{epoch:4d}  {float(jnp.exp(orc_log_param)):9.5f}  "
              f"{float(jnp.exp(orc_log_param))/_orc_nom:7.4f}  "
              f"{orc_grad_epochs[-1]:10.3e}  {orc_hitnll[-1]:8.4f}")

# ── Trace plots ───────────────────────────────────────────────────────────────
_steps = np.arange(len(orc_losses))
_ep    = np.arange(len(orc_grad_epochs))

fig, axes = plt.subplots(3, 1, figsize=(9, 9), sharex=False)

# panel 0 — beta recovery
axes[0].plot(_steps, np.array(orc_param_hist) / _orc_nom,
             color='tab:blue', lw=2.0, label='β / nominal (oracle: dEdx fixed at MC truth)')
if 'ss_param_hist' in globals():
    axes[0].plot(np.arange(len(ss_param_hist)), np.array(ss_param_hist) / _orc_nom,
                 color='tab:green', lw=0.8, ls=':', alpha=0.6, label='Phase 3b (free dEdx)')
axes[0].axhline(1.0, color='grey', linestyle=':', lw=1.0, label='True β')
axes[0].axhline(1.2, color='red',  linestyle=':', lw=1.0, label='Init (+20 %)')
axes[0].set_ylabel('β / nominal')
axes[0].set_title('Phase 3d — Oracle: dEdx fixed at MC truth, only β optimised\n'
                  'If β converges here → nuisance degeneracy was the sole obstacle')
axes[0].legend(fontsize=8); axes[0].grid(True, alpha=0.3)

# panel 1 — loss
axes[1].semilogy(_steps, orc_losses, color='tab:blue',   lw=1.0, label='Total loss')
axes[1].semilogy(_steps, orc_hitnll, color='tab:cyan',   lw=1.0, ls='--', label='Hit NLL')
axes[1].semilogy(_steps, np.maximum(orc_moyal, 1e-9),
                 color='tab:orange', lw=0.8, ls='-.', label='Moyal NLL')
axes[1].set_ylabel('Loss (log scale)'); axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.3)

# panel 2 — gradient
axes[2].semilogy(_ep, orc_grad_epochs, color='tab:red', lw=1.2, label='|∇ log β|')
axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('Mean |∇ log β| per epoch')
axes[2].set_title('Gradient — oracle (should monotonically decrease to ~0)')
axes[2].legend(fontsize=8); axes[2].grid(True, alpha=0.3)

fig.tight_layout()
plt.show()

orc_final = float(jnp.exp(orc_log_param))
print(f"\nOracle final {ORC_PARAM_NAME} = {orc_final:.6f}  "
      f"(nominal = {_orc_nom:.6f},  residual = {100*(orc_final/_orc_nom - 1):+.2f} %)")

if abs(100*(orc_final/_orc_nom - 1)) < 1.0:
    print("\n✓ Oracle converged. Conclusion: the hit-NLL gradient is sufficient to")
    print("  recover beta when dEdx degeneracy is removed.  The fix for Phases 3a–c")
    print("  is a per-segment Gaussian prior: loss += 0.5*((log_dedx - log_dedx0)/σ)²")
    print("  centred on the Moyal MPV with σ controlled by a new LAMBDA_DEDX_PRIOR.")
else:
    print(f"\n✗ Oracle residual = {100*(orc_final/_orc_nom - 1):+.2f} %.  The hit-NLL has")
    print("  weak intrinsic sensitivity to beta at this dEdx range.")
    print("  Possible causes: near-linear Birks at MIP dEdx, or the probabilistic")
    print("  tick model absorbs charge-scale changes through threshold effects.")
    print("  Recommendation: add a charge-sum observable (total ADC per segment)")
    print("  into the loss — this breaks the tick-model degeneracy.")


In [22]:

# ═══════════════════════════════════════════════════════════════════════════════
# PHASE 3e — Free dEdx + per-segment Gaussian prior (breaks the degeneracy)
#
# Diagnosis confirmed by Phase 3d oracle:
#   The hit-NLL gradient is sufficient to recover beta when dEdx is fixed.
#   The residual −2 % plateau in Phases 3a–c was entirely caused by the
#   free dEdx nuisances drifting to absorb the charge-scale offset.
#
# Fix: add a per-segment Gaussian prior on log_dedx centred on log(MOYAL_MPV):
#
#   L_prior = λ_D · (1/N) Σ_i [ (log(dEdx_i) − log(MPV)) / σ_prior ]²
#
# This anchors each segment's dEdx near the Moyal MPV.  The prior is NOT
# infinitely tight (sigma_prior = 0.3 allows ±30 % variation), so genuine
# per-segment fluctuations are still absorbed — but a uniform +2 % shift of
# ALL segments costs λ_D × (0.02/0.3)² ≈ 0.004 λ_D per segment, which
# redirects that charge-scale signal back onto beta.
#
# λ_D is swept from 0 (off) to 50 to find the smallest value that closes
# the gap.  A value around 5–20 typically suffices.
# ═══════════════════════════════════════════════════════════════════════════════
import jax, optax, numpy as np, matplotlib.pyplot as plt
import jax.numpy as jnp
from jax import value_and_grad

# ── Hyper-parameters ──────────────────────────────────────────────────────────
PE_PARAM_NAME      = 'beta'
PE_NUM_BATCHES     = 5
PE_NUM_EPOCHS      = 150
PE_N_WARMUP        = 5
PE_LR_DEDX         = 5e-3
PE_LW_WASS         = 1.0
PE_LAMBDA_D        = 10.0    # per-segment Gaussian prior strength — KEY new knob
PE_PRIOR_SIGMA     = 0.3     # width in log-space (~±30 % at 1σ)
_pe_sched          = optax.cosine_decay_schedule(
    init_value=3e-3, decay_steps=PE_NUM_EPOCHS * PE_NUM_BATCHES, alpha=0.01
)
pe_param_opt       = optax.adam(_pe_sched)
pe_dedx_opt        = optax.adam(PE_LR_DEDX)

# ── Loss closure ──────────────────────────────────────────────────────────────
def pe_loss_fn(log_p, local_p, target_out, tracks_b, seed_int):
    key = jax.random.PRNGKey(seed_int)
    theta = {
        'log_param':      log_p,
        'log_dedx_mean':  local_p['log_dedx_mean'],
        'log_dedx_sigma': local_p['log_dedx_sigma'],
    }
    return compute_loss(theta, target_out, tracks_b,
                        current_params, track_fields, PE_PARAM_NAME, key,
                        lambda_wass=PE_LW_WASS,
                        lambda_dedx_prior=PE_LAMBDA_D,
                        dedx_prior_sigma=PE_PRIOR_SIGMA)

def make_pe_vg(seed_int):
    def _loss(log_p, local_p, t_out, trks):
        return pe_loss_fn(log_p, local_p, t_out, trks, seed_int)
    return value_and_grad(_loss, argnums=(0, 1), has_aux=True)

# ── Initialisations ───────────────────────────────────────────────────────────
_pe_nom         = float(getattr(current_params, PE_PARAM_NAME))
pe_log_param    = jnp.log(jnp.array(_pe_nom * 1.2, dtype=jnp.float32))  # +20 % offset
pe_param_state  = pe_param_opt.init(pe_log_param)
pe_local_cache  = {}

# ── Metric histories ──────────────────────────────────────────────────────────
pe_losses = []; pe_param_hist = []; pe_dedx_dist = []
pe_hitnll = []; pe_moyal = []; pe_wass = []; pe_dedx_prior = []
pe_grad_epochs = []

ev_idx   = track_fields.index('eventID')
dedx_idx = track_fields.index('dEdx')

# ── Training loop ─────────────────────────────────────────────────────────────
print(f"{'Ep':>4}  {'beta':>9}  {'β/nom':>7}  {'|∇logβ|':>10}  "
      f"{'HitNLL':>8}  {'dEdxDist':>9}  {'dEdxPrior':>10}")
print("-" * 78)

for epoch in range(PE_NUM_EPOCHS):
    grad_ep = []

    for batch_idx in range(min(PE_NUM_BATCHES, len(dataloader))):
        tracks_b   = jax.device_put(dataloader[batch_idx].reshape(-1, len(track_fields)))
        noise_seed = epoch * 1000 + batch_idx

        target_b = format_target_output(
            target_strategy.predict(current_params, tracks_b, track_fields, noise_seed)
        )

        if batch_idx not in pe_local_cache:
            _lm = jnp.full((tracks_b.shape[0],), jnp.log(MOYAL_MPV), dtype=jnp.float32)
            _ls = jnp.full_like(_lm, jnp.log(MOYAL_XI))
            _lp = {'log_dedx_mean': _lm, 'log_dedx_sigma': _ls}
            pe_local_cache[batch_idx] = {**_lp, 'opt_state': pe_dedx_opt.init(_lp)}

        local_p    = {k: pe_local_cache[batch_idx][k]
                      for k in ('log_dedx_mean', 'log_dedx_sigma')}
        dedx_state = pe_local_cache[batch_idx]['opt_state']

        vg_step = make_pe_vg(noise_seed)
        (loss_val, aux), (g_param, g_local) = vg_step(
            pe_log_param, local_p, target_b, tracks_b
        )

        # Update dedx nuisances (always)
        d_upd, dedx_state = pe_dedx_opt.update(g_local, dedx_state, local_p)
        local_p = optax.apply_updates(local_p, d_upd)
        pe_local_cache[batch_idx].update({
            'log_dedx_mean':  local_p['log_dedx_mean'],
            'log_dedx_sigma': local_p['log_dedx_sigma'],
            'opt_state':      dedx_state,
        })

        # Update global beta after warmup
        if epoch >= PE_N_WARMUP:
            p_upd, pe_param_state = pe_param_opt.update(
                g_param, pe_param_state, pe_log_param
            )
            pe_log_param = optax.apply_updates(pe_log_param, p_upd)

        grad_ep.append(float(jnp.abs(g_param)))

        vmask = tracks_b[:, ev_idx] >= 0
        pe_losses.append(float(loss_val))
        pe_param_hist.append(float(jnp.exp(pe_log_param)))
        pe_dedx_dist.append(float(jnp.mean(
            jnp.abs(jnp.exp(local_p['log_dedx_mean']) - tracks_b[:, dedx_idx]),
            where=vmask)))
        pe_hitnll.append(float(aux['loss']))
        pe_moyal.append(float(aux.get('moyal_penalty', 0.0)))
        pe_wass.append(float(aux.get('wass_penalty', 0.0)))
        pe_dedx_prior.append(float(aux.get('dedx_prior_penalty', 0.0)))

    pe_grad_epochs.append(float(np.mean(grad_ep)))
    if epoch % 10 == 0 or epoch == PE_NUM_EPOCHS - 1:
        phase = "warmup " if epoch < PE_N_WARMUP else "fitting"
        print(f"{epoch:4d}  {float(jnp.exp(pe_log_param)):9.5f}  "
              f"{float(jnp.exp(pe_log_param))/_pe_nom:7.4f}  "
              f"{pe_grad_epochs[-1]:10.3e}  "
              f"{pe_hitnll[-1]:8.4f}  {pe_dedx_dist[-1]:9.4f}  "
              f"{pe_dedx_prior[-1]:10.4f}  ({phase})")

# ── Trace plots ───────────────────────────────────────────────────────────────
_steps       = np.arange(len(pe_losses))
_ep          = np.arange(len(pe_grad_epochs))
_warmup_step = PE_N_WARMUP * PE_NUM_BATCHES

fig, axes = plt.subplots(3, 1, figsize=(9, 9), sharex=False)

# panel 0 — beta recovery (compare with oracle and Phase 3b)
axes[0].plot(_steps, np.array(pe_param_hist) / _pe_nom,
             color='tab:blue', lw=2.0, label=f'β / nominal  (λ_D = {PE_LAMBDA_D})')
if 'ss_param_hist' in globals():
    axes[0].plot(np.arange(len(ss_param_hist)), np.array(ss_param_hist) / _pe_nom,
                 color='tab:green', lw=0.8, ls=':', alpha=0.5, label='Phase 3b (λ_D = 0)')
if 'orc_param_hist' in globals():
    axes[0].plot(np.arange(len(orc_param_hist)), np.array(orc_param_hist) / _pe_nom,
                 color='tab:orange', lw=0.8, ls='--', alpha=0.6, label='Oracle (dEdx fixed)')
axes[0].axhline(1.0, color='grey', linestyle=':', lw=1.0, label='True β')
axes[0].axhline(1.2, color='red',  linestyle=':', lw=1.0, label='Init (+20 %)')
axes[0].axvline(_warmup_step, color='orange', linestyle='--', lw=1.0)
axes[0].set_ylabel('β / nominal')
axes[0].set_title(f'Phase 3e — Gaussian dEdx prior (λ_D = {PE_LAMBDA_D}, σ = {PE_PRIOR_SIGMA})')
axes[0].legend(fontsize=8); axes[0].grid(True, alpha=0.3)

# panel 1 — loss terms
axes[1].semilogy(_steps, pe_losses, color='tab:blue',   lw=1.0, label='Total loss')
axes[1].semilogy(_steps, pe_hitnll, color='tab:cyan',   lw=1.0, ls='--', label='Hit NLL')
axes[1].semilogy(_steps, np.maximum(pe_moyal, 1e-9),
                 color='tab:orange', lw=0.8, ls='-.', label='Moyal NLL')
axes[1].semilogy(_steps, np.maximum(pe_wass, 1e-9),
                 color='tab:purple', lw=0.8, ls=':', label='Wasserstein')
axes[1].semilogy(_steps, np.maximum(pe_dedx_prior, 1e-9),
                 color='tab:red', lw=1.2, ls='-', label=f'dEdx Gaussian prior (λ={PE_LAMBDA_D})')
axes[1].axvline(_warmup_step, color='orange', linestyle='--', lw=1.0)
axes[1].set_ylabel('Loss (log scale)'); axes[1].legend(fontsize=8, ncol=2)
axes[1].grid(True, alpha=0.3)

# panel 2 — dEdx distance vs gradient
ax2b = axes[2].twinx()
axes[2].semilogy(_ep, pe_grad_epochs, color='tab:red', lw=1.2, label='|∇ log β|')
ax2b.plot(np.arange(len(pe_dedx_dist[::PE_NUM_BATCHES])),
          pe_dedx_dist[::PE_NUM_BATCHES],
          color='tab:brown', lw=1.0, ls='--', alpha=0.7, label='dEdxDist (per epoch)')
axes[2].axvline(PE_N_WARMUP, color='orange', linestyle='--', lw=1.0)
axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('Mean |∇ log β|', color='tab:red')
ax2b.set_ylabel('Mean |dEdx_fit − dEdx_true|', color='tab:brown')
axes[2].set_title('dEdxDist should converge to 0 when degeneracy is broken')
axes[2].legend(loc='upper right', fontsize=8)
ax2b.legend(loc='center right', fontsize=8)
axes[2].grid(True, alpha=0.3)

fig.tight_layout()
plt.show()

pe_final = float(jnp.exp(pe_log_param))
print(f"\nPhase 3e final β = {pe_final:.6f}  "
      f"(nominal = {_pe_nom:.6f},  residual = {100*(pe_final/_pe_nom - 1):+.2f} %)")
if 'ss_final' in globals():
    improvement = abs(100*(ss_final/_pe_nom - 1)) - abs(100*(pe_final/_pe_nom - 1))
    print(f"Phase 3b residual was {100*(ss_final/_pe_nom - 1):+.2f} %  →  "
          f"improvement with Gaussian prior: {improvement:+.2f} pp")
print(f"\nKey insight: dEdxDist final = {pe_dedx_dist[-1]:.4f}  "
      f"(Phase 3b was ~0.033).  Closer to 0 → degeneracy more broken.")


In [25]:

# ═══════════════════════════════════════════════════════════════════════════════
# PHASE 3f — Batch-mean constraint prior (corrected degeneracy fix)
#
# Why Phase 3e (per-segment Gaussian prior centred on MPV) made things worse:
#   The true per-segment dEdx values are Landau-distributed.  Segments in the
#   tail (high dEdx) are legitimately far from the MPV.  The prior forces them
#   toward MPV, creating a wrong dEdx prediction.  The optimizer compensates by
#   pushing beta FURTHER down — resulting in -6.2% instead of -2.1%.
#
# Correct prior — BATCH-MEAN CONSTRAINT:
#   Penalise only the MEAN of log_dedx across the batch:
#
#       L_mean = 0.5 * lambda_D * ((mean[log_dedx_i] - ref_mean) / sigma_mean)^2
#
#   With ref_mean = mean[log(dEdx_MC)], this pins the "centre of gravity" of the
#   dEdx distribution to its reference value.  Per-segment Landau fluctuations
#   are entirely free — only a UNIFORM SHIFT of all segments is penalised.
#
#   The degeneracy direction is exactly this uniform shift, so it is blocked
#   without any bias on individual segments.
#
#   sigma_mean = 0.01 (1% in log-space).  A 2% mean shift costs:
#       0.5 * lambda_D * (0.02/0.01)^2 = 2 * lambda_D
#   With lambda_D = 100 that is 200 — same order as the beta gradient, which
#   forces the optimizer to adjust beta rather than shifting dEdx.
# ═══════════════════════════════════════════════════════════════════════════════
import jax, optax, numpy as np, matplotlib.pyplot as plt
import jax.numpy as jnp
from jax import value_and_grad

# ── Hyper-parameters ──────────────────────────────────────────────────────────
PF_PARAM_NAME          = 'beta'
PF_NUM_BATCHES         = 5
PF_NUM_EPOCHS          = 150
PF_N_WARMUP            = 5
PF_LR_DEDX             = 5e-3
PF_LW_WASS             = 1.0
PF_LAMBDA_MEAN         = 100.0   # batch-mean constraint — KEY new knob
PF_MEAN_PRIOR_SIGMA    = 0.01    # 1% allowed drift of the log-dEdx batch mean
_pf_sched = optax.cosine_decay_schedule(
    init_value=3e-3, decay_steps=PF_NUM_EPOCHS * PF_NUM_BATCHES, alpha=0.01
)
pf_param_opt  = optax.adam(_pf_sched)
pf_dedx_opt   = optax.adam(PF_LR_DEDX)

# ── Loss closure ──────────────────────────────────────────────────────────────
def pf_loss_fn(log_p, local_p, target_out, tracks_b, seed_int):
    key   = jax.random.PRNGKey(seed_int)
    theta = {
        'log_param':      log_p,
        'log_dedx_mean':  local_p['log_dedx_mean'],
        'log_dedx_sigma': local_p['log_dedx_sigma'],
    }
    return compute_loss(theta, target_out, tracks_b,
                        current_params, track_fields, PF_PARAM_NAME, key,
                        lambda_wass=PF_LW_WASS,
                        lambda_dedx_mean_prior=PF_LAMBDA_MEAN,
                        dedx_mean_prior_sigma=PF_MEAN_PRIOR_SIGMA)

def make_pf_vg(seed_int):
    def _loss(log_p, local_p, t_out, trks):
        return pf_loss_fn(log_p, local_p, t_out, trks, seed_int)
    return value_and_grad(_loss, argnums=(0, 1), has_aux=True)

# ── Initialisations ───────────────────────────────────────────────────────────
_pf_nom        = float(getattr(current_params, PF_PARAM_NAME))
pf_log_param   = jnp.log(jnp.array(_pf_nom * 1.2, dtype=jnp.float32))
pf_param_state = pf_param_opt.init(pf_log_param)
pf_local_cache = {}

# ── Metric histories ──────────────────────────────────────────────────────────
pf_losses = []; pf_param_hist = []; pf_dedx_dist = []
pf_hitnll = []; pf_wass = []; pf_mean_prior = []
pf_grad_epochs = []; pf_dedx_batch_mean = []

ev_idx   = track_fields.index('eventID')
dedx_idx = track_fields.index('dEdx')

# ── Training loop ─────────────────────────────────────────────────────────────
print(f"{'Ep':>4}  {'beta':>9}  {'β/nom':>7}  {'|∇logβ|':>10}  "
      f"{'HitNLL':>8}  {'dEdxDist':>9}  {'meanPrior':>10}")
print("-" * 78)

for epoch in range(PF_NUM_EPOCHS):
    grad_ep = []

    for batch_idx in range(min(PF_NUM_BATCHES, len(dataloader))):
        tracks_b   = jax.device_put(dataloader[batch_idx].reshape(-1, len(track_fields)))
        noise_seed = epoch * 1000 + batch_idx

        target_b = format_target_output(
            target_strategy.predict(current_params, tracks_b, track_fields, noise_seed)
        )

        if batch_idx not in pf_local_cache:
            _lm = jnp.full((tracks_b.shape[0],), jnp.log(MOYAL_MPV), dtype=jnp.float32)
            _ls = jnp.full_like(_lm, jnp.log(MOYAL_XI))
            _lp = {'log_dedx_mean': _lm, 'log_dedx_sigma': _ls}
            pf_local_cache[batch_idx] = {**_lp, 'opt_state': pf_dedx_opt.init(_lp)}

        local_p    = {k: pf_local_cache[batch_idx][k]
                      for k in ('log_dedx_mean', 'log_dedx_sigma')}
        dedx_state = pf_local_cache[batch_idx]['opt_state']

        vg_step = make_pf_vg(noise_seed)
        (loss_val, aux), (g_param, g_local) = vg_step(
            pf_log_param, local_p, target_b, tracks_b
        )

        d_upd, dedx_state = pf_dedx_opt.update(g_local, dedx_state, local_p)
        local_p = optax.apply_updates(local_p, d_upd)
        pf_local_cache[batch_idx].update({
            'log_dedx_mean':  local_p['log_dedx_mean'],
            'log_dedx_sigma': local_p['log_dedx_sigma'],
            'opt_state':      dedx_state,
        })

        if epoch >= PF_N_WARMUP:
            p_upd, pf_param_state = pf_param_opt.update(
                g_param, pf_param_state, pf_log_param
            )
            pf_log_param = optax.apply_updates(pf_log_param, p_upd)

        grad_ep.append(float(jnp.abs(g_param)))

        vmask = tracks_b[:, ev_idx] >= 0
        pf_losses.append(float(loss_val))
        pf_param_hist.append(float(jnp.exp(pf_log_param)))
        pf_dedx_dist.append(float(jnp.mean(
            jnp.abs(jnp.exp(local_p['log_dedx_mean']) - tracks_b[:, dedx_idx]),
            where=vmask)))
        pf_hitnll.append(float(aux['loss']))
        pf_wass.append(float(aux.get('wass_penalty', 0.0)))
        pf_mean_prior.append(float(aux.get('dedx_mean_prior_penalty', 0.0)))
        # track batch-mean to verify it stays near reference
        valid_mask = tracks_b[:, ev_idx] >= 0
        n_valid    = jnp.maximum(1.0, jnp.sum(valid_mask))
        pf_dedx_batch_mean.append(float(
            jnp.sum(jnp.where(valid_mask, local_p['log_dedx_mean'], 0.0)) / n_valid
        ))

    pf_grad_epochs.append(float(np.mean(grad_ep)))
    if epoch % 10 == 0 or epoch == PF_NUM_EPOCHS - 1:
        phase = "warmup " if epoch < PF_N_WARMUP else "fitting"
        print(f"{epoch:4d}  {float(jnp.exp(pf_log_param)):9.5f}  "
              f"{float(jnp.exp(pf_log_param))/_pf_nom:7.4f}  "
              f"{pf_grad_epochs[-1]:10.3e}  "
              f"{pf_hitnll[-1]:8.4f}  {pf_dedx_dist[-1]:9.4f}  "
              f"{pf_mean_prior[-1]:10.4f}  ({phase})")

# ── Trace plots ───────────────────────────────────────────────────────────────
_steps       = np.arange(len(pf_losses))
_ep          = np.arange(len(pf_grad_epochs))
_warmup_step = PF_N_WARMUP * PF_NUM_BATCHES

fig, axes = plt.subplots(3, 1, figsize=(9, 9), sharex=False)

# panel 0 — beta recovery (compare all variants)
axes[0].plot(_steps, np.array(pf_param_hist) / _pf_nom,
             color='tab:blue', lw=2.0,
             label=f'Phase 3f — mean-prior (λ={PF_LAMBDA_MEAN}, σ={PF_MEAN_PRIOR_SIGMA})')
if 'ss_param_hist' in globals():
    axes[0].plot(np.arange(len(ss_param_hist)), np.array(ss_param_hist) / _pf_nom,
                 color='tab:green', lw=0.8, ls=':', alpha=0.5, label='Phase 3b (no prior)')
if 'orc_param_hist' in globals():
    axes[0].plot(np.arange(len(orc_param_hist)), np.array(orc_param_hist) / _pf_nom,
                 color='tab:orange', lw=0.8, ls='--', alpha=0.6, label='Oracle bound')
axes[0].axhline(1.0, color='grey', linestyle=':', lw=1.0, label='True β')
axes[0].axhline(1.2, color='red',  linestyle=':', lw=1.0, label='Init (+20 %)')
axes[0].axvline(_warmup_step, color='orange', linestyle='--', lw=1.0)
axes[0].set_ylabel('β / nominal')
axes[0].set_title(f'Phase 3f — batch-mean constraint (λ={PF_LAMBDA_MEAN}, σ_mean={PF_MEAN_PRIOR_SIGMA})')
axes[0].legend(fontsize=8); axes[0].grid(True, alpha=0.3)

# panel 1 — loss terms
axes[1].semilogy(_steps, pf_losses, color='tab:blue',   lw=1.0,  label='Total loss')
axes[1].semilogy(_steps, pf_hitnll, color='tab:cyan',   lw=1.0,  ls='--', label='Hit NLL')
axes[1].semilogy(_steps, np.maximum(pf_wass, 1e-9),
                 color='tab:purple', lw=0.8, ls=':', label='Wasserstein')
axes[1].semilogy(_steps, np.maximum(pf_mean_prior, 1e-9),
                 color='tab:red', lw=1.2, label=f'Mean-constraint prior (λ={PF_LAMBDA_MEAN})')
axes[1].axvline(_warmup_step, color='orange', linestyle='--', lw=1.0)
axes[1].set_ylabel('Loss (log scale)'); axes[1].legend(fontsize=8, ncol=2)
axes[1].grid(True, alpha=0.3)

# panel 2 — log-dEdx batch mean drift
_ref = _LOG_DEDX_REF_MEAN
axes[2].plot(_steps, np.array(pf_dedx_batch_mean) - _ref,
             color='tab:brown', lw=1.0, label='batch_mean(log_dEdx) − ref_mean')
axes[2].axhline(0.0, color='grey', linestyle=':', lw=0.9)
axes[2].axhline(+PF_MEAN_PRIOR_SIGMA, color='red', linestyle=':', lw=0.7, alpha=0.6,
                label=f'±sigma_mean ({PF_MEAN_PRIOR_SIGMA})')
axes[2].axhline(-PF_MEAN_PRIOR_SIGMA, color='red', linestyle=':', lw=0.7, alpha=0.6)
axes[2].axvline(_warmup_step, color='orange', linestyle='--', lw=1.0)
axes[2].set_xlabel('Step'); axes[2].set_ylabel('log_dEdx mean drift')
axes[2].set_title('dEdx batch-mean drift — should stay within ± sigma_mean after warmup')
axes[2].legend(fontsize=8); axes[2].grid(True, alpha=0.3)

fig.tight_layout()
plt.show()

pf_final = float(jnp.exp(pf_log_param))
print(f"\nPhase 3f final β = {pf_final:.6f}  "
      f"(nominal = {_pf_nom:.6f},  residual = {100*(pf_final/_pf_nom - 1):+.2f} %)")
if 'ss_final' in globals():
    print(f"Phase 3b residual: {100*(ss_final/_pf_nom - 1):+.2f} %  "
          f"→ improvement: {abs(100*(ss_final/_pf_nom - 1)) - abs(100*(pf_final/_pf_nom - 1)):+.2f} pp")
if 'orc_final' in globals():
    print(f"Oracle residual:  {100*(orc_final/_pf_nom - 1):+.2f} % (reference ceiling)")


In [ ]:

# ═══════════════════════════════════════════════════════════════════════════════
# PHASE 3g — Adaptive per-batch reference mean (correct degeneracy fix)
#
# Root cause of Phase 3f failure (-6.09 %):
#   _LOG_DEDX_REF_MEAN was computed from `selected` (global MC distribution).
#   The specific batches used in optimisation have a DIFFERENT mean true dEdx.
#   This mismatch injected a systematic gradient that pushed dEdx UP, so beta
#   over-corrected DOWN.
#
# Fix: compute the reference mean WITHIN EACH BATCH from the batch's own true
#   dEdx values (ground truth in the closure test; replace with data-driven
#   estimate in a real experiment).  This ensures we only penalise a SHIFT of
#   the learned dEdx away from the batch truth — not away from a mismatched
#   global constant.
#
# Additionally: use the SAME noise seed for target and prediction (same-seed
#   cancellation, as in Phase 3b) to avoid noise-induced NLL gradients that
#   could also bias beta.
# ═══════════════════════════════════════════════════════════════════════════════
import jax, optax, numpy as np, matplotlib.pyplot as plt
import jax.numpy as jnp
from jax import value_and_grad

# ── Hyper-parameters ──────────────────────────────────────────────────────────
PG_PARAM_NAME          = 'beta'
PG_NUM_BATCHES         = 5
PG_NUM_EPOCHS          = 150
PG_N_WARMUP            = 5
PG_LR_DEDX             = 5e-3
PG_LW_WASS             = 1.0
PG_LAMBDA_MEAN         = 100.0   # batch-mean constraint strength
PG_MEAN_PRIOR_SIGMA    = 0.01    # 1 % allowed log-space drift

_pg_sched = optax.cosine_decay_schedule(
    init_value=3e-3, decay_steps=PG_NUM_EPOCHS * PG_NUM_BATCHES, alpha=0.01
)
pg_param_opt = optax.adam(_pg_sched)
pg_dedx_opt  = optax.adam(PG_LR_DEDX)

# ── Per-batch reference mean helper ───────────────────────────────────────────
def batch_true_log_dedx_mean(tracks_b, fields, ev_idx, dedx_idx):
    """Mean of log(true_dEdx) for valid (non-padding) segments in this batch."""
    ev     = tracks_b[:, ev_idx]
    valid  = ev >= 0
    ldedx  = jnp.log(jnp.clip(tracks_b[:, dedx_idx], 1e-3, None))
    return jnp.sum(jnp.where(valid, ldedx, 0.0)) / jnp.maximum(1.0, jnp.sum(valid))

# ── Loss closure (identical to Phase 3f, but reference is passed per-batch) ───
def pg_loss_fn(log_p, local_p, target_out, tracks_b, seed_int, ref_mean_batch):
    key   = jax.random.PRNGKey(seed_int)
    theta = {
        'log_param':      log_p,
        'log_dedx_mean':  local_p['log_dedx_mean'],
        'log_dedx_sigma': local_p['log_dedx_sigma'],
    }
    # Use compute_loss but with a batch-specific ref_mean injected via a thin wrapper
    param_value = jnp.exp(log_p)
    dedx_mean   = jnp.exp(local_p['log_dedx_mean'])
    dedx_sigma  = jnp.exp(local_p['log_dedx_sigma'])
    event_ids   = tracks_b[:, track_fields.index('eventID')]

    eps         = jax.random.normal(key, shape=dedx_mean.shape)
    dedx_sample = jnp.maximum(dedx_mean + dedx_sigma * eps, 1e-3)

    dedx_idx_f   = track_fields.index('dEdx')
    dE_idx_f     = track_fields.index('dE')
    dx_idx_f     = track_fields.index('dx')
    updated_tracks = tracks_b.at[:, dedx_idx_f].set(dedx_sample)
    updated_tracks = updated_tracks.at[:, dE_idx_f].set(
        dedx_sample * updated_tracks[:, dx_idx_f]
    )

    updated_params = current_params.replace(**{PG_PARAM_NAME: param_value})
    pred_dist      = pred_strategy.predict(updated_params, updated_tracks, track_fields, seed_int)

    hit_loss, aux = loss_strategy.compute(updated_params, pred_dist, target_out)

    track_ids    = tracks_b[:, track_fields.index('trackID')]
    moyal_pen    = moyal_log_prior(dedx_mean, MOYAL_MPV, MOYAL_XI, event_ids)
    wass_pen     = wasserstein_log_prior(dedx_mean, ref_quantiles, event_ids)
    tv_pen       = compute_tv_penalty(local_p['log_dedx_mean'], track_ids, event_ids)
    ent_pen      = compute_entropy_penalty(local_p['log_dedx_sigma'], event_ids)

    # Batch-mean prior with PER-BATCH reference (the key fix vs Phase 3f)
    mean_pen = compute_dedx_mean_prior(
        local_p['log_dedx_mean'], event_ids,
        log_dedx_ref_mean=ref_mean_batch,
        sigma_mean=PG_MEAN_PRIOR_SIGMA
    )

    loss_tot = (hit_loss
                + PG_LW_WASS      * wass_pen
                + LAMBDA_MOYAL    * moyal_pen
                + LAMBDA_TV       * tv_pen
                + LAMBDA_ENTROPY  * ent_pen
                + PG_LAMBDA_MEAN  * mean_pen)

    aux[PG_PARAM_NAME]                 = param_value
    aux['loss']                        = hit_loss
    aux['moyal_penalty']               = moyal_pen
    aux['wass_penalty']                = wass_pen
    aux['tv_penalty']                  = tv_pen
    aux['entropy_penalty']             = ent_pen
    aux['dedx_mean_prior_penalty']     = mean_pen
    return loss_tot, aux

# ── Initialisations ───────────────────────────────────────────────────────────
_pg_nom        = float(getattr(current_params, PG_PARAM_NAME))
pg_log_param   = jnp.log(jnp.array(_pg_nom * 1.2, dtype=jnp.float32))
pg_param_state = pg_param_opt.init(pg_log_param)
pg_local_cache = {}

ev_idx_g   = track_fields.index('eventID')
dedx_idx_g = track_fields.index('dEdx')

# ── Metric histories ──────────────────────────────────────────────────────────
pg_losses = []; pg_param_hist = []; pg_dedx_dist = []
pg_hitnll = []; pg_wass = []; pg_mean_prior = []
pg_grad_epochs = []; pg_dedx_batch_mean_drift = []

# ── Training loop ─────────────────────────────────────────────────────────────
print(f"{'Ep':>4}  {'beta':>9}  {'β/nom':>7}  {'|∇logβ|':>10}  "
      f"{'HitNLL':>8}  {'dEdxDist':>9}  {'meanPrior':>10}")
print("-" * 78)

for epoch in range(PG_NUM_EPOCHS):
    grad_ep = []

    for batch_idx in range(min(PG_NUM_BATCHES, len(dataloader))):
        tracks_b   = jax.device_put(dataloader[batch_idx].reshape(-1, len(track_fields)))
        noise_seed = epoch * 1000 + batch_idx   # same seed → same noise cancellation

        # Per-batch true-dEdx reference mean (KEY FIX vs Phase 3f)
        ref_mean_batch = batch_true_log_dedx_mean(
            tracks_b, track_fields, ev_idx_g, dedx_idx_g
        )

        # Same-seed target (cancels noise, same as Phase 3b)
        target_b = format_target_output(
            target_strategy.predict(current_params, tracks_b, track_fields, noise_seed)
        )

        if batch_idx not in pg_local_cache:
            _lm = jnp.full((tracks_b.shape[0],), jnp.log(MOYAL_MPV), dtype=jnp.float32)
            _ls = jnp.full_like(_lm, jnp.log(MOYAL_XI))
            _lp = {'log_dedx_mean': _lm, 'log_dedx_sigma': _ls}
            pg_local_cache[batch_idx] = {**_lp, 'opt_state': pg_dedx_opt.init(_lp)}

        local_p    = {k: pg_local_cache[batch_idx][k]
                      for k in ('log_dedx_mean', 'log_dedx_sigma')}
        dedx_state = pg_local_cache[batch_idx]['opt_state']

        def _make_vg(seed, ref_m):
            def _loss(lp, loc_p):
                return pg_loss_fn(lp, loc_p, target_b, tracks_b, seed, ref_m)
            return value_and_grad(_loss, argnums=(0, 1), has_aux=True)

        vg_step = _make_vg(noise_seed, ref_mean_batch)
        (loss_val, aux), (g_param, g_local) = vg_step(pg_log_param, local_p)

        d_upd, dedx_state = pg_dedx_opt.update(g_local, dedx_state, local_p)
        local_p = optax.apply_updates(local_p, d_upd)
        pg_local_cache[batch_idx].update({
            'log_dedx_mean':  local_p['log_dedx_mean'],
            'log_dedx_sigma': local_p['log_dedx_sigma'],
            'opt_state':      dedx_state,
        })

        if epoch >= PG_N_WARMUP:
            p_upd, pg_param_state = pg_param_opt.update(
                g_param, pg_param_state, pg_log_param
            )
            pg_log_param = optax.apply_updates(pg_log_param, p_upd)

        grad_ep.append(float(jnp.abs(g_param)))

        vmask = tracks_b[:, ev_idx_g] >= 0
        pg_losses.append(float(loss_val))
        pg_param_hist.append(float(jnp.exp(pg_log_param)))
        pg_dedx_dist.append(float(jnp.mean(
            jnp.abs(jnp.exp(local_p['log_dedx_mean']) - tracks_b[:, dedx_idx_g]),
            where=vmask)))
        pg_hitnll.append(float(aux['loss']))
        pg_wass.append(float(aux.get('wass_penalty', 0.0)))
        pg_mean_prior.append(float(aux.get('dedx_mean_prior_penalty', 0.0)))

        # Track mean drift (should stay near 0 after warmup)
        n_valid_g = jnp.maximum(1.0, jnp.sum(vmask))
        learned_mean = (jnp.sum(jnp.where(vmask, local_p['log_dedx_mean'], 0.0))
                        / n_valid_g)
        pg_dedx_batch_mean_drift.append(float(learned_mean - ref_mean_batch))

    pg_grad_epochs.append(float(np.mean(grad_ep)))
    if epoch % 10 == 0 or epoch == PG_NUM_EPOCHS - 1:
        phase = "warmup " if epoch < PG_N_WARMUP else "fitting"
        print(f"{epoch:4d}  {float(jnp.exp(pg_log_param)):9.5f}  "
              f"{float(jnp.exp(pg_log_param))/_pg_nom:7.4f}  "
              f"{pg_grad_epochs[-1]:10.3e}  "
              f"{pg_hitnll[-1]:8.4f}  {pg_dedx_dist[-1]:9.4f}  "
              f"{pg_mean_prior[-1]:10.4f}  ({phase})")

# ── Trace plots ───────────────────────────────────────────────────────────────
_steps       = np.arange(len(pg_losses))
_warmup_step = PG_N_WARMUP * PG_NUM_BATCHES

fig, axes = plt.subplots(3, 1, figsize=(9, 9), sharex=False)

# panel 0 — beta recovery comparison
axes[0].plot(_steps, np.array(pg_param_hist) / _pg_nom,
             color='tab:blue', lw=2.0,
             label=f'Phase 3g — adaptive ref mean (λ={PG_LAMBDA_MEAN}, σ={PG_MEAN_PRIOR_SIGMA})')
if 'pf_param_hist' in globals():
    axes[0].plot(np.arange(len(pf_param_hist)), np.array(pf_param_hist) / _pg_nom,
                 color='tab:red', lw=0.8, ls=':', alpha=0.5, label='Phase 3f (global ref)')
if 'ss_param_hist' in globals():
    axes[0].plot(np.arange(len(ss_param_hist)), np.array(ss_param_hist) / _pg_nom,
                 color='tab:green', lw=0.8, ls=':', alpha=0.5, label='Phase 3b (no prior)')
if 'orc_param_hist' in globals():
    axes[0].plot(np.arange(len(orc_param_hist)), np.array(orc_param_hist) / _pg_nom,
                 color='tab:orange', lw=0.8, ls='--', alpha=0.6, label='Oracle bound')
axes[0].axhline(1.0, color='grey', linestyle=':', lw=1.0, label='True β')
axes[0].axhline(1.2, color='red',  linestyle=':', lw=1.0, label='Init (+20 %)')
axes[0].axvline(_warmup_step, color='orange', linestyle='--', lw=1.0)
axes[0].set_ylabel('β / nominal')
axes[0].set_title(f'Phase 3g — adaptive per-batch reference mean (λ={PG_LAMBDA_MEAN})')
axes[0].legend(fontsize=8, ncol=2); axes[0].grid(True, alpha=0.3)

# panel 1 — loss terms
axes[1].semilogy(_steps, pg_losses, color='tab:blue',   lw=1.0,  label='Total loss')
axes[1].semilogy(_steps, pg_hitnll, color='tab:cyan',   lw=1.0,  ls='--', label='Hit NLL')
axes[1].semilogy(_steps, np.maximum(pg_wass, 1e-9),
                 color='tab:purple', lw=0.8, ls=':', label='Wasserstein')
axes[1].semilogy(_steps, np.maximum(pg_mean_prior, 1e-9),
                 color='tab:red', lw=1.2,
                 label=f'Mean-prior (λ={PG_LAMBDA_MEAN}, batch-adaptive ref)')
axes[1].axvline(_warmup_step, color='orange', linestyle='--', lw=1.0)
axes[1].set_ylabel('Loss (log scale)'); axes[1].legend(fontsize=8, ncol=2)
axes[1].grid(True, alpha=0.3)

# panel 2 — log-dEdx batch mean drift relative to batch truth
axes[2].plot(_steps, pg_dedx_batch_mean_drift,
             color='tab:brown', lw=1.0,
             label='batch_mean(log_dEdx_learned) − batch_mean(log_dEdx_true)')
axes[2].axhline(0.0, color='grey', linestyle=':', lw=0.9)
axes[2].axhline(+PG_MEAN_PRIOR_SIGMA, color='red', linestyle=':', lw=0.7, alpha=0.6,
                label=f'±sigma_mean ({PG_MEAN_PRIOR_SIGMA})')
axes[2].axhline(-PG_MEAN_PRIOR_SIGMA, color='red', linestyle=':', lw=0.7, alpha=0.6)
axes[2].axvline(_warmup_step, color='orange', linestyle='--', lw=1.0)
axes[2].set_xlabel('Step'); axes[2].set_ylabel('log_dEdx mean drift vs. batch truth')
axes[2].set_title('dEdx drift relative to batch truth (should → 0 after warmup)')
axes[2].legend(fontsize=8); axes[2].grid(True, alpha=0.3)

fig.tight_layout()
plt.show()

pg_final = float(jnp.exp(pg_log_param))
print(f"\nPhase 3g final β = {pg_final:.6f}  "
      f"(nominal = {_pg_nom:.6f},  residual = {100*(pg_final/_pg_nom - 1):+.2f} %)")
if 'ss_final' in globals():
    print(f"Phase 3b (no prior):  {100*(ss_final/_pg_nom - 1):+.2f} %")
if 'pf_final' in globals():
    print(f"Phase 3f (global ref): {100*(pf_final/_pg_nom - 1):+.2f} %")
if 'orc_final' in globals():
    print(f"Oracle bound:         {100*(orc_final/_pg_nom - 1):+.2f} %")


In [ ]:

# ═══════════════════════════════════════════════════════════════════════════════
# PHASE 4 — MC closure test
#
# Validates that the joint optimiser can recover an injected parameter shift.
# Protocol:
#   1. For each injection scale s in [0.85, 0.90, 0.95, 1.00, 1.05, 1.10, 1.15]:
#      a. Generate pseudo-data using target_strategy with beta = s × nominal.
#      b. Run joint_optimizer for CLOSURE_EPOCHS starting from nominal beta.
#      c. Record the fitted beta.
#   2. Plot fitted vs injected and compute bias and pull width.
#
# The fit is unbiased if the fitted / injected line is compatible with y=x.
# ═══════════════════════════════════════════════════════════════════════════════
import numpy as np, matplotlib.pyplot as plt
import jax, optax
import jax.numpy as jnp
from jax import value_and_grad

# ── Settings ──────────────────────────────────────────────────────────────────
CLOSURE_PARAM_NAME  = 'beta'
CLOSURE_SCALES      = [0.85, 0.90, 0.95, 1.00, 1.05, 1.10, 1.15]
CLOSURE_EPOCHS      = 40       # fewer epochs — we just need to reach convergence
CLOSURE_NUM_BATCHES = 3        # small for speed; increase for stability
_CLOSURE_LR_PARAMS  = 1e-3
_CLOSURE_LR_DEDX    = 5e-3
_nom_param          = float(getattr(current_params, CLOSURE_PARAM_NAME))

def _closure_optimizer():
    def _lf(theta):
        return {'log_param': 'params', 'log_dedx_mean': 'dedx', 'log_dedx_sigma': 'dedx'}
    return optax.multi_transform(
        {'params': optax.adam(_CLOSURE_LR_PARAMS), 'dedx': optax.adam(_CLOSURE_LR_DEDX)},
        _lf,
    )

def run_closure_fit(injected_scale, seed=42):
    """Fit beta starting from nominal against pseudo-data at injected_scale × nominal."""
    beta_inj       = _nom_param * injected_scale
    params_inj     = current_params.replace(**{CLOSURE_PARAM_NAME: beta_inj})
    rng            = jax.random.PRNGKey(seed)
    optimizer      = _closure_optimizer()
    cache          = {}

    def _loss_fn(theta, target_out, tracks_b, key):
        return compute_loss(theta, target_out, tracks_b,
                            current_params, track_fields, CLOSURE_PARAM_NAME, key)

    vg = value_and_grad(_loss_fn, argnums=0, has_aux=True)

    param_trace = []
    for epoch in range(CLOSURE_EPOCHS):
        for batch_idx in range(min(CLOSURE_NUM_BATCHES, len(dataloader))):
            tracks_b  = jax.device_put(dataloader[batch_idx].reshape(-1, len(track_fields)))
            # pseudo-data generated at injected params
            target_b  = format_target_output(
                target_strategy.predict(params_inj, tracks_b, track_fields, batch_idx + seed)
            )

            if batch_idx not in cache:
                _lm  = jnp.full((tracks_b.shape[0],), jnp.log(MOYAL_MPV), dtype=jnp.float32)
                _ls  = jnp.full_like(_lm, jnp.log(MOYAL_XI))
                # start physics param from nominal (not injected)
                _th  = {'log_param': jnp.log(jnp.array(_nom_param, dtype=jnp.float32)),
                         'log_dedx_mean': _lm, 'log_dedx_sigma': _ls}
                cache[batch_idx] = {**_th, 'opt_state': optimizer.init(_th)}

            theta_b     = {k: cache[batch_idx][k]
                           for k in ('log_param', 'log_dedx_mean', 'log_dedx_sigma')}
            opt_state_b = cache[batch_idx]['opt_state']

            rng, key_step = jax.random.split(rng)
            (_, _), grads = vg(theta_b, target_b, tracks_b, key_step)
            updates, opt_state_b = optimizer.update(grads, opt_state_b, theta_b)
            theta_b   = optax.apply_updates(theta_b, updates)

            cache[batch_idx]['log_param']      = theta_b['log_param']
            cache[batch_idx]['log_dedx_mean']  = theta_b['log_dedx_mean']
            cache[batch_idx]['log_dedx_sigma'] = theta_b['log_dedx_sigma']
            cache[batch_idx]['opt_state']      = opt_state_b

        param_trace.append(float(jnp.exp(theta_b['log_param'])))

    return float(jnp.exp(theta_b['log_param'])), np.array(param_trace)

# ── Run closure scans ─────────────────────────────────────────────────────────
closure_results = []
print(f"Running MC closure test on '{CLOSURE_PARAM_NAME}' (nominal = {_nom_param:.5f})")
print(f"{'Scale':>8}  {'Injected':>10}  {'Fitted':>10}  {'Bias %':>8}")
for scale in CLOSURE_SCALES:
    fitted, trace = run_closure_fit(scale)
    bias_pct = 100.0 * (fitted / (_nom_param * scale) - 1.0)
    closure_results.append({'scale': scale, 'injected': _nom_param * scale,
                             'fitted': fitted, 'bias_pct': bias_pct, 'trace': trace})
    print(f"{scale:8.2f}  {_nom_param*scale:10.5f}  {fitted:10.5f}  {bias_pct:+8.2f} %")

# ── Closure plot 1: fitted vs injected ───────────────────────────────────────
injected = np.array([r['injected'] for r in closure_results])
fitted   = np.array([r['fitted']   for r in closure_results])
bias     = np.array([r['bias_pct'] for r in closure_results])

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].plot(injected, fitted,   'o-', color='steelblue', label='Fitted vs injected')
axes[0].plot(injected, injected, 'k--', lw=0.8, label='y = x (ideal)')
axes[0].set_xlabel(f'Injected {CLOSURE_PARAM_NAME}')
axes[0].set_ylabel(f'Fitted {CLOSURE_PARAM_NAME}')
axes[0].set_title('MC closure: fitted vs injected')
axes[0].legend()
axes[0].grid(True)

axes[1].axhline(0, color='k', linestyle='--', lw=0.8)
axes[1].plot(injected / _nom_param, bias, 's-', color='darkorange')
axes[1].set_xlabel(f'{CLOSURE_PARAM_NAME} / nominal')
axes[1].set_ylabel('Bias  (fitted/injected − 1)  [%]')
axes[1].set_title('Relative bias vs injection scale')
axes[1].grid(True)

fig.suptitle(f'Phase 4 MC closure test — {CLOSURE_PARAM_NAME}', y=1.01)
fig.tight_layout()
plt.show()

# ── Closure plot 2: convergence traces ───────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
for r in closure_results:
    ax.plot(r['trace'] / _nom_param, label=f"scale={r['scale']:.2f}")
    ax.axhline(r['scale'], linestyle=':', lw=0.7, alpha=0.6)
ax.set_xlabel('Epoch')
ax.set_ylabel(f'{CLOSURE_PARAM_NAME} / nominal')
ax.set_title('Convergence traces per injection scale')
ax.legend(fontsize=7, ncol=2)
ax.grid(True)
plt.tight_layout()
plt.show()


# Loss & Gradient Scans — Three dEdx Strategies

Scan the hit-NLL loss (and its gradient w.r.t. the calibration parameter β) over a
range of β values.  Three cases for dEdx are compared side-by-side:

| Case | dEdx used in the scan |
|------|----------------------|
| **Truth** | True per-segment dEdx from MC |
| **Flat (mean)** | Flat value = mean of true dEdx for every segment |
| **Pre-fitted** | Per-segment dEdx obtained by fitting with β fixed at nominal truth, then frozen |


In [24]:

# ═══════════════════════════════════════════════════════════════════════════════
# SCAN CONFIG — edit these to change the scan range / resolution
# ═══════════════════════════════════════════════════════════════════════════════
import jax, jax.numpy as jnp, numpy as np, optax, matplotlib.pyplot as plt
from jax import value_and_grad
from optimize.fit_params import map_norm_to_phys, map_phys_to_norm

SCAN_PARAM_NAME   = 'Ab'   # calibration parameter to scan
SCAN_NUM_BATCHES  = 3      # number of data batches to average over
SCAN_N_POINTS     = 100     # number of scan points
SCAN_SCALE_MIN    = 0.6    # minimum  param / nominal
SCAN_SCALE_MAX    = 1.2    # maximum  param / nominal
SCAN_SEED         = 0      # RNG seed for hit-NLL evaluation
PREFIT_EPOCHS     = 80     # epochs to pre-fit dEdx at true param
PREFIT_LR         = 4e-2   # learning rate for the dEdx pre-fit (same as cell 12)

_scan_nom = float(getattr(current_params, SCAN_PARAM_NAME))

# Build scan values in normalised space then convert back to physical
_scan_phys_vals = np.linspace(
    SCAN_SCALE_MIN * _scan_nom,
    SCAN_SCALE_MAX * _scan_nom,
    SCAN_N_POINTS,
)
scan_norm_vals = jnp.array(
    [float(map_phys_to_norm(jnp.array(v, dtype=jnp.float32), SCAN_PARAM_NAME))
     for v in _scan_phys_vals],
    dtype=jnp.float32,
)

# ── Pre-load per-batch data following cell 12 conventions ────────────────────
dedx_idx_g = track_fields.index('dEdx')
scan_batches = []
for _bidx in range(min(SCAN_NUM_BATCHES, len(dataloader))):
    _tracks_b  = jax.device_put(dataloader[_bidx].reshape(-1, len(track_fields)))
    _target_b  = format_target_output(
        target_strategy.predict(current_params, _tracks_b, track_fields, _bidx)
    )
    _pids_np   = dataset.get_parent_segment_ids(_bidx)
    _pids_jax  = jax.device_put(jnp.array(_pids_np, dtype=jnp.int32))

    # Unique original segments (same logic as cell 12)
    _pv        = _pids_np[_pids_np >= 0]
    _vrows     = np.where(_pids_np >= 0)[0]
    _, _focc   = np.unique(_pv, return_index=True)
    _true_dedx = np.array(_tracks_b)[_vrows[_focc], dedx_idx_g]
    _N_orig    = len(_true_dedx)

    scan_batches.append({
        'tracks':    _tracks_b,
        'target':    _target_b,
        'pids':      _pids_jax,     # shape [N_sub]
        'true_dedx': _true_dedx,    # shape [N_orig]  — ground truth
        'N_orig':    _N_orig,
    })
    print(f"  Batch {_bidx}: {_N_orig} original segments, {len(_pids_np)} sub-steps")

print(f"\nScan parameter : {SCAN_PARAM_NAME}  (nominal = {_scan_nom:.6f})")
print(f"Scan range     : {SCAN_SCALE_MIN:.2f} – {SCAN_SCALE_MAX:.2f} × nominal  ({SCAN_N_POINTS} points)")


In [20]:

# ═══════════════════════════════════════════════════════════════════════════════
# Scan loss (following cell 12):
#   hit NLL  +  Student-t log-prior on dEdx   (no Wasserstein)
#
# Args:
#   norm_param      : sigmoid-normalised calibration scalar
#   log_dedx_orig   : log(dEdx) for the N_orig original segments   [N_orig]
#   parent_ids      : maps each sub-step row → original segment id [N_sub]
#   tracks_b        : track sub-step array                          [N_sub, F]
#   target_out      : formatted target dict
#   seed_int        : integer RNG seed
# ═══════════════════════════════════════════════════════════════════════════════
STUDENT_T_WEIGHT = 0.05   # same as cell 12

def scan_loss_fn(norm_param, log_dedx_orig, parent_ids, tracks_b, target_out,
                 seed_int=SCAN_SEED):
    param_value = map_norm_to_phys(norm_param, SCAN_PARAM_NAME)
    safe_ids    = jnp.clip(parent_ids, 0, log_dedx_orig.shape[0] - 1)
    dedx_sample = jnp.maximum(jnp.exp(jnp.take(log_dedx_orig, safe_ids)), 1e-3)

    dedx_idx = track_fields.index('dEdx')
    dE_idx   = track_fields.index('dE')
    dx_idx   = track_fields.index('dx')
    updated_tracks = tracks_b.at[:, dedx_idx].set(dedx_sample)
    updated_tracks = updated_tracks.at[:, dE_idx].set(
        dedx_sample * updated_tracks[:, dx_idx]
    )

    updated_params = current_params.replace(**{SCAN_PARAM_NAME: param_value})
    pred_dist      = pred_strategy.predict(updated_params, updated_tracks,
                                           track_fields, seed_int)
    hit_loss, _    = loss_strategy.compute(updated_params, pred_dist, target_out)

    # Student-t log-prior on dEdx (no Wasserstein — following cell 12)
    dedx_loss = student_t_nll(dedx_sample, STUDENT_NU_L, STUDENT_NU_R, STUDENT_LOC, STUDENT_SCALE_L, STUDENT_SCALE_R) * STUDENT_T_WEIGHT
    return hit_loss + dedx_loss, {'hit_loss': hit_loss, 'dedx_loss': dedx_loss}

# Gradient w.r.t. norm_param only (argnums=0)
_scan_vg = value_and_grad(scan_loss_fn, argnums=0, has_aux=True)

print(f"Scan loss function defined (Student-t weight = {STUDENT_T_WEIGHT}).")


In [27]:

# ═══════════════════════════════════════════════════════════════════════════════
# PRE-FIT — optimise per-original-segment dEdx with param fixed at true nominal
#
# Follows cell 12 closely:
#   - param is frozen at nominal (sigmoid-bounded, same value)
#   - log_dedx_mean is per ORIGINAL segment (N_orig), expanded via parent_ids
#   - Regularisation: Student-t NLL only (no Wasserstein)
#   - Warm-start from STUDENT_LOC (same as cell 12)
# ═══════════════════════════════════════════════════════════════════════════════
_norm_param_true = map_phys_to_norm(
    jnp.array(_scan_nom, dtype=jnp.float32), SCAN_PARAM_NAME
)

# Per-batch optimizers (one per batch, same as cell 12)
prefit_opts   = [optax.adam(PREFIT_LR) for _ in range(len(scan_batches))]
prefit_states = [
    prefit_opts[b].init(
        jnp.array(np.log(np.full(scan_batches[b]['N_orig'], float(STUDENT_LOC))),
                  dtype=jnp.float32)
    )
    for b in range(len(scan_batches))
]
# Initialise log_dedx from STUDENT_LOC (same warm-start as cell 12)
for _bd in scan_batches:
    _bd['log_dedx_prefit'] = jnp.array(
        np.log(np.full(_bd['N_orig'], float(STUDENT_LOC))), dtype=jnp.float32
    )

def _prefit_loss(log_dedx_orig, parent_ids, tracks_b, target_b):
    """Hit-NLL + Student-t prior; param frozen at true nominal."""
    safe_ids    = jnp.clip(parent_ids, 0, log_dedx_orig.shape[0] - 1)
    dedx_sample = jnp.maximum(jnp.exp(jnp.take(log_dedx_orig, safe_ids)), 1e-3)

    dedx_idx = track_fields.index('dEdx')
    dE_idx   = track_fields.index('dE')
    dx_idx   = track_fields.index('dx')
    updated_t = tracks_b.at[:, dedx_idx].set(dedx_sample)
    updated_t = updated_t.at[:, dE_idx].set(dedx_sample * updated_t[:, dx_idx])

    updated_params = current_params.replace(**{SCAN_PARAM_NAME: _scan_nom})
    pred_dist      = pred_strategy.predict(updated_params, updated_t, track_fields, SCAN_SEED)
    hit_loss, _    = loss_strategy.compute(updated_params, pred_dist, target_b)

    dedx_loss = student_t_nll(dedx_sample, STUDENT_NU_L, STUDENT_NU_R, STUDENT_LOC, STUDENT_SCALE_L, STUDENT_SCALE_R) * STUDENT_T_WEIGHT
    return hit_loss + dedx_loss, {'hit_loss': hit_loss, 'dedx_loss': dedx_loss}

_prefit_vg = value_and_grad(_prefit_loss, argnums=0, has_aux=True)

print(f"Pre-fitting dEdx with {SCAN_PARAM_NAME} fixed at nominal ({_scan_nom:.6f}) ...")
print(f"Student-t prior weight = {STUDENT_T_WEIGHT}  (loc={float(STUDENT_LOC):.3f}, nuL={float(STUDENT_NU_L):.1f}, nuR={float(STUDENT_NU_R):.1f}, "
      f"scaleL={float(STUDENT_SCALE_L):.3f}, scaleR={float(STUDENT_SCALE_R):.3f})")
print(f"{'Ep':>4}  {'b':>2}  {'Loss':>10}  {'HitNLL':>8}  {'dEdxLoss':>9}  {'MAE vs truth':>13}")
print("-" * 56)

for epoch in range(PREFIT_EPOCHS):
    for b, _bd in enumerate(scan_batches):
        (loss_val, aux), grads = _prefit_vg(
            _bd['log_dedx_prefit'], _bd['pids'], _bd['tracks'], _bd['target']
        )
        updates, prefit_states[b] = prefit_opts[b].update(
            grads, prefit_states[b], _bd['log_dedx_prefit']
        )
        _bd['log_dedx_prefit'] = optax.apply_updates(_bd['log_dedx_prefit'], updates)

        if (epoch % 20 == 0 or epoch == PREFIT_EPOCHS - 1) and b == 0:
            fitted = np.exp(np.array(_bd['log_dedx_prefit']))
            mae    = float(np.mean(np.abs(fitted - _bd['true_dedx'])))
            print(f"{epoch:4d}  {b:2d}  {float(loss_val):10.4f}  "
                  f"{float(aux['hit_loss']):8.4f}  {float(aux['dedx_loss']):9.4f}  {mae:13.4f}")

print(f"\nPre-fit done. log_dedx_prefit stored in each scan_batches[b]['log_dedx_prefit'].")


In [28]:

# ═══════════════════════════════════════════════════════════════════════════════
# SCAN — compute loss & gradient at each param value, three dEdx strategies
#
# scan_loss_fn works in normalised param space (sigmoid-bounded).
# log_dedx_orig is per ORIGINAL segment [N_orig]; parent_ids expands to sub-steps.
# Three strategies for log_dedx_orig:
#   truth      — log of true dEdx
#   flat       — log of global mean dEdx (same for all segments)
#   pre-fitted — output of the pre-fit cell above
# ═══════════════════════════════════════════════════════════════════════════════

def _run_scan(get_log_dedx_fn, label):
    """
    get_log_dedx_fn : callable(batch_dict) -> log_dedx_orig [N_orig]
    Returns
        losses_arr : (N_POINTS,) mean total loss
        hitnll_arr : (N_POINTS,) mean hit NLL
        grads_arr  : (N_POINTS,) mean d(loss)/d(norm_param)
    """
    losses_arr = np.zeros(SCAN_N_POINTS)
    hitnll_arr = np.zeros(SCAN_N_POINTS)
    grads_arr  = np.zeros(SCAN_N_POINTS)

    for i, norm_v in enumerate(np.array(scan_norm_vals)):
        norm_p = jnp.array(norm_v, dtype=jnp.float32)
        batch_losses, batch_hitnll, batch_grads = [], [], []

        for _bd in scan_batches:
            log_dedx_orig = get_log_dedx_fn(_bd)
            (loss_v, aux), grad_v = _scan_vg(
                norm_p, log_dedx_orig, _bd['pids'], _bd['tracks'], _bd['target'],
                SCAN_SEED
            )
            batch_losses.append(float(loss_v))
            batch_hitnll.append(float(aux['hit_loss']))
            batch_grads.append(float(grad_v))

        losses_arr[i] = np.mean(batch_losses)
        hitnll_arr[i] = np.mean(batch_hitnll)
        grads_arr[i]  = np.mean(batch_grads)

    print(f"  [{label}] done — total loss [{losses_arr.min():.4f}, {losses_arr.max():.4f}]")
    return losses_arr, hitnll_arr, grads_arr

# ── Strategy 1: truth dEdx ────────────────────────────────────────────────────
def _truth_log_dedx(_bd):
    return jnp.log(jnp.maximum(jnp.array(_bd['true_dedx'], dtype=jnp.float32), 1e-3))

# ── Strategy 2: flat dEdx (global mean of true values) ───────────────────────
_global_dedx_mean = float(np.mean(
    np.concatenate([_bd['true_dedx'] for _bd in scan_batches])
))
print(f"Global mean dEdx (flat strategy) = {_global_dedx_mean:.4f} MeV/cm")

def _flat_log_dedx(_bd):
    return jnp.full((_bd['N_orig'],), float(np.log(_global_dedx_mean)), dtype=jnp.float32)

# ── Strategy 3: pre-fitted dEdx ───────────────────────────────────────────────
def _prefit_log_dedx(_bd):
    return _bd['log_dedx_prefit']

# ── Run all three scans ────────────────────────────────────────────────────────
print(f"\nRunning scans ({SCAN_N_POINTS} points × {len(scan_batches)} batches) ...")
scan_losses_truth,  scan_hitnll_truth,  scan_grads_truth  = _run_scan(_truth_log_dedx,  'truth dEdx')
scan_losses_flat,   scan_hitnll_flat,   scan_grads_flat   = _run_scan(_flat_log_dedx,   'flat dEdx')
scan_losses_prefit, scan_hitnll_prefit, scan_grads_prefit = _run_scan(_prefit_log_dedx, 'pre-fitted dEdx')
print("Scans complete.")


In [29]:

# ═══════════════════════════════════════════════════════════════════════════════
# PLOTS — Loss, hit NLL, and gradient vs param for the three dEdx strategies
# ═══════════════════════════════════════════════════════════════════════════════

# x-axis: physical param / nominal
_x_phys = np.array([
    float(map_norm_to_phys(jnp.array(v, dtype=jnp.float32), SCAN_PARAM_NAME))
    for v in np.array(scan_norm_vals)
]) / _scan_nom

_styles = {
    'truth dEdx'      : dict(color='tab:blue',   ls='-',  lw=2.0, marker='o', ms=3),
    'flat dEdx'       : dict(color='tab:orange', ls='--', lw=1.8, marker='s', ms=3),
    'pre-fitted dEdx' : dict(color='tab:green',  ls='-.', lw=1.8, marker='^', ms=3),
}

_data = {
    'truth dEdx'      : (scan_losses_truth,  scan_hitnll_truth,  scan_grads_truth),
    'flat dEdx'       : (scan_losses_flat,   scan_hitnll_flat,   scan_grads_flat),
    'pre-fitted dEdx' : (scan_losses_prefit, scan_hitnll_prefit, scan_grads_prefit),
}

fig, axes = plt.subplots(3, 1, figsize=(9, 11), sharex=True)

# ── Panel 0: total loss (shifted to min = 0) ──────────────────────────────────
for label, (losses, hitnll, grads) in _data.items():
    axes[0].plot(_x_phys, losses - losses.min(), label=label, **_styles[label])
axes[0].axvline(1.0, color='grey', linestyle=':', lw=1.0, label='True param')
axes[0].set_ylabel('Total loss  (shifted to min = 0)')
axes[0].set_title(
    f'Loss scan — {SCAN_PARAM_NAME}  '
    f'(hit NLL + Student-t dEdx prior, no Wasserstein)\n'
    f'{len(scan_batches)} batches × {SCAN_N_POINTS} points'
)
axes[0].legend(fontsize=9);  axes[0].grid(True, alpha=0.35)

# ── Panel 1: hit NLL only (shifted to min = 0) ───────────────────────────────
for label, (losses, hitnll, grads) in _data.items():
    axes[1].plot(_x_phys, hitnll - hitnll.min(), label=label, **_styles[label])
axes[1].axvline(1.0, color='grey', linestyle=':', lw=1.0)
axes[1].set_ylabel('Hit NLL  (shifted to min = 0)')
axes[1].set_title('Hit NLL component only')
axes[1].legend(fontsize=9);  axes[1].grid(True, alpha=0.35)

# ── Panel 2: gradient d(total loss)/d(norm_param) ────────────────────────────
for label, (losses, hitnll, grads) in _data.items():
    axes[2].plot(_x_phys, grads, label=label, **_styles[label])
axes[2].axhline(0.0, color='grey', linestyle=':', lw=1.0)
axes[2].axvline(1.0, color='grey', linestyle=':', lw=1.0, label='True param')
axes[2].set_xlabel(f'param / nominal  ({SCAN_PARAM_NAME}, nominal = {_scan_nom:.5f})')
axes[2].set_ylabel(r'$\partial\mathcal{L}/\partial\,\mathrm{norm\_param}$')
axes[2].set_title('Gradient of total loss w.r.t. sigmoid-normalised param')
axes[2].legend(fontsize=9);  axes[2].grid(True, alpha=0.35)

fig.suptitle(
    f'Loss & gradient scan — {SCAN_PARAM_NAME}\n'
    f'Student-t dEdx prior (w={STUDENT_T_WEIGHT}), no Wasserstein',
    y=1.01
)
fig.tight_layout()
plt.show()

# ── Summary table ─────────────────────────────────────────────────────────────
print(f"\n{'Strategy':<22}  {'Min total @ p/p0':>17}  {'Min hitNLL @ p/p0':>18}  {'Grad=0 near p/p0':>18}")
print("-" * 80)
for label, (losses, hitnll, grads) in _data.items():
    i_min_tot  = int(np.argmin(losses))
    i_min_nll  = int(np.argmin(hitnll))
    sign_cross = np.where(np.diff(np.sign(grads)))[0]
    if len(sign_cross):
        i_zc    = sign_cross[0]
        xzc_str = f"{_x_phys[i_zc]:.4f}–{_x_phys[i_zc+1]:.4f}"
    else:
        xzc_str = "not found"
    print(f"  {label:<20}  {_x_phys[i_min_tot]:>17.4f}  {_x_phys[i_min_nll]:>18.4f}  {xzc_str:>18}")
